# Setup

In [ ]:
import Interface as I

from tqdm import tqdm

# example_data_dir = I.os.path.join(getting_started_dir, 'example_data')

morphology_id = 'WR64' # 'wr71'
biophysics_id = 0

current_dir = '/Users/robinweiler/Documents/PhD/Project/Code/ISF/in_silico_framework/robinweiler'
data_dir = I.os.path.join(current_dir, morphology_id)

results_morphology_id_dir = I.os.path.join(current_dir, 'results', morphology_id)
if not I.os.path.exists(results_morphology_id_dir):
    I.os.mkdir(results_morphology_id_dir)
results_biophysics_id_dir = I.os.path.join(results_morphology_id_dir, str(biophysics_id))
if not I.os.path.exists(results_biophysics_id_dir):
    I.os.mkdir(results_biophysics_id_dir)
results_dir = results_biophysics_id_dir

In [ ]:
# client = I.get_client(ip='localhost', client_port=8786)

In [ ]:
# # For now, delete previous database
# for filename in I.os.listdir(db_dir):
#     file_path = I.os.path.join(db_dir, filename)
#     print(file_path)

#     try:
#         if I.os.path.isfile(file_path) or I.os.path.islink(file_path):
#             I.os.unlink(file_path)
#         elif I.os.path.isdir(file_path):
#             I.shutil.rmtree(file_path)
#     except Exception as e:
#         print('Failed to delete %s. Reason: %s' % (file_path, e))

In [ ]:
# db = I.DataBase(db_dir)
# db = db.create_sub_db('baseline')

## Load files

### ModelDatabase

In [ ]:
import ibs_projects

loaded_db = I.DataBase(I.os.path.join(current_dir, '20250220_robin_mdb'))
print(loaded_db.keys())

model_db = loaded_db.getitem(morphology_id)
model_db.keys()

### Simulator

In [ ]:
loaded_simulator = model_db.getitem('get_Simulator')(model_db)
loaded_simulator.setup.get_stims()

### Cell params

In [ ]:
loaded_cell_params = model_db.getitem('20250220_eg_models')
loaded_cell_params

In [ ]:
loaded_cell_params.loc[biophysics_id]

### Fixed params

In [ ]:
loaded_fixed_params = model_db.getitem('get_fixed_params')(model_db)
loaded_fixed_params

### Synapses

In [ ]:
loaded_syn_weights = model_db.getitem('avr_syn_stregths')
loaded_syn_weights

In [ ]:
con_file_path = I.os.path.join(
    data_dir,
    'con.con'
)

In [ ]:
syn_file_path = I.os.path.join(
    data_dir,
    'con.syn'
)

In [ ]:
EXC_RECEPTOR_DICT = {
    "glutamate_syn": {
        "threshold": 0.0,
        "delay": 0.0,
        "parameter": {
            "tau1": 26.0,
            "tau2": 2.0,
            "tau3": 2.0,
            "tau4": 0.1,
            "decayampa": 1.0,
            "decaynmda": 1.0,
            "facilampa": 0.0,
            "facilnmda": 0.0,
        },
        "weight": [None, None],
    },
}

INH_RECEPTOR_DICT = {
    "gaba_syn": {
        "threshold": 0.0,
        "delay": 0.0,
        "parameter": {
            "decaygaba": 1.0,
            "decaytime": 20.0,
            "e": -80.0,
            "facilgaba": 0.0,
            "risetime": 1.0,
        },
        "weight": 1.0,
    },
}

In [ ]:
from single_cell_parser.reader import read_synapse_realization
from single_cell_parser.analyze.synanalysis import compute_syn_distance
from single_cell_parser.synapse_mapper import SynapseMapper

from barrel_cortex import EXCITATORY, INHIBITORY

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

# Map synapses
syn_dist = read_synapse_realization(syn_file_path)
synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
synapse_mapper.map_synapse_realization()

syn_stats_df = I.pd.DataFrame(columns=['presyn_cell_label', 'syn_id'])

total_syn_counter = 0
exc_syn_counter = 0
apical_exc_syn_counter = 0
# all_syns = []
for presyn_cell_type in tqdm(cell.synapses.keys()):
    total_syn_counter += len(cell.synapses[presyn_cell_type])

    for syn_id, syn in enumerate(cell.synapses[presyn_cell_type]):
        if (presyn_cell_type in EXCITATORY) or (presyn_cell_type.split('_')[0] in EXCITATORY):
            # all_syns.append(syn)
            exc_syn_counter += 1

            if cell.sections[syn.secID].has_membrane('NaTa_t'):
                apical_exc_syn_counter += 1

            synapse_strength_celltype = [x for x in loaded_syn_weights.keys() if x in presyn_cell_type]
            assert(len(synapse_strength_celltype) == 1)
            synapse_strength_celltype = synapse_strength_celltype[0]
            weight = loaded_syn_weights[synapse_strength_celltype]

            stats = {
                'presyn_cell_label': presyn_cell_type,
                'syn_id': syn_id,
                'syn_weight': weight,
                'soma_dist': compute_syn_distance(cell, syn)
            }
            syn_stats_df = syn_stats_df.append(stats, ignore_index=True)

syn_stats_df['presyn_cell_type'] = syn_stats_df['presyn_cell_label'].apply(lambda x: next((cat for cat in loaded_syn_weights.keys() if cat in x), 'Unknown'))

max_soma_dist = max(syn_stats_df['soma_dist'])

print(f'Total num synapses: {total_syn_counter}')
print(f'Num excitatory synapses: {exc_syn_counter}')
print(f'Num apical excitatory synapses: {apical_exc_syn_counter}')
print(f'Maximum distance to soma: {max_soma_dist}')

syn_stats_df

In [ ]:
df = syn_stats_df.copy()

df = df.dropna(subset=['soma_dist'])
df['soma_dist'] = df['soma_dist'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['soma_dist'], bins=20, edgecolor='black')

I.plt.xlabel('Soma distance (µm)')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
# Group soma_dist values by presyn_cell_type
grouped = syn_stats_df.groupby('presyn_cell_type')['soma_dist'].apply(list)

# Extract data and labels
data = grouped.values
labels = grouped.index

fig, ax = I.plt.subplots()
ax.violinplot(data, showmeans=True, showmedians=False, showextrema=False)

# Set custom categorical labels for the x-axis
ax.set_xticks(I.np.arange(1, len(data)+1))
ax.set_xticklabels(labels, rotation=45)

ax.set_ylabel('Soma distance (µm)')

%matplotlib inline
I.plt.show()

In [ ]:
df = syn_stats_df.copy()

df_no_none = df.dropna(subset=['syn_weight'])
df_no_none['syn_weight'] = df_no_none['syn_weight'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

value_counts = df_no_none['syn_weight'].value_counts()
value_counts = value_counts.sort_index(ascending=True)

value_counts.plot(kind='bar', edgecolor='black')

x_tick_labels = value_counts.index
x_tick_label_addons = [k for v in x_tick_labels for k, val in loaded_syn_weights.items() if val == v]
rounded_labels = [f'{label:.3f} ({label_addon})' for label, label_addon in zip(x_tick_labels, x_tick_label_addons)]
I.plt.xticks(ticks=range(len(rounded_labels)), labels=rounded_labels, rotation=45)

I.plt.xlabel('Synaptic weight')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

## Test model

In [ ]:
record_vars = [
    'Ca_HVA.ica',
    'Ca_LVAst.ica',
    'netglutamate.inmda',
    'cai',
]

def record_rangevars(cell, params):
    for rv in record_vars:
        cell.record_range_var(rv)
    return cell
    
loaded_simulator.setup.cell_modify_funs.append(('BAC.record_range_vars', record_rangevars))

In [ ]:
from biophysics_fitting import get_main_bifurcation_section
from biophysics_fitting.utils import _get_apical_sec_and_i_at_distance

cell, cell_params = loaded_simulator.get_simulated_cell(loaded_cell_params.loc[biophysics_id], 'BAC')

%matplotlib inline
I.plt.plot(cell.tVec, cell.sections[0].recVList[0])
# bifur_sect = get_main_bifurcation_section(cell)
apical_sect = _get_apical_sec_and_i_at_distance(cell, loaded_fixed_params['BAC.hay_measure.recSite'])
vm_dend = I.np.array(apical_sect[0].recVList)
I.plt.plot(cell.tVec, vm_dend[0])
I.plt.show()

In [ ]:
for sec_id in range(len(cell.sections)):
    cai = cell.sections[sec_id].recordVars['cai']
    if len(cai) > 0:
        for seg_cai in cai:
            I.plt.plot(cell.tVec, seg_cai)
    # else:
    #     print(cell.sections[sec_id].name)

In [ ]:
# from visualize.current_visualizer import CurrentAnalysis

# %matplotlib inline
# ca = CurrentAnalysis(cell, rangeVars=record_vars)
# ca.plot_areas(plot_voltage=True, t_stim=280, select_window_relative_to_stim=(0, 500))

## Protocols

In [ ]:
# record_vars = [
#     'Ca_HVA.ica',
#     'Ca_LVAst.ica',
#     # 'netglutamate.inmda',
#     'cai',
# ]

# def record_rangevars(cell, params):
#     for rv in record_vars:
#         cell.record_range_var(rv)
#     return cell

### Glutamate uncaging

In [ ]:
from functools import partial

from biophysics_fitting.parameters import param_to_kwargs
from biophysics_fitting.simulator import run_fun
from biophysics_fitting.hay_complete_default_setup import record_Step, record_BAC

from sumatra.parameters import NTParameterSet

from single_cell_parser.cell import PointCell

from biophysics_fitting.setup_stim import setup_soma_step

def setup_syn_stim(cell, synapses, syn_stim_times=[200]):
    presynaptic_cell = PointCell(syn_stim_times)
    presynaptic_cell.play()

    # Activate synapses
    for syn in synapses:
        new_synapse = cell.add_synapse(syn.secID, syn.ptID, None, syn.preCellType, syn.postCellType)
        new_synapse.weight = syn.weight
        new_synapse.activate_hoc_syn(source=presynaptic_cell, preCell=presynaptic_cell, targetCell=cell, receptors=NTParameterSet(syn.receptors))

loaded_simulator.setup.stim_setup_funs.append(
    ['GlutamateUncaging.stim', param_to_kwargs(setup_syn_stim)]
)

run_fun_base = partial(
    run_fun,
    T=34.0,
    Vinit=-75.0,
    dt=0.025,
    recordingSites=[],
    tStart=0.0,
    tStop=600.0,
    vardt=True
)
loaded_simulator.setup.stim_run_funs.append(['GlutamateUncaging.run', param_to_kwargs(run_fun_base)])

loaded_simulator.setup.stim_response_measure_funs.append(
    ['GlutamateUncaging.measure', param_to_kwargs(record_Step)]
)

# loaded_simulator.setup.cell_modify_funs.append(('GlutamateUncaging.record_range_vars', record_rangevars))

### Somatic current injection

In [ ]:
def setup_soma_stim(cell, current_inj_times=[400], current_amp=2, current_duration=5):
    for inj_time in current_inj_times:
        setup_soma_step(cell, amplitude=current_amp, delay=inj_time, duration=current_duration)

loaded_simulator.setup.stim_setup_funs.append(
    ['SomaticCurrentInjection.stim', param_to_kwargs(setup_soma_stim)]
)

run_fun_base = partial(
    run_fun,
    T=34.0,
    Vinit=-75.0,
    dt=0.025,
    recordingSites=[],
    tStart=0.0,
    tStop=600.0,
    vardt=True
)
loaded_simulator.setup.stim_run_funs.append(['SomaticCurrentInjection.run', param_to_kwargs(run_fun_base)])

loaded_simulator.setup.stim_response_measure_funs.append(
    ['SomaticCurrentInjection.measure', param_to_kwargs(record_Step)]
)

# loaded_simulator.setup.cell_modify_funs.append(('SomaticCurrentInjection.record_range_vars', record_rangevars))

### STDP

In [ ]:
def setup_stdp_stim(cell, synapses, syn_stim_times=[400], postsyn_stim_times=[410], postsyn_stim_amp=2, postsyn_stim_duration=5):  # , postsyn_time_to_spike=5, pairing_delay=10):
    setup_syn_stim(cell, synapses, syn_stim_times)

    # Setup soma current injection
    if postsyn_stim_amp != 0:
        # postsyn_stim_times = []
        # for spike_time in syn_stim_times:
        #     postsyn_inj_delay = spike_time - postsyn_time_to_spike + pairing_delay  # (ms)
        #     postsyn_stim_times.append(postsyn_inj_delay)

        setup_soma_stim(cell, postsyn_stim_times, postsyn_stim_amp, postsyn_stim_duration)

loaded_simulator.setup.stim_setup_funs.append(
    ['STDP.stim', param_to_kwargs(setup_stdp_stim)]
)

loaded_simulator.setup.stim_run_funs.append(['STDP.run', param_to_kwargs(run_fun_base)])

loaded_simulator.setup.stim_response_measure_funs.append(
    ['STDP.measure', param_to_kwargs(record_Step)]
)

# loaded_simulator.setup.cell_modify_funs.append(('STDP.record_range_vars', record_rangevars))

### Letzkus et al., 2006

In [ ]:
def setup_LetzkusEtAl2006_bac_stim(cell, synapses=[], syn_stim_times=[400], postsyn_stim_times=[410], postsyn_stim_amp=2, postsyn_stim_duration=5, dend_stim_times=[410], dend_stim_amp=0.5, dend_stim_duration=100, dend_stim_dist=500):
    if len(synapses) > 0:
        setup_syn_stim(cell, synapses, syn_stim_times)

    # Setup soma current injection
    if postsyn_stim_amp != 0:
        # postsyn_stim_times = []
        # for spike_time in syn_stim_times:
        #     postsyn_inj_delay = spike_time - postsyn_time_to_spike + pairing_delay  # (ms)
        #     postsyn_stim_times.append(postsyn_inj_delay)

        setup_soma_stim(cell, postsyn_stim_times, postsyn_stim_amp, postsyn_stim_duration)

    if dend_stim_amp != 0:
        for inj_time in dend_stim_times:
            setup_soma_step(cell, dend_stim_amp, inj_time, dend_stim_duration, dend_stim_dist)

loaded_simulator.setup.stim_setup_funs.append(
    ['LetzkusEtAl2006_BAC.stim', param_to_kwargs(setup_LetzkusEtAl2006_bac_stim)]
)

loaded_simulator.setup.stim_run_funs.append(['LetzkusEtAl2006_BAC.run', param_to_kwargs(run_fun_base)])

loaded_simulator.setup.stim_response_measure_funs.append(
    ['LetzkusEtAl2006_BAC.measure', param_to_kwargs(record_BAC)]
)

# loaded_simulator.setup.cell_modify_funs.append(('LetzkusEtAl2006_BAC.record_range_vars', record_rangevars))

### Markram et al., 1997

In [ ]:
def setup_MarkramEtAl1997_stim(cell, synapses, stim_start=300, num_AP_burst=5, burst_AP_freq=10, postsyn_stim_amp=2, postsyn_stim_duration=5, postsyn_time_to_spike=5, pairing_delay=10):
    end_burst = stim_start + num_AP_burst * (1000 / burst_AP_freq)
    syn_stim_times = I.np.arange(start=stim_start, stop=end_burst, step=1000/burst_AP_freq)

    setup_stdp_stim(cell, synapses, syn_stim_times, postsyn_stim_amp, postsyn_stim_duration, postsyn_time_to_spike, pairing_delay)

loaded_simulator.setup.stim_setup_funs.append(
    ['MarkramEtAl1997.stim', param_to_kwargs(setup_MarkramEtAl1997_stim)])

run_fun_MarkramEtAl1997 = partial(
        run_fun,
        T=34.0,
        Vinit=-75.0,
        dt=0.025,
        recordingSites=[],
        tStart=0.0,
        tStop=1000.0,
        vardt=True
    )
loaded_simulator.setup.stim_run_funs.append(['MarkramEtAl1997.run', param_to_kwargs(run_fun_MarkramEtAl1997)])

loaded_simulator.setup.stim_response_measure_funs.append(
    ['MarkramEtAl1997.measure', param_to_kwargs(record_Step)]
)

# loaded_simulator.setup.cell_modify_funs.append(('MarkramEtAl1997.record_range_vars', record_rangevars))

### Bittner et al., 2017

In [ ]:
def setup_BittnerEtAl2017_stim(cell, synapses, stim_start=300, presyn_stim_freq=10, presyn_stim_num=10, postsyn_stim_amp=0.6, postsyn_duration=300, pairing_delay=0):
    syn_stim_times = I.np.arange(start=stim_start, stop=(stim_start+presyn_stim_num*(1000/presyn_stim_freq)), step=1000/presyn_stim_freq)

    setup_syn_stim(cell, synapses, syn_stim_times)

    if postsyn_stim_amp != 0:
        postsyn_clamp_delay = syn_stim_times[(len(syn_stim_times)-1)//2] + pairing_delay  # (ms)
        setup_soma_step(cell, amplitude=postsyn_stim_amp, delay=postsyn_clamp_delay, duration=postsyn_duration)

loaded_simulator.setup.stim_setup_funs.append(
    ['BittnerEtAl2017.stim', param_to_kwargs(setup_BittnerEtAl2017_stim)])

run_fun_BittnerEtAl2017 = partial(
        run_fun,
        T=34.0,
        Vinit=-75.0,
        dt=0.025,
        recordingSites=[],
        tStart=0.0,
        tStop=2000.0,
        vardt=False
    )
loaded_simulator.setup.stim_run_funs.append(['BittnerEtAl2017.run', param_to_kwargs(run_fun_BittnerEtAl2017)])

loaded_simulator.setup.stim_response_measure_funs.append(
    ['BittnerEtAl2017.measure', param_to_kwargs(record_Step)]
)

# loaded_simulator.setup.cell_modify_funs.append(('BittnerEtAl2017.record_range_vars', record_rangevars))

# Simulations

In [ ]:
loaded_simulator.setup.stim_setup_funs

## Helperfunctions

### Stats functions

In [ ]:
def get_data_in_timeframe(timepoints, data, t_start=None, t_stop=None):
    if t_start != None and t_stop != None:
        # Only use values between t_start and t_stop
        data_in_timeframe = I.np.array([value for t, value in zip(timepoints, data) if t_start <= t and t_stop > t])
        timepoints_in_timeframe = I.np.array([t for t in timepoints if t_start <= t and t_stop > t])
    elif t_start != None:
        # Only use values after t_start
        data_in_timeframe = I.np.array([value for t, value in zip(timepoints, data) if t_start <= t])
        timepoints_in_timeframe = I.np.array([t for t in timepoints if t_start <= t])
    elif t_stop != None:
        # Only use values before t_stop
        data_in_timeframe = I.np.array([value for t, value in zip(timepoints, data) if t_stop > t])
        timepoints_in_timeframe = I.np.array([t for t in timepoints if t_stop > t])
    else:
        data_in_timeframe = data.copy()
        timepoints_in_timeframe = timepoints.copy()

    return data_in_timeframe, timepoints_in_timeframe

In [ ]:
def get_peaks(timepoints, data, min_height=None, t_start=None, t_stop=None):
    data_in_timeframe, timepoints_in_timeframe = get_data_in_timeframe(timepoints, data, t_start, t_stop)

    # Find all peaks after t_start that have an amplitude of at least min_height or value at t_start
    peak_ids, _ = I.scipy.signal.find_peaks(data_in_timeframe, height=min_height if min_height != None else data_in_timeframe[0])
    # print(peak_ids)

    if len(peak_ids) > 0:
        return data_in_timeframe[peak_ids], timepoints_in_timeframe[peak_ids], data_in_timeframe[peak_ids] - data_in_timeframe[0]
    elif min_height == None:
        if data_in_timeframe[-1] > data_in_timeframe[0]:
            return [data_in_timeframe[-1]], [timepoints_in_timeframe[-1]], [data_in_timeframe[-1] - data_in_timeframe[0]]
        else:
            return [], [], []
    else:
        return [], [], []

In [ ]:
from biophysics_fitting.ephys import find_crossing

def get_syn_auc(timepoints, data, t_start, t_stop=None, baseline='rest'):
    data_in_timeframe, timepoints_in_timeframe = get_data_in_timeframe(timepoints, data, t_start=t_start, t_stop=t_stop)

    if baseline == 'min':
        baseline = min(data_in_timeframe)
    elif baseline == 'rest':
        baseline = data_in_timeframe[0]
    elif isinstance(baseline, int) or isinstance(baseline, float):
        pass
    else:
        raise ValueError(f'Baseline can only be a number, "min", or "rest". Received {baseline}')

    baseline_crossing = find_crossing(data_in_timeframe, thresh=baseline)
    if len(baseline_crossing[0]) == 0 and len(baseline_crossing[1]) > 0:
        baseline_crossing[0] = [0 for _ in baseline_crossing[1]]
    elif len(baseline_crossing[1]) == 0 and len(baseline_crossing[0]) > 0:
        baseline_crossing[1] = [len(timepoints_in_timeframe) - 1 for _ in baseline_crossing[0]]
    elif len(baseline_crossing[0]) == 0 and len(baseline_crossing[1]) == 0:
        raise Exception(f'No crossing of {baseline} was found.')

    # print(timepoints_in_timeframe[baseline_crossing[0]])
    # print(timepoints_in_timeframe[baseline_crossing[1]])

    relative_voltage = data_in_timeframe - baseline

    aucs, auc_timeframes = [], []
    for baseline_upcross, baseline_downcross in zip(baseline_crossing[0], baseline_crossing[1]):
        aucs.append(I.scipy.integrate.trapz(relative_voltage[baseline_upcross:baseline_downcross], timepoints_in_timeframe[baseline_upcross:baseline_downcross]))
        auc_timeframes.append((timepoints_in_timeframe[baseline_upcross], timepoints_in_timeframe[baseline_downcross]))

    return aucs, auc_timeframes, data_in_timeframe[0]

In [ ]:
# from biophysics_fitting.ephys import find_crossing

# def get_syn_auc(timepoints, data, t_start, t_stop=None, baseline='min', terminate_at_baseline=False):
#     data_in_timeframe, timepoints_in_timeframe = get_data_in_timeframe(timepoints, data, t_start=t_start, t_stop=t_stop)

#     if baseline == 'min':
#         baseline = min(data_in_timeframe)
#     elif baseline == 'rest':
#         baseline = data_in_timeframe[0]
#     elif isinstance(baseline, int) or isinstance(baseline, float):
#         pass
#     else:
#         raise ValueError(f'Baseline can only be a number, "min", or "rest". Received {baseline}')

#     if terminate_at_baseline:
#         max_voltage_index = I.np.argmax(data_in_timeframe)

#         # Find the first index where the voltage goes below the baseline
#         below_baseline_indices = I.np.where(data_in_timeframe[max_voltage_index:] < baseline)[0]

#         if len(below_baseline_indices) > 0:
#             # If the voltage drops below the baseline, get the first index after the max voltage
#             below_baseline_index_after_max = below_baseline_indices[0] + max_voltage_index
#         else:
#             # If the voltage never drops below the baseline, use the full trace
#             below_baseline_index_after_max = len(data_in_timeframe)

#         # Truncate the voltage and time trace to include only values above the baseline
#         data_in_timeframe = data_in_timeframe[:below_baseline_index_after_max]
#         timepoints_in_timeframe = timepoints_in_timeframe[:below_baseline_index_after_max]

#     relative_voltage = data_in_timeframe - baseline

#     auc = I.scipy.integrate.trapz(relative_voltage, timepoints_in_timeframe)

#     return auc, (timepoints_in_timeframe[0], timepoints_in_timeframe[-1]), data_in_timeframe[0]

### Run functions

In [ ]:
from single_cell_parser.analyze.synanalysis import compute_syn_distance

def filter_syns_by_dist(all_syns, cell, min_dist=None, max_dist=None):
    if min_dist is not None and max_dist is not None:
        return [syn for syn in all_syns if min_dist <= compute_syn_distance(cell, syn['syn']) <= max_dist]
    elif min_dist is not None:
        return [syn for syn in all_syns if compute_syn_distance(cell, syn['syn']) >= min_dist]
    elif max_dist is not None:
        return [syn for syn in all_syns if compute_syn_distance(cell, syn['syn']) <= max_dist]
    else:
        raise ValueError('min_dist and/or max_dist is required.')

In [ ]:
def run_protocol_soma_only(results_dir, simulator, cell_params, protocol_params, protocol_name='SomaticCurrentInjection', plot=True):
    if not I.os.path.exists(results_dir):
        I.os.mkdir(results_dir)

    cell, cell_params = simulator.setup.get(cell_params)

    # Map synapses
    syn_dist = read_synapse_realization(syn_file_path)
    synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
    synapse_mapper.map_synapse_realization()

    simulated_cell_params = cell_params.copy()
    for key, param in protocol_params.items():
        simulated_cell_params[key] = param

    simulated_cell, param = loaded_simulator.get_simulated_cell(simulated_cell_params, protocol_name)

    # soma_data_file_path = I.os.path.join(results_dir, f'soma.npy')
    # soma_data = I.np.stack((simulated_cell.tVec.as_numpy(), simulated_cell.sections[0].recVList[0].as_numpy()))
    # I.np.save(soma_data_file_path, soma_data)

    fig, axes = I.plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(8, 10))

    # breaker=False
    for presyn_cell_type in tqdm(cell.synapses.keys()):
        # if breaker:
        #     break
        for syn_id, syn in enumerate(cell.synapses[presyn_cell_type]):
            # if breaker:
            #     break
            if (presyn_cell_type in EXCITATORY) or (presyn_cell_type.split('_')[0] in EXCITATORY):
                data_file_path = I.os.path.join(results_dir, f'{presyn_cell_type}-{syn_id}.npz')
                if not I.os.path.exists(data_file_path):
                    syn_seg_id = int(syn.x * simulated_cell.sections[syn.secID].nseg)
                    if syn_seg_id == simulated_cell.sections[syn.secID].nseg:
                        syn_seg_id -= 1

                    # syn_data = I.np.stack((simulated_cell.tVec.as_numpy(), simulated_cell.sections[0].recVList[0].as_numpy(), simulated_cell.sections[syn.secID].recVList[syn_seg_id].as_numpy()))

                    # if len(simulated_cell.sections[syn.secID].recordVars['cai']) > 0:
                    #     syn_data = I.np.vstack((syn_data, simulated_cell.sections[syn.secID].recordVars['cai'][syn_seg_id].as_numpy().reshape(1, -1)))

                    # I.np.save(data_file_path, syn_data)

                    data_dict = {
                        't': simulated_cell.tVec.as_numpy(),
                        'v': simulated_cell.sections[0].recVList[0].as_numpy(),
                        'v_syn': simulated_cell.sections[syn.secID].recVList[syn_seg_id].as_numpy()
                    }
                    # for record_var in simulated_cell.sections[syn.secID].recordVars:
                    #     if len(simulated_cell.sections[syn.secID].recordVars[record_var]) > 0:
                    #         print(simulated_cell.sections[syn.secID].psection())
                    #         data_dict[f'syn_{record_var}'] = simulated_cell.sections[syn.secID].recordVars[record_var][syn_seg_id].as_numpy()
                    # print(data_dict.keys())

                    I.np.savez_compressed(data_file_path, I.np.array([data_dict]))

                    if plot:
                        axes[0].plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0])  # soma
                        axes[1].plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recVList[syn_seg_id])  # synapse
                        # if len(simulated_cell.sections[syn.secID].recordVars['cai']) > 0:
                        #     axes[2].plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recordVars['cai'][syn_seg_id])  # synapse calcium
                        # breaker = True

    if plot:
        axes[0].set_title('Soma')
        axes[1].set_title('Synapse')
        # axes[2].set_title('Synapse')
        axes[1].set_xlabel('Time (ms)')
        axes[1].set_ylabel('Membrane potential (mV)')
        # axes[2].set_ylabel('Calcium concentration')

        %matplotlib inline
        I.plt.show()

In [ ]:
from single_cell_parser.reader import read_synapse_realization
from single_cell_parser.synapse_mapper import SynapseMapper

def run_protocol_all_synapses(results_dir, protocol_name, simulator, cell_params, syn_weights, protocol_params, plot=True):
    if not I.os.path.exists(results_dir):
        I.os.mkdir(results_dir)

    cell, cell_params = simulator.setup.get(cell_params)

    # Map synapses
    syn_dist = read_synapse_realization(syn_file_path)
    synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
    synapse_mapper.map_synapse_realization()

    fig, axes = I.plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(8, 10))# for zoomed-in soma trace

    # breaker = 0
    for presyn_cell_type in tqdm(cell.synapses.keys()):
        # if breaker == 5:
        #     break
        for syn_id, syn in enumerate(cell.synapses[presyn_cell_type]):
            # if breaker == 5:
            #     break
            data_file_path = I.os.path.join(results_dir, f'{presyn_cell_type}-{syn_id}.npz')

            if not I.os.path.exists(data_file_path):
                if (presyn_cell_type in EXCITATORY) or (presyn_cell_type.split('_')[0] in EXCITATORY):
                    synapse_strength_celltype = [x for x in syn_weights.keys() if x in presyn_cell_type]
                    assert(len(synapse_strength_celltype) == 1)
                    synapse_strength_celltype = synapse_strength_celltype[0]
                    weight = loaded_syn_weights[synapse_strength_celltype]

                    syn.weight = {'glutamate_syn': [weight, weight]}
                    syn.receptors = EXC_RECEPTOR_DICT
                else:
                    continue

                simulated_cell_params = cell_params.copy()
                for key, param in protocol_params.items():
                    simulated_cell_params[key] = param
                simulated_cell_params[f'{protocol_name}.stim.synapses'] = [syn]

                simulated_cell, simulated_cell_params = loaded_simulator.get_simulated_cell(simulated_cell_params, protocol_name)

                syn_seg_id = int(syn.x * simulated_cell.sections[syn.secID].nseg)
                if syn_seg_id == simulated_cell.sections[syn.secID].nseg:
                    syn_seg_id -= 1

                # results_arr = I.np.stack((simulated_cell.tVec.as_numpy(), simulated_cell.sections[0].recVList[0].as_numpy(), simulated_cell.sections[syn.secID].recVList[syn_seg_id].as_numpy()))

                # if len(simulated_cell.sections[syn.secID].recordVars['cai']) > 0:
                #     results_arr = I.np.vstack((results_arr, simulated_cell.sections[syn.secID].recordVars['cai'][syn_seg_id].as_numpy().reshape(1, -1)))

                # I.np.save(data_file_path, results_arr)

                data_dict = {
                    't': simulated_cell.tVec.as_numpy(),
                    'v': simulated_cell.sections[0].recVList[0].as_numpy(),
                    'v_syn': simulated_cell.sections[syn.secID].recVList[syn_seg_id].as_numpy()
                }
                # for record_var in simulated_cell.sections[syn.secID].recordVars:
                #     if len(simulated_cell.sections[syn.secID].recordVars[record_var]) > 0:
                #         print(simulated_cell.sections[syn.secID].psection())
                #         data_dict[f'syn_{record_var}'] = simulated_cell.sections[syn.secID].recordVars[record_var][syn_seg_id].as_numpy()
                # print(data_dict.keys())

                I.np.savez_compressed(data_file_path, I.np.array([data_dict]))

                if plot:
                    axes[0].plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0])  # soma
                    axes[1].plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recVList[syn_seg_id])  # synapse
                    # if len(simulated_cell.sections[syn.secID].recordVars['cai']) > 0:
                    #     axes[2].plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recordVars['cai'][syn_seg_id])  # synapse calcium
                # breaker += 1

    if plot:
        axes[0].set_title('Soma')
        axes[1].set_title('Synapse')
        # axes[2].set_title('Synapse')
        axes[1].set_xlabel('Time (ms)')
        axes[1].set_ylabel('Membrane potential (mV)')
        # axes[2].set_ylabel('Calcium concentration')

        %matplotlib inline
        I.plt.show()

In [ ]:
# with h5py.File(data_file_path, 'w') as f:
#     f.create_dataset('t', data=simulated_cell.tVec.as_numpy())
#     f.create_dataset('v', data=simulated_cell.sections[0].recVList[0].as_numpy())
#     f.create_dataset('syn_v', data=simulated_cell.sections[syn.secID].recVList[syn_seg_id].as_numpy())
#     for record_var in simulated_cell.sections[syn.secID].recordVars:
#         if len(simulated_cell.sections[syn.secID].recordVars[record_var]) > 0:
#             f.create_dataset(f'syn_{record_var}', data=simulated_cell.sections[syn.secID].recordVars[record_var][syn_seg_id].as_numpy())

In [ ]:
import warnings
import copy

import neuron
h = neuron.h

from single_cell_parser.reader import read_synapse_realization
from single_cell_parser.synapse_mapper import SynapseMapper

def find_extracellular_stim_syns(results_dir, protocol_name, simulator, cell_params, syn_weights, protocol_params, min_epsp_amp, max_distance, distance_bin_size=50, max_num_synapses=30, plot=True):
    I.np.random.seed(0)

    if not I.os.path.exists(results_dir):
        I.os.mkdir(results_dir)

    cell, cell_params = simulator.setup.get(cell_params)

    # Determine stimulation times
    syn_stim_time = None
    for key in protocol_params.keys():
        if key.endswith('.stim.syn_stim_times'):
            assert len(protocol_params[key]) == 1
            syn_stim_time = protocol_params[key][0]
    assert syn_stim_time is not None

    # Map synapses
    syn_dist = read_synapse_realization(syn_file_path)
    synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
    synapse_mapper.map_synapse_realization()

    all_syns = []
    for presyn_cell_type in cell.synapses.keys():
        for syn_id, syn in enumerate(cell.synapses[presyn_cell_type]):
            if (presyn_cell_type in EXCITATORY) or (presyn_cell_type.split('_')[0] in EXCITATORY):
                all_syns.append({'syn': syn, 'presyn_cell_type': presyn_cell_type, 'syn_id': syn_id})

    distance_bins = I.np.arange(start=0, stop=max_distance, step=distance_bin_size, dtype=I.np.int32)
    print(distance_bins)

    for distance_i, distance_bin in enumerate(distance_bins):
        if distance_i < len(distance_bins) - 1:
            title_text = f'{distance_bin} < distance from soma < {distance_bins[distance_i+1]}'
            print(title_text)

            syns_in_range = filter_syns_by_dist(all_syns, cell, min_dist=distance_bin, max_dist=distance_bins[distance_i+1])
        else:
            title_text = f'{distance_bin} < distance from soma'
            print(title_text)

            syns_in_range = filter_syns_by_dist(all_syns, cell, min_dist=distance_bin)

        print(f'Num possible synapses: {len(syns_in_range)}')

        all_syn_coords = []
        all_data_dicts = []

        rep = 0
        while len(syns_in_range) > 1:
            # Draw a random synapse from syns_in_range
            random_syn_in_range = I.np.random.choice(syns_in_range, 1, replace=False)[0]

            # Find index of random synapse in all_syns
            random_syn_in_range_i = all_syns.index(random_syn_in_range)

            # Remove random synapse from copy of all_syns
            closest_syns_pool = copy.deepcopy(all_syns)
            closest_syns_pool.pop(random_syn_in_range_i)

            data_file_path = I.os.path.join(results_dir, f'{distance_bin}-{rep}.npz')
            rep += 1

            if I.os.path.exists(data_file_path):
                loaded_data = I.np.load(data_file_path, allow_pickle=True)
                data_dict = loaded_data['arr_0'][0]

                presyn_cell_labels = data_dict['presyn_cell_labels']
                syn_ids = data_dict['syn_ids']

                # Remove all synapses that were testes from selection pool
                for presyn_cell_label, syn_id in zip(presyn_cell_labels, syn_ids):
                    for syn_in_range in syns_in_range:
                        if presyn_cell_label == syn_in_range['presyn_cell_type'] and syn_id == syn_in_range['syn_id']:
                            syns_in_range.remove(syn_in_range)

                            # next stimulated synapse
                            # break

                all_syn_coords.append(data_dict['syn_coords'])
                all_data_dicts.append(data_dict)
            else:
                current_epsp_amp = 0
                clustered_syns = [random_syn_in_range]

                # Try at most max_num_synapses synapses to reach min_epsp_amp
                for iteration in range(max_num_synapses):
                    # Get next synapse closest to first synapse
                    closest_syn_i = min(
                        range(len(closest_syns_pool)),
                        key=lambda i: cell.distance_between_pts(sec1=cell.sections[closest_syns_pool[i]['syn'].secID], x1=closest_syns_pool[i]['syn'].x, sec2=cell.sections[clustered_syns[0]['syn'].secID], x2=clustered_syns[0]['syn'].x)
                    )
                    clustered_syns.append(closest_syns_pool[closest_syn_i])

                    # Remove closest synapse from selection pool
                    closest_syns_pool.pop(closest_syn_i)

                    # Reset simulation cell_params
                    simulated_cell_params = cell_params.copy()
                    for key, param in protocol_params.items():
                        simulated_cell_params[key] = param
                    simulated_cell_params[f'{protocol_name}.stim.synapses'] = []

                    # Setup synapse params
                    for syn_dict in clustered_syns:
                        syn = syn_dict['syn']
                        presyn_cell_type = syn_dict['presyn_cell_type']
                        syn_id = syn_dict['syn_id']

                        synapse_strength_celltype = [x for x in syn_weights.keys() if x in presyn_cell_type]
                        assert(len(synapse_strength_celltype) == 1)
                        synapse_strength_celltype = synapse_strength_celltype[0]
                        weight = loaded_syn_weights[synapse_strength_celltype]

                        syn.weight = {'glutamate_syn': [weight, weight]}
                        syn.receptors = EXC_RECEPTOR_DICT

                        simulated_cell_params[f'{protocol_name}.stim.synapses'].append(syn)

                    simulated_cell, simulated_cell_params = loaded_simulator.get_simulated_cell(simulated_cell_params, protocol_name)

                    # Check EPSP amp
                    soma_v_after_syn_stim, _ = get_data_in_timeframe(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], t_start=syn_stim_time)
                    current_epsp_amp = max(soma_v_after_syn_stim) - soma_v_after_syn_stim[0]

                    # print(f"Num synapses: {len(simulated_cell_params[f'{protocol_name}.stim.synapses'])}")
                    # print(f'EPSP amp: {current_epsp_amp}')

                    if current_epsp_amp > min_epsp_amp:
                        print(f"Passed minimum EPSP amp {min_epsp_amp} with {len(simulated_cell_params[f'{protocol_name}.stim.synapses'])} synapses, resulting in {round(current_epsp_amp, 3)} mV EPSP amp.")

                        data_dict = {
                            't': simulated_cell.tVec.as_numpy(),
                            'v': simulated_cell.sections[0].recVList[0].as_numpy(),
                        }

                        syn_pts = []
                        for syn_i, syn in enumerate(simulated_cell_params[f'{protocol_name}.stim.synapses']):
                            syn_seg_id = int(syn.x * simulated_cell.sections[syn.secID].nseg)
                            if syn_seg_id == simulated_cell.sections[syn.secID].nseg:
                                syn_seg_id -= 1

                            data_dict[f'v_syn_{syn_i}'] = simulated_cell.sections[syn.secID].recVList[syn_seg_id].as_numpy()

                            # Collect activated synapses coordinates
                            syn_coord = cell.sections[syn.secID].pts[syn.ptID]
                            syn_pts.append(syn_coord)

                        all_syn_coords.append(syn_pts)

                        data_dict['presyn_cell_labels'] = []
                        data_dict['syn_ids'] = []
                        for syn_dict in clustered_syns:
                            data_dict['presyn_cell_labels'].append(syn_dict['presyn_cell_type'])
                            data_dict['syn_ids'].append(syn_dict['syn_id'])

                        data_dict['syn_coords'] = syn_pts

                        all_data_dicts.append(copy.deepcopy(data_dict))

                        I.np.savez_compressed(data_file_path, I.np.array([data_dict]))

                        # Break out of iteration loop
                        break

                    if iteration == max_num_synapses-1:
                        warnings.warn(f'Unable to reach minimum EPSP amp {min_epsp_amp} with <{max_num_synapses} synapses. Reached {current_epsp_amp}.', RuntimeWarning)

                        if plot:
                            fig = I.plt.figure(figsize=(8, 10))
                            warning_axes = fig.add_subplot(111, projection='3d')

                            # Plot morphology
                            for sec in cell.sections:
                                # Extract 2D coordinates of each section
                                x = I.np.array([h.x3d(i, sec=sec) for i in range(int(h.n3d(sec=sec)))])
                                y = I.np.array([h.y3d(i, sec=sec) for i in range(int(h.n3d(sec=sec)))])
                                z = I.np.array([h.z3d(i, sec=sec) for i in range(int(h.n3d(sec=sec)))])

                                warning_axes.plot(x, y, z, c='gray')

                            warning_axes.view_init(elev=10, azim=90, roll=0)

                            syn_pts = []
                            for syn_i, syn in enumerate(simulated_cell_params[f'{protocol_name}.stim.synapses']):
                                # Collect activated synapses coordinates
                                syn_coord = cell.sections[syn.secID].pts[syn.ptID]
                                syn_pts.append(syn_coord)

                                if cell.sections[syn.secID].has_membrane('NaTa_t'):
                                    print('Synapse was located on apical dendrite')
                                else:
                                    print('Synapse was located on passive dendrite')

                            min_x = I.np.inf
                            max_x = I.np.inf * -1
                            min_y = I.np.inf
                            max_y = I.np.inf * -1
                            min_z = I.np.inf
                            max_z = I.np.inf * -1

                            # Highlight failed synapse locations
                            syn_pts = I.np.array(syn_pts)
                            syn_x = syn_pts[:, 0]
                            syn_y = syn_pts[:, 1]
                            syn_z = syn_pts[:, 2]

                            warning_axes.scatter(syn_x, syn_y, syn_z, s=20)

                            if I.np.min(syn_x) < min_x:
                                min_x = I.np.min(syn_x)
                            if I.np.min(syn_y) < min_y:
                                min_y = I.np.min(syn_y)
                            if I.np.min(syn_z) < min_z:
                                min_z = I.np.min(syn_z)

                            if I.np.max(syn_x) > max_x:
                                max_x = I.np.max(syn_x)
                            if I.np.max(syn_y) > max_y:
                                max_y = I.np.max(syn_y)
                            if I.np.max(syn_z) > max_z:
                                max_z = I.np.max(syn_z)

                            warning_axes.set_xlabel('X Coordinate (μm)')
                            warning_axes.set_ylabel('Y Coordinate (μm)')
                            warning_axes.set_zlabel('Z Coordinate (μm)')

                            warning_axes.set_xlim(min_x-50, max_x+50)
                            warning_axes.set_ylim(min_y-50, max_y+50)
                            warning_axes.set_zlim(min_z-50, max_z+50)

                            %matplotlib inline
                            I.plt.show()

                # Remove all synapses that were tested from selection pool
                for stimulated_syn in clustered_syns:
                    for syn_in_range in syns_in_range:
                        if stimulated_syn['presyn_cell_type'] == syn_in_range['presyn_cell_type'] and stimulated_syn['syn_id'] == syn_in_range['syn_id']:
                            syns_in_range.remove(syn_in_range)

                            # next stimulated_syn
                            # break

        print(f'{len(all_data_dicts)}/{rep} extracellular stimulations succeeded in reaching EPSP amp of {min_epsp_amp} mV.')

        if plot and len(all_syn_coords) > 0:
            fig, axes = I.plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(8, 10))

            # Get position of bottom axis
            bbox = axes[1].get_position()

            # Define inset size relative to bottom axis
            inset_width = 0.5 * (bbox.x1 - bbox.x0)
            inset_height = 0.5 * (bbox.y1 - bbox.y0)

            # Place inset in top-left corner of bottom axis
            inset_left = bbox.x0 + 0.02  # small margin
            inset_bottom = bbox.y1 - inset_height - 0.02  # from top of axis

            # Add 3D inset
            ax_inset = fig.add_axes([inset_left, inset_bottom, inset_width, inset_height], projection='3d')

            # Plot morphology in inset
            for sec in cell.sections:
                # Extract 2D coordinates of each section
                x = I.np.array([h.x3d(i, sec=sec) for i in range(int(h.n3d(sec=sec)))])
                y = I.np.array([h.y3d(i, sec=sec) for i in range(int(h.n3d(sec=sec)))])
                z = I.np.array([h.z3d(i, sec=sec) for i in range(int(h.n3d(sec=sec)))])

                ax_inset.plot(x, y, z, c='gray')

            ax_inset.view_init(elev=10, azim=90, roll=0)

            for data_dict in all_data_dicts:
                axes[0].plot(data_dict['t'], data_dict['v'])  # soma

                for syn_i in range(len(data_dict['syn_ids'])):
                    axes[1].plot(data_dict['t'], data_dict[f'v_syn_{syn_i}'])  # synapse

            fig.suptitle(title_text)
            axes[0].set_title('Soma')
            axes[1].set_title('Synapse')

            axes[1].set_xlabel('Time (ms)')
            axes[1].set_ylabel('Membrane potential (mV)')

            axes[1].set_xlim(left=syn_stim_time-10)

            min_x = I.np.inf
            max_x = I.np.inf * -1
            min_y = I.np.inf
            max_y = I.np.inf * -1
            min_z = I.np.inf
            max_z = I.np.inf * -1

            # Highlight synapse locations in inset
            for syn_pts in all_syn_coords:
                syn_pts = I.np.array(syn_pts)
                syn_x = syn_pts[:, 0]
                syn_y = syn_pts[:, 1]
                syn_z = syn_pts[:, 2]

                ax_inset.scatter(syn_x, syn_y, syn_z, s=20)

                if I.np.min(syn_x) < min_x:
                    min_x = I.np.min(syn_x)
                if I.np.min(syn_y) < min_y:
                    min_y = I.np.min(syn_y)
                if I.np.min(syn_z) < min_z:
                    min_z = I.np.min(syn_z)

                if I.np.max(syn_x) > max_x:
                    max_x = I.np.max(syn_x)
                if I.np.max(syn_y) > max_y:
                    max_y = I.np.max(syn_y)
                if I.np.max(syn_z) > max_z:
                    max_z = I.np.max(syn_z)

            # ax_inset.set_xlabel('X Coordinate (μm)')
            # ax_inset.set_ylabel('Y Coordinate (μm)')
            # ax_inset.set_zlabel('Z Coordinate (μm)')

            ax_inset.set_xlim(min_x-50, max_x+50)
            ax_inset.set_ylim(min_y-50, max_y+50)
            ax_inset.set_zlim(min_z-50, max_z+50)
            
            # Remove grid
            ax_inset.grid(False)

            # Remove ticks
            ax_inset.set_xticks([])
            ax_inset.set_yticks([])
            ax_inset.set_zticks([])

            # Remove pane backgrounds
            ax_inset.xaxis.pane.fill = False
            ax_inset.yaxis.pane.fill = False
            ax_inset.zaxis.pane.fill = False

            # Remove pane edges (frame lines)
            ax_inset.xaxis.pane.set_edgecolor('w')
            ax_inset.yaxis.pane.set_edgecolor('w')
            ax_inset.zaxis.pane.set_edgecolor('w')

            # Remove axis lines (spines)
            ax_inset.xaxis.line.set_color((0.0, 0.0, 0.0, 0.0))
            ax_inset.yaxis.line.set_color((0.0, 0.0, 0.0, 0.0))
            ax_inset.zaxis.line.set_color((0.0, 0.0, 0.0, 0.0))

            # Optional: remove tick labels too, just in case
            ax_inset.set_xticklabels([])
            ax_inset.set_yticklabels([])
            ax_inset.set_zticklabels([])

            %matplotlib inline
            I.plt.show()

In [ ]:
from single_cell_parser.reader import read_synapse_realization
from single_cell_parser.synapse_mapper import SynapseMapper

def run_protocol_extracellular_stim(results_dir, synapse_dir, protocol_name, simulator, cell_params, syn_weights, protocol_params, plot=True):
    if not I.os.path.exists(results_dir):
        I.os.mkdir(results_dir)

    cell, cell_params = simulator.setup.get(cell_params)

    # Map synapses
    syn_dist = read_synapse_realization(syn_file_path)
    synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
    synapse_mapper.map_synapse_realization()

    if plot:
        fig, axes = I.plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(8, 10))

    for filename in tqdm(I.os.listdir(synapse_dir)):
        if filename.endswith('.npz'):
            data_file_path = I.os.path.join(results_dir, filename)

            if not I.os.path.exists(data_file_path):
                file_path = I.os.path.join(synapse_dir, filename)

                file_name_parts = I.os.path.splitext(filename)[0].split('-')
                loaded_distance_bin = int(file_name_parts[-2])
                loaded_rep = int(file_name_parts[-1])

                loaded_data = I.np.load(file_path, allow_pickle=True)
                loaded_data = loaded_data['arr_0'][0]

                presyn_cell_labels = loaded_data['presyn_cell_labels']
                syn_ids = loaded_data['syn_ids']

                simulated_cell_params = cell_params.copy()
                for key, param in protocol_params.items():
                    simulated_cell_params[key] = param
                simulated_cell_params[f'{protocol_name}.stim.synapses'] = []

                data_dict = {}
                data_dict['presyn_cell_labels'] = []
                data_dict['syn_ids'] = []

                for syn_i in range(len(syn_ids)):
                    for presyn_cell_type in cell.synapses.keys():
                        if presyn_cell_type == presyn_cell_labels[syn_i]:
                            for syn_id, syn in enumerate(cell.synapses[presyn_cell_type]):
                                if syn_id == syn_ids[syn_i]:
                                    synapse_strength_celltype = [x for x in syn_weights.keys() if x in presyn_cell_type]
                                    assert(len(synapse_strength_celltype) == 1)
                                    synapse_strength_celltype = synapse_strength_celltype[0]
                                    weight = loaded_syn_weights[synapse_strength_celltype]

                                    syn.weight = {'glutamate_syn': [weight, weight]}
                                    syn.receptors = EXC_RECEPTOR_DICT

                                    simulated_cell_params[f'{protocol_name}.stim.synapses'].append(syn)

                                    data_dict['presyn_cell_labels'].append(presyn_cell_type)
                                    data_dict['syn_ids'].append(syn_id)

                simulated_cell, simulated_cell_params = loaded_simulator.get_simulated_cell(simulated_cell_params, protocol_name)

                data_dict['t'] = simulated_cell.tVec.as_numpy()
                data_dict['v'] = simulated_cell.sections[0].recVList[0].as_numpy()

                if plot:
                    axes[0].plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=f"Distance: {loaded_distance_bin}; Rep {loaded_rep}; {len(simulated_cell_params[f'{protocol_name}.stim.synapses'])} synapses")  # soma

                for syn_i, syn in enumerate(simulated_cell_params[f'{protocol_name}.stim.synapses']):
                    syn_seg_id = int(syn.x * simulated_cell.sections[syn.secID].nseg)
                    if syn_seg_id == simulated_cell.sections[syn.secID].nseg:
                        syn_seg_id -= 1

                    data_dict[f'v_syn_{syn_i}'] = simulated_cell.sections[syn.secID].recVList[syn_seg_id].as_numpy()

                    if plot:
                        axes[1].plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recVList[syn_seg_id], label=f'Distance: {loaded_distance_bin}; Rep {loaded_rep}; Syn {syn_i}')  # synapse

                I.np.savez_compressed(data_file_path, I.np.array([data_dict]))

    if plot:
        axes[0].set_title('Soma')
        axes[1].set_title('Synapse')
        # axes[2].set_title('Synapse')

        axes[1].set_xlabel('Time (ms)')
        axes[1].set_ylabel('Membrane potential (mV)')
        # axes[2].set_ylabel('Calcium concentration')

        # axes[0].legend()
        # axes[1].legend()

        %matplotlib inline
        I.plt.show()

### Load functions

In [ ]:
import warnings

from single_cell_parser.analyze.synanalysis import compute_syn_distance

def load_protocol_results(results_dir, cell, protocol_params, syn_weights, plot=True):
    results_df = I.pd.DataFrame(columns=['presyn_cell_label', 'syn_id'])

    # Map synapses
    syn_dist = read_synapse_realization(syn_file_path)
    synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
    synapse_mapper.map_synapse_realization()

    # Determine stimulation times
    syn_spike_times = []
    postsyn_stim_times = []
    for key in protocol_params.keys():
        if key.endswith('.stim.syn_stim_times'):
            syn_spike_times = protocol_params[key]
        elif key.endswith('.stim.postsyn_stim_times'):
            postsyn_stim_times = protocol_params[key]
        elif key.endswith('.stim.current_inj_times'):
            postsyn_stim_times = protocol_params[key]

    # Determine first stimulation
    if len(syn_spike_times) > 0 and len(postsyn_stim_times) > 0:
        if postsyn_stim_times[0] > syn_spike_times[0]:
            stim_start = syn_spike_times[0]
        else:
            stim_start = postsyn_stim_times[0]
    elif len(syn_spike_times) > 0 and len(postsyn_stim_times) == 0:
        stim_start = syn_spike_times[0]
    elif len(syn_spike_times) == 0 and len(postsyn_stim_times) > 0:
        stim_start = postsyn_stim_times[0]
    else:
        raise ValueError(f'Either syn_spike_times or postsyn_stim_times must be non-empty. Received {syn_spike_times} and {postsyn_stim_times}')

    fig, axes = I.plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(8, 10))

    for filename in tqdm(I.os.listdir(results_dir)):
        if filename.endswith('.npz'):
            file_path = I.os.path.join(results_dir, filename)

            file_name_parts = I.os.path.splitext(filename)[0].split('-')
            loaded_presyn_cell_type = file_name_parts[-2]
            loaded_syn_id = int(file_name_parts[-1])
            
            loaded_data = I.np.load(file_path, allow_pickle=True)
            loaded_data = loaded_data['arr_0'][0]

            for presyn_cell_type in cell.synapses.keys():
                if presyn_cell_type == loaded_presyn_cell_type:
                    for syn_id, syn in enumerate(cell.synapses[presyn_cell_type]):
                        if syn_id == loaded_syn_id:
                            # Somatic features
                            soma_spike_heights, soma_spike_times, _ = get_peaks(loaded_data['t'], loaded_data['v'], min_height=0, t_start=stim_start)

                            epsp_heights, epsp_times, epsp_amps = [], [], []
                            if len(syn_spike_times) > 1 and len(postsyn_stim_times) == 0:
                                # Only calculate EPSP features if there are only synaptic stimulations
                                all_stim_starts = I.np.concatenate((syn_spike_times, postsyn_stim_times))
                                all_stim_starts = I.np.sort(all_stim_starts)
                                for syn_spike_time in syn_spike_times:
                                    # Get the first stimulation time bigger than syn_spike_time or None if syn_spike_time is the last
                                    next_stim = next((stim for stim in all_stim_starts if stim > syn_spike_time), None)
                                    if next_stim != None:
                                        stim_peak_heights, stim_peak_times, stim_peak_amps = get_peaks(loaded_data['t'], loaded_data['v'], t_start=syn_spike_time, t_stop=next_stim)
                                    else:
                                        stim_peak_heights, stim_peak_times, stim_peak_amps = get_peaks(loaded_data['t'], loaded_data['v'], t_start=syn_spike_time)

                                    # TODO make sure this case works properly
                                    if len(stim_peak_amps) > 1:
                                        # fig, axes = I.plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(8, 12))
                                        # axes[0].plot(data[0], data[1])  # soma
                                        # axes[1].plot(data[0], data[1])
                                        # for syn_peak_index in range(len(stim_peak_times)):
                                        #     axes[1].scatter(stim_peak_times[syn_peak_index], stim_peak_heights[syn_peak_index], marker='x')

                                        # axes[1].set_ylim(-70, -50)
                                        # axes[1].set_xlim(syn_spike_time-10, syn_spike_time+20)
                                        # axes[1].grid(which='major', axis='x', linestyle='--', color='gray')
                                        # %matplotlib inline
                                        # I.plt.show()

                                        # print(stim_peak_times)
                                        # print(stim_peak_heights)
                                        # print(stim_peak_amps)
                                        epsp_heights.append(stim_peak_heights[0])
                                        epsp_times.append(stim_peak_times[0])
                                        epsp_amps.append(stim_peak_amps[0])
                                        warnings.warn(f'Found {len(stim_peak_amps)} peak amplitudes at soma between {syn_spike_time} and {next_stim}', RuntimeWarning)
                                    elif len(stim_peak_amps) == 0:
                                        epsp_amps.append(0.0)
                                    else:
                                        epsp_heights.append(stim_peak_heights[0])
                                        epsp_times.append(stim_peak_times[0])
                                        epsp_amps.append(stim_peak_amps[0])

                            # Synaptic features
                            synapse_strength_celltype = [x for x in syn_weights.keys() if x in presyn_cell_type]
                            assert(len(synapse_strength_celltype) == 1)
                            synapse_strength_celltype = synapse_strength_celltype[0]
                            weight = syn_weights[synapse_strength_celltype]

                            syn_seg_id = int(syn.x * cell.sections[syn.secID].nseg)
                            if syn_seg_id == cell.sections[syn.secID].nseg:
                                syn_seg_id -= 1

                            syn_peak_heights, syn_peak_times, _ = get_peaks(loaded_data['t'], loaded_data['v_syn'], t_start=stim_start)
                            syn_aucs, syn_auc_timeframes, syn_baseline_v = get_syn_auc(loaded_data['t'], loaded_data['v_syn'], t_start=stim_start, baseline='rest')

                            dendrite_type = 'basal'
                            if cell.sections[syn.secID].has_membrane('NaTa_t'):
                                dendrite_type = 'apical'

                            # Cai features
                            # if data.shape[0] > 3:
                            #     syn_cai_aucs, syn_cai_auc_timeframes, syn_cai_baseline = get_syn_auc(data[0], data[3], t_start=stim_start, baseline='rest')
                            # else:
                            #     syn_cai_aucs, syn_cai_auc_timeframes, syn_cai_baseline = [], [], []

                            stats = {
                                'presyn_cell_label': presyn_cell_type,
                                'syn_id': syn_id,
                                'syn_weight': weight,
                                'soma_dist': compute_syn_distance(cell, syn),
                                'soma_spike_times': soma_spike_times,
                                'soma_spike_heights': soma_spike_heights,
                                'EPSP_amps': epsp_amps if len(epsp_amps) > 0 else None,
                                'EPSP_heights': epsp_heights if len(epsp_heights) > 0 else None,
                                'EPSP_times': epsp_times if len(epsp_times) > 0 else None,
                                'max_EPSP_amp': I.np.max(epsp_amps) if len(epsp_amps) > 0 else None,
                                'syn_auc_total': I.np.sum(syn_aucs) if len(syn_aucs) > 0 else None,
                                'syn_aucs': syn_aucs,
                                'syn_auc_timeframes': syn_auc_timeframes,
                                'syn_auc_baseline': syn_baseline_v,
                                'syn_peak_heights': syn_peak_heights,
                                'syn_peak_times': syn_peak_times,
                                'syn_max_peak_height': I.np.max(syn_peak_heights) if len(syn_peak_heights) > 0 else None,
                                'syn_dendrite_type': dendrite_type,
                                # 'syn_cai_max_peak': max(data[3]) if data.shape[0] > 3 else None,
                                # 'syn_cai_auc_total': I.np.sum(syn_cai_aucs) if len(syn_cai_aucs) > 0 else None,
                                # 'syn_cai_aucs': syn_cai_aucs,
                                # 'syn_cai_auc_timeframes': syn_cai_auc_timeframes,
                                # 'syn_cai_baseline': syn_cai_baseline,
                                't': loaded_data['t'],
                                'v': loaded_data['v'],
                                'v_syn': loaded_data['v_syn'],
                                # 'syn_cai': data[3] if data.shape[0] > 3 else None
                            }
                            results_df = results_df.append(stats, ignore_index=True)

                            if plot:
                                axes[0].plot(loaded_data['t'], loaded_data['v'])  # soma
                                axes[1].plot(loaded_data['t'], loaded_data['v_syn'])  # synapse
                                # if data.shape[0] > 3:
                                #     axes[2].plot(data[0], data[3])

                            break
                    break
            # break  # for testing

    # Assign presynaptic cell type
    results_df['presyn_cell_type'] = results_df['presyn_cell_label'].apply(lambda x: next((cat for cat in syn_weights.keys() if cat in x), 'Unknown'))

    # Calculate the difference between somatic spike time and first synaptic peak time
    def calculate_delay(row):
        soma_spike_times = row['soma_spike_times']

        if len(soma_spike_times) == len(syn_spike_times):
            # Calculate delay between each pair of somatic and synaptic spike time
            return [soma - syn for soma, syn in zip(soma_spike_times, syn_spike_times)]
        elif len(soma_spike_times) > 0 and len(syn_spike_times) > 0:
            # Calculate delay between first somatic spike time and each synaptic spike time
            first_soma_spike = soma_spike_times[0]
            return [first_soma_spike - syn for syn in syn_spike_times]
        else:
            return None

    results_df['actual_delay'] = results_df.apply(calculate_delay, axis=1)

    # Sort dataframe
    results_df = results_df.sort_values(by=['presyn_cell_label', 'syn_id'])

    if plot:
        axes[0].set_title('Soma')
        axes[1].set_title('Synapse')
        # axes[2].set_title('Synapse')

        axes[0].set_ylabel('Membrane potential (mV)')
        axes[1].set_ylabel('Membrane potential (mV)')
        # axes[2].set_ylabel('Calcium concentration')

        axes[1].set_xlabel('Time (ms)')
        axes[1].set_xlim(left=stim_start-10)

        %matplotlib inline
        I.plt.show()

    return results_df

In [ ]:
from single_cell_parser.reader import read_synapse_realization
from single_cell_parser.synapse_mapper import SynapseMapper

def load_results_extracellular_stim(results_dir, cell, protocol_params, syn_weights, plot=True):
    results_df = I.pd.DataFrame(columns=['presyn_cell_labels', 'syn_ids'])

    # Map synapses
    syn_dist = read_synapse_realization(syn_file_path)
    synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
    synapse_mapper.map_synapse_realization()

    # Determine stimulation times
    syn_spike_times = []
    postsyn_spike_times = []
    for key in protocol_params.keys():
        if key.endswith('.stim.syn_stim_times'):
            syn_spike_times = protocol_params[key]
        elif key.endswith('.stim.postsyn_stim_times'):
            postsyn_spike_times = protocol_params[key]
        elif key.endswith('.stim.current_inj_times'):
            postsyn_spike_times = protocol_params[key]

    # Determine first stimulation
    if len(syn_spike_times) > 0 and len(postsyn_spike_times) > 0:
        if postsyn_spike_times[0] > syn_spike_times[0]:
            stim_start = syn_spike_times[0]
        else:
            stim_start = postsyn_spike_times[0]
    elif len(syn_spike_times) > 0 and len(postsyn_spike_times) == 0:
        stim_start = syn_spike_times[0]
    elif len(syn_spike_times) == 0 and len(postsyn_spike_times) > 0:
        stim_start = postsyn_spike_times[0]
    else:
        raise ValueError(f'Either syn_spike_times or postsyn_spike_times must be non-empty. Received {syn_spike_times} and {postsyn_spike_times}')

    fig, axes = I.plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(8, 10))

    for filename in tqdm(I.os.listdir(results_dir)):
        if filename.endswith('.npz'):
            file_path = I.os.path.join(results_dir, filename)

            file_name_parts = I.os.path.splitext(filename)[0].split('-')
            loaded_distance_bin = int(file_name_parts[-2])
            loaded_rep = int(file_name_parts[-1])

            loaded_data = I.np.load(file_path, allow_pickle=True)
            loaded_data = loaded_data['arr_0'][0]

            presyn_cell_labels = loaded_data['presyn_cell_labels']
            syn_ids = loaded_data['syn_ids']

            # Somatic features
            soma_spike_heights, soma_spike_times, _ = get_peaks(loaded_data['t'], loaded_data['v'], min_height=0, t_start=stim_start)

            epsp_heights, epsp_times, epsp_amps = [], [], []
            all_stim_starts = I.np.concatenate((syn_spike_times, postsyn_spike_times))
            all_stim_starts = I.np.sort(all_stim_starts)
            for syn_spike_time in syn_spike_times:
                # Get the first stimulation time bigger than syn_spike_time or None if syn_spike_time is the last
                next_stim = next((stim for stim in all_stim_starts if stim > syn_spike_time), None)
                if next_stim != None:
                    stim_peak_heights, stim_peak_times, stim_peak_amps = get_peaks(loaded_data['t'], loaded_data['v'], t_start=syn_spike_time, t_stop=next_stim)
                else:
                    stim_peak_heights, stim_peak_times, stim_peak_amps = get_peaks(loaded_data['t'], loaded_data['v'], t_start=syn_spike_time)

                if len(postsyn_spike_times) == 0:
                    if len(stim_peak_amps) > 1:
                        epsp_heights.append(stim_peak_heights)
                        epsp_times.append(stim_peak_times)
                        epsp_amps.append(stim_peak_amps)
                        warnings.warn(f'Found {len(stim_peak_amps)} peak amplitudes at soma between {syn_spike_time} and {next_stim}', RuntimeWarning)
                    elif len(stim_peak_amps) == 0:
                        epsp_amps.append(0.0)
                    else:
                        epsp_heights.append(stim_peak_heights[0])
                        epsp_times.append(stim_peak_times[0])
                        epsp_amps.append(stim_peak_amps[0])

            stats = {
                'presyn_cell_labels': [],
                'syn_ids': [],
                'syn_weights': [],
                'soma_dists': [],
                'soma_dist': loaded_distance_bin,
                'soma_spike_times': soma_spike_times,
                'soma_spike_heights': soma_spike_heights,
                'num_soma_spikes': len(soma_spike_times),
                'EPSP_amps': epsp_amps if len(epsp_amps) > 0 else None,
                'EPSP_heights': epsp_heights if len(epsp_heights) > 0 else None,
                'EPSP_times': epsp_times if len(epsp_times) > 0 else None,
                'max_EPSP_amp': I.np.max(epsp_amps) if len(epsp_amps) > 0 else None,
                't': loaded_data['t'],
                'v': loaded_data['v'],
                'syn_auc_totals': [],
                'syn_aucs': [],
                'syn_auc_timeframes': [],
                'syn_auc_baselines': [],
                'syn_peak_heights': [],
                'syn_peak_times': [],
                'syn_max_peak_heights': [],
                'syn_dendrite_type': [],
                'v_syns': [],
            }

            # Synaptic features
            for syn_i in range(len(syn_ids)):
                for presyn_cell_type in cell.synapses.keys():
                    if presyn_cell_type == presyn_cell_labels[syn_i]:
                        for syn_id, syn in enumerate(cell.synapses[presyn_cell_type]):
                            if syn_id == syn_ids[syn_i]:
                                synapse_strength_celltype = [x for x in syn_weights.keys() if x in presyn_cell_type]
                                assert(len(synapse_strength_celltype) == 1)
                                synapse_strength_celltype = synapse_strength_celltype[0]
                                weight = loaded_syn_weights[synapse_strength_celltype]

                                syn_seg_id = int(syn.x * cell.sections[syn.secID].nseg)
                                if syn_seg_id == cell.sections[syn.secID].nseg:
                                    syn_seg_id -= 1

                                syn_peak_heights, syn_peak_times, _ = get_peaks(loaded_data['t'], loaded_data[f"v_syn_{len(stats['presyn_cell_labels'])}"], t_start=stim_start)
                                syn_aucs, syn_auc_timeframes, syn_baseline_v = get_syn_auc(loaded_data['t'], loaded_data[f"v_syn_{len(stats['presyn_cell_labels'])}"], t_start=stim_start, baseline='rest')

                                dendrite_type = 'basal'
                                if cell.sections[syn.secID].has_membrane('NaTa_t'):
                                    dendrite_type = 'apical'

                                stats['presyn_cell_labels'].append(presyn_cell_type)
                                stats['syn_ids'].append(syn_id)
                                stats['syn_weights'].append(weight)
                                stats['soma_dists'].append(compute_syn_distance(cell, syn))
                                stats['syn_auc_totals'].append(I.np.sum(syn_aucs) if len(syn_aucs) > 0 else 0)
                                stats['syn_aucs'].append(syn_aucs)
                                stats['syn_auc_timeframes'].append(syn_auc_timeframes)
                                stats['syn_auc_baselines'].append(syn_baseline_v)
                                stats['syn_peak_heights'].append(syn_peak_heights)
                                stats['syn_peak_times'].append(syn_peak_times)
                                stats['syn_max_peak_heights'].append(I.np.max(syn_peak_heights) if len(syn_peak_heights) > 0 else 0)
                                stats['syn_dendrite_type'].append(dendrite_type)
                                stats['v_syns'].append(loaded_data[f"v_syn_{len(stats['presyn_cell_labels'])-1}"])

            results_df = results_df.append(stats, ignore_index=True)

            if plot:
                axes[0].plot(loaded_data['t'], loaded_data['v'], label=f"Distance: {loaded_distance_bin}; Rep {loaded_rep}; {len(stats['syn_ids'])} synapses")  # soma

                for syn_i in range(len(stats['syn_ids'])):
                    axes[1].plot(loaded_data['t'], loaded_data[f'v_syn_{syn_i}'], label=f'Distance: {loaded_distance_bin}; Rep {loaded_rep}; Syn {syn_i}')  # synapse

    # Overall features
    results_df['num_syns'] = results_df['syn_ids'].apply(lambda x: len(x))
    results_df['mean_syn_weight'] = results_df['syn_weights'].apply(lambda x: I.np.mean(x))
    results_df['mean_soma_dist'] = results_df['soma_dists'].apply(lambda x: I.np.mean(x))
    results_df['max_syn_peak_heights'] = results_df['syn_max_peak_heights'].apply(lambda x: I.np.max(x))
    results_df['mean_syn_peak_heights'] = results_df['syn_max_peak_heights'].apply(lambda x: I.np.mean(x))
    results_df['max_syn_auc_total'] = results_df['syn_auc_totals'].apply(lambda x: I.np.max(x))
    results_df['mean_syn_auc_total'] = results_df['syn_auc_totals'].apply(lambda x: I.np.mean(x))
    # results_df['presyn_cell_type'] = results_df['presyn_cell_label'].apply(lambda x: next((cat for cat in syn_weights.keys() if cat in x), 'Unknown'))

    # Calculate the difference between somatic spike time and first synaptic peak time
    def calculate_delay(row):
        soma_spike_times = row['soma_spike_times']

        if len(soma_spike_times) == len(syn_spike_times):
            # Calculate delay between each pair of somatic and synaptic spike time
            return [soma - syn for soma, syn in zip(soma_spike_times, syn_spike_times)]
        elif len(soma_spike_times) > 0 and len(syn_spike_times) > 0:
            # Calculate delay between first somatic spike time and each synaptic spike time
            first_soma_spike = soma_spike_times[0]
            return [first_soma_spike - syn for syn in syn_spike_times]
        else:
            return None

    results_df['actual_delay'] = results_df.apply(calculate_delay, axis=1)

    results_df = results_df.sort_values(by=['soma_dist'])

    if plot:
        axes[0].set_title('Soma')
        axes[1].set_title('Synapse')
        # axes[2].set_title('Synapse')

        axes[1].set_xlabel('Time (ms)')
        axes[1].set_ylabel('Membrane potential (mV)')
        # axes[2].set_ylabel('Calcium concentration')

        # axes[0].legend()
        # axes[1].legend()

        %matplotlib inline
        I.plt.show()

    return results_df

### Plotting functions

In [ ]:
import neuron
h = neuron.h

import plotly.graph_objects as go

def plot_cell_3d_plotly(show_soma=False, show_synapses=False, recording_sites=[]):
    fig = go.Figure()

    max_range = 0
    min_range = 0
    soma_midpoint = None
    for sec in h.allsec():
        # Extract 3D coordinates for each segment in the section
        x = [h.x3d(i, sec=sec) for i in range(int(h.n3d(sec=sec)))]
        y = [h.y3d(i, sec=sec) for i in range(int(h.n3d(sec=sec)))]
        z = [h.z3d(i, sec=sec) for i in range(int(h.n3d(sec=sec)))]
        # diam = [h.diam3d(i, sec=sec) for i in range(int(h.n3d(sec=sec)))]
        # print(diam)

        max_coord = I.np.max([x, y, z])
        if max_coord > max_range:
            max_range = max_coord

        min_coord = I.np.min([x, y, z])
        if min_coord < min_range:
            min_range = min_coord

        if 'soma' in sec.name() and show_soma:
            midpoint_index = len(x) // 2
            soma_midpoint = (x[midpoint_index], y[midpoint_index], z[midpoint_index])

            fig.add_trace(
                go.Scatter3d(
                    x=[soma_midpoint[0]],
                    y=[soma_midpoint[1]],
                    z=[soma_midpoint[2]],
                    name='Soma',
                    # hovertemplate='<b>%{fullData.name}</b>' + '<extra></extra>',
                    opacity=0.5,
                    mode='markers',
                    marker=dict(
                        color='blue',
                        size=8
                    ),
                )
            )

        fig.add_trace(
            go.Scatter3d(
                x=x,
                y=y,
                z=z,
                name=sec.name(),
                # hovertemplate='<b>%{fullData.name}</b>' + '<extra></extra>',
                mode='lines',
                line=dict(
                    color='black',
                ),
            )
        )

        if show_synapses:
            point_processes = sec.psection()['point_processes']
            x_syn = []
            y_syn = []
            z_syn = []
            if 'GluSynapse' in point_processes:
                for syn in point_processes['GluSynapse']:
                    if isinstance(syn, type(h.GluSynapse)):
                        loc = syn.get_segment().x
                        x_syn.append(h.x3d(int(h.n3d(sec=sec) * loc), sec=sec))
                        y_syn.append(h.y3d(int(h.n3d(sec=sec) * loc), sec=sec))
                        z_syn.append(h.z3d(int(h.n3d(sec=sec) * loc), sec=sec))
            elif 'Exp2SynNMDA' in point_processes:
                for exp2syn_nmda in point_processes['Exp2SynNMDA']:
                    if isinstance(exp2syn_nmda, type(h.Exp2SynNMDA)):
                        loc = exp2syn_nmda.get_segment().x
                        x_syn.append(h.x3d(int(h.n3d(sec=sec) * loc), sec=sec))
                        y_syn.append(h.y3d(int(h.n3d(sec=sec) * loc), sec=sec))
                        z_syn.append(h.z3d(int(h.n3d(sec=sec) * loc), sec=sec))

            fig.add_trace(
                go.Scatter3d(
                    x=x_syn,
                    y=y_syn,
                    z=z_syn,
                    name=f'Synapse {sec.name()}',
                    # hovertemplate='<b>%{fullData.name}</b>' + '<extra></extra>',
                    opacity=0.5,
                    mode='markers',
                    marker=dict(
                        color='red',
                        size=3
                    ),
                )
            )

    if len(recording_sites) > 0:
        for recording_site in recording_sites:
            section = getattr(h, recording_site['section'], None)
            if section is None:
                raise ValueError(f'{recording_site["section"]} does not exist.')

            x_rec = h.x3d(int(h.n3d(sec=section) * recording_site['segment']), sec=section)
            y_rec = h.y3d(int(h.n3d(sec=section) * recording_site['segment']), sec=section)
            z_rec = h.z3d(int(h.n3d(sec=section) * recording_site['segment']), sec=section)
            # print(x_rec, y_rec, z_rec)

            fig.add_trace(
                go.Cone(
                    x=[x_rec],
                    y=[y_rec],
                    z=[z_rec],
                    u=[0],
                    v=[0],
                    w=[-10],
                    name=f'Recording {recording_site["section"]}',
                    sizemode='absolute',
                    sizeref=80,
                    anchor='tip',
                    showscale=False,
                    colorscale=[[0, recording_site['color']], [1, recording_site['color']]],
                    opacity=0.6
                )
            )

    axes_range = dict(range=[round(min_range - 10), round(max_range + 10)])

    fig.update_layout(
        scene=dict(
            xaxis_title='X Coordinate (μm)',
            yaxis_title='Y Coordinate (μm)',
            zaxis_title='Z Coordinate (μm)',
            xaxis=axes_range,
            yaxis=axes_range,
            zaxis=axes_range,
        ),
        scene_camera=dict(
            eye=dict(x=0, y=0, z=1)
        ),
        scene_aspectmode='cube',
        showlegend=False
    )

    return fig

In [ ]:
def plot_cell_3d_mpl(cell, synapses=[]):
    fig = I.plt.figure()
    ax_cell = fig.add_subplot(111, projection='3d')

    max_range = I.np.inf * (-1)
    min_range = I.np.inf
    for sec in cell.sections:
        # Extract 2D coordinates of each section
        x = I.np.array([h.x3d(i, sec=sec) for i in range(int(h.n3d(sec=sec)))])
        y = I.np.array([h.y3d(i, sec=sec) for i in range(int(h.n3d(sec=sec)))])
        z = I.np.array([h.z3d(i, sec=sec) for i in range(int(h.n3d(sec=sec)))])

        # Check for min/max coordinates to equalize axes ranges
        min_coord = I.np.min([x, y, z])
        if min_coord < min_range:
            min_range = min_coord

        max_coord = I.np.max([x, y, z])
        if max_coord > max_range:
            max_range = max_coord

        ax_cell.plot(x, y, z, c='k')

    if len(synapses) > 0:
        syn_pts = []
        for syn in synapses:
            # sec = cell.sections[syn.secID]
            syn_coord = cell.sections[syn.secID].pts[syn.ptID]
            syn_pts.append(syn_coord)

        syn_pts = I.np.array(syn_pts)
        syn_x = syn_pts[:, 0]
        syn_y = syn_pts[:, 1]
        syn_z = syn_pts[:, 2]

        ax_cell.scatter(syn_x, syn_y, syn_z, c='red', s=0.1)

    min_range = round(min_range - 10)
    max_range = round(max_range + 10)

    ax_cell.set_xlabel('X Coordinate (μm)')
    ax_cell.set_ylabel('Y Coordinate (μm)')
    ax_cell.set_zlabel('Z Coordinate (μm)')
    ax_cell.set_xlim(-500, 500)
    ax_cell.set_ylim(-500, 500)
    ax_cell.set_zlim(-500, 500)

    ax_cell.view_init(elev=10, azim=90, roll=0)

    %matplotlib inline
    I.plt.show()

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def plot_traces(df, inputs=[], x_lim=None, y_lim_soma=None, y_lim_syn=None, x_lim_soma_inset=None, y_lim_soma_inset=None, num_traces=5, random_seed=0):
    if len(inputs) == 0:
        fig, axes = I.plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(8, 10))
    elif len(inputs) == 1:
        fig, axes = I.plt.subplots(nrows=3, ncols=1, sharex=True, figsize=(8, 11), height_ratios=[1, 4, 4])
    elif len(inputs) == 2:
        fig, axes = I.plt.subplots(nrows=4, ncols=1, sharex=True, figsize=(8, 12), height_ratios=[1, 1, 4, 4])
    else:
        raise NotImplementedError('Currently, only 2 input signals can be plotted.')

    # for zoomed-in soma trace
    ax0_inset = inset_axes(axes[-2], width="30%", height="30%", loc="upper right")

    I.np.random.seed(random_seed)
    random_rows = I.np.random.choice(I.np.arange(df.shape[0]), num_traces, replace=False)
    print(random_rows)
    for i in random_rows:
        data = (df.iloc[i]['t'], df.iloc[i]['v'], df.iloc[i]['v_syn'])

        axes[-2].plot(data[0], data[1])  # soma
        inset_line, = ax0_inset.plot(data[0], data[1])  # soma zoomed
        if 'EPSP_heights' in df.columns and df.iloc[i]['EPSP_heights'] is not None:
            for epsp_index in range(len(df.iloc[i]['EPSP_heights'])):
                ax0_inset.scatter(df.iloc[i]['EPSP_times'][epsp_index], df.iloc[i]['EPSP_heights'][epsp_index], marker='x', color=inset_line.get_color())

        if x_lim_soma_inset != None:
            ax0_inset.set_xlim(x_lim_soma_inset)
        if y_lim_soma_inset != None:
            ax0_inset.set_ylim(y_lim_soma_inset)

        line, = axes[-1].plot(data[0], data[2])  # synapse
        for syn_auc_timeframe in df.iloc[i]['syn_auc_timeframes']:
            auc_time_index_1 = I.np.searchsorted(data[0], syn_auc_timeframe[0])
            auc_time_index_2 = I.np.searchsorted(data[0], syn_auc_timeframe[1])
            axes[-1].fill_between(
                data[0][auc_time_index_1:auc_time_index_2],
                data[2][auc_time_index_1:auc_time_index_2],
                I.np.full(auc_time_index_2-auc_time_index_1, df.iloc[i]['syn_auc_baseline']),
                color=line.get_color(), alpha=0.4
            )
        for syn_peak_index in range(len(df.iloc[i]['syn_peak_heights'])):
            axes[-1].scatter(df.iloc[i]['syn_peak_times'][syn_peak_index], df.iloc[i]['syn_peak_heights'][syn_peak_index], marker='x', color=line.get_color())

    # Plot input
    for input_index, input_dict in enumerate(inputs):
        y_input = I.np.zeros_like(data[0])
        for time in input_dict['stim_times']:
            i = I.np.argmin(I.np.abs(data[0] - time))  # Find index closest to time
            if 'duration' not in input_dict:
                y_input[i] = input_dict['amp']  # Set y-value to amp at corresponding time
            else:
                j = I.np.argmin(I.np.abs(data[0] - (time + input_dict['duration'])))
                y_input[i:j] = input_dict['amp']
        axes[input_index].plot(data[0], y_input)
        axes[input_index].set_ylabel(input_dict['label'])

    axes[-2].set_title('Soma')
    axes[-1].set_title('Synapse')

    axes[-2].set_ylabel('Membrane potential (mV)')
    if y_lim_soma != None:
        axes[-2].set_ylim(y_lim_soma)
    axes[-1].set_ylabel('Membrane potential (mV)')
    if y_lim_syn != None:
        axes[-1].set_ylim(y_lim_syn)

    axes[-1].set_xlabel('Time (ms)')
    if x_lim != None:
        axes[-1].set_xlim(x_lim)

    I.plt.tight_layout()

    %matplotlib inline
    I.plt.show()

In [ ]:
def plot_histogram(df, column, bins=20, xlabel=None, ylabel=None, labels=[]):
    if isinstance(df, list):
        # Sort all DataFrames by common keys
        # df_list_sorted = [df_temp.copy().sort_values(by=['presyn_cell_label', 'syn_id']) for df_temp in df]

        # # Get valid indices (rows without NaN in all DataFrames)
        # valid_indices = df_list_sorted[0].dropna(subset=[column]).index
        # for df_temp in df_list_sorted[1:]:
        #     valid_indices = valid_indices.intersection(df_temp.dropna(subset=[column]).index)

        # # Filter all DataFrames based on valid indices
        # df_list_no_none = [df_temp.loc[valid_indices].reset_index(drop=True) for df_temp in df_list_sorted]
        df_list_no_none = copy.deepcopy(df)

        # Flatten single-element lists in column1
        for df_temp in df_list_no_none:
            df_temp[column] = df_temp[column].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)
    else:
        df = df.copy()

        df = df.dropna(subset=[column])
        df[column] = df[column].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

    if isinstance(df, list):
        for i, temp_df in enumerate(df):
            I.plt.hist(temp_df[column], bins=bins, edgecolor='black', label=labels[i] if len(labels) > i else f'df{i}', alpha=0.4)
        I.plt.legend()
    else:
        I.plt.hist(df[column], bins=bins, edgecolor='black', label=labels[0] if len(labels) > 0 else 'df0')

    I.plt.xlabel(xlabel if xlabel!=None else column)
    I.plt.ylabel(ylabel if ylabel!=None else 'Num entries')

    %matplotlib inline
    I.plt.show()

In [ ]:
def plot_count(df, column, kind='bar', xlabel=None, ylabel=None):
    df_no_none = df.dropna(subset=[column])
    df_no_none[column] = df_no_none[column].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

    value_counts = df_no_none[column].value_counts()
    value_counts = value_counts.sort_index(ascending=True)

    value_counts.plot(kind=kind, edgecolor='black')

    x_tick_labels = value_counts.index
    rounded_labels = [f'{label:.3f}' for label in x_tick_labels]
    I.plt.xticks(ticks=range(len(rounded_labels)), labels=rounded_labels, rotation=0)

    I.plt.xlabel(xlabel if xlabel!=None else column)
    I.plt.ylabel(ylabel if ylabel!=None else 'Num entries')

    %matplotlib inline
    I.plt.show()

In [ ]:
def plot_scatter(df, column1, column2, color_column=None, ctype='cont', cmap='viridis', xlabel=None, ylabel=None, clabel=None):
    df_no_none = df.dropna(subset=[column1, column2])
    df_no_none[column1] = df_no_none[column1].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)
    df_no_none[column2] = df_no_none[column2].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

    if color_column != None:
        if ctype == 'cont':
            scatter = I.plt.scatter(df_no_none[column1], df_no_none[column2], c=df_no_none[color_column], cmap=cmap, edgecolor='black')

            cbar = I.plt.colorbar(scatter)
            cbar.set_label(clabel if clabel!= None else color_column)
        elif ctype == 'type':
            color_types = df_no_none[color_column].unique()
            colors = I.plt.cm.get_cmap('Set1', len(color_types))

            # Create a dictionary to map each type to a color
            color_mapping = {color_type: i for i, color_type in enumerate(color_types)}
            color_indices = df_no_none[color_column].map(color_mapping)

            scatter = I.plt.scatter(df_no_none[column1], df_no_none[column2], c=color_indices, cmap=colors, edgecolor='black')
            
            # Create a custom legend for the categories
            handles = [I.plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors(i), markersize=10) for i in range(len(color_types))]
            I.plt.legend(handles, color_types, title=clabel if clabel!= None else color_column)
        else:
            raise ValueError(f'ctype can only be "cont" or "type". Received {ctype}')
    else:
        scatter = I.plt.scatter(df_no_none[column1], df_no_none[column2], edgecolor='black')

    I.plt.xlabel(xlabel if xlabel!=None else column1)
    I.plt.ylabel(ylabel if ylabel!=None else column2)

    %matplotlib inline
    I.plt.show()

In [ ]:
def plot_scatter_subtract(dfs, column1, column2, color_column=None, ctype='cont', cmap='viridis', xlabel=None, ylabel=None, clabel=None, labels=[]):
    # Sort all DataFrames
    df_list_sorted = [df_temp.copy().sort_values(by=['presyn_cell_label', 'syn_id']) for df_temp in dfs]

    # Get valid indices (rows without NaN in all DataFrames)
    valid_indices = df_list_sorted[0].dropna(subset=[column1, column2]).index
    for df_temp in df_list_sorted[1:]:
        valid_indices = valid_indices.intersection(df_temp.dropna(subset=[column1, column2]).index)

    # Filter all DataFrames based on valid indices
    df_list_no_none = [df_temp.loc[valid_indices].reset_index(drop=True) for df_temp in df_list_sorted]

    # Flatten single-element lists in column1
    for df_temp in df_list_no_none:
        df_temp[column1] = df_temp[column1].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)
        df_temp[column2] = df_temp[column2].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

    # Sort all DataFrames again
    df_list_sorted = [df_temp.sort_values(by=['presyn_cell_label', 'syn_id']) for df_temp in df_list_no_none]

    # Make sure all x-values are the same across all dataframes
    for df_temp in df_list_no_none:
        for df_temp2 in df_list_no_none:
            I.pd.testing.assert_series_equal(df_temp[column1], df_temp2[column1], check_dtype=False)

    # Subtract dataframes from each other for y-values
    y_values = df_list_sorted[0][column2]
    for i in range(1, len(df_list_no_none)):
        y_values -= df_list_sorted[i][column2]

    if color_column != None:
        if ctype == 'cont':
            scatter = I.plt.scatter(df_list_sorted[0][column1], y_values, c=df_list_sorted[0][color_column], cmap=cmap, edgecolor='black')

            cbar = I.plt.colorbar(scatter)
            cbar.set_label(clabel if clabel!= None else color_column)
        elif ctype == 'type':
            color_types = df_list_sorted[0][color_column].unique()
            colors = I.plt.cm.get_cmap('Set1', len(color_types))

            # Create a dictionary to map each type to a color
            color_mapping = {color_type: i for i, color_type in enumerate(color_types)}
            color_indices = df_list_sorted[0][color_column].map(color_mapping)

            scatter = I.plt.scatter(df_list_sorted[0][column1], y_values, c=color_indices, cmap=colors, edgecolor='black')
            
            # Create a custom legend for the categories
            handles = [I.plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors(i), markersize=10) for i in range(len(color_types))]
            I.plt.legend(handles, color_types, title=clabel if clabel!= None else color_column)
        else:
            raise ValueError(f'ctype can only be "cont" or "type". Received {ctype}')
    else:
        scatter = I.plt.scatter(df_list_sorted[0][column1], y_values, edgecolor='black')

    if len(labels) > 0:
        title = f'{labels[0]}'
        for i in range(1, len(labels)):
            title += f' - {labels[i]}'
        I.plt.title(title)
    I.plt.xlabel(xlabel if xlabel!=None else column1)
    I.plt.ylabel(ylabel if ylabel!=None else column2)

    # I.plt.xlim(max_range[0]-(max_range[1]-max_range[0])*0.1, max_range[1]+(max_range[1]-max_range[0])*0.1)
    # I.plt.ylim(max_range[0]-(max_range[1]-max_range[0])*0.1, max_range[1]+(max_range[1]-max_range[0])*0.1)

    %matplotlib inline
    I.plt.show()

In [ ]:
def plot_scatter_compare(dfs, column1, column2, color_column=None, ctype='cont', cmap='viridis', xlabel=None, ylabel=None, clabel=None, labels=[]):
    # Remove nan values in column1 and column2 in all DataFrames
    df_list_no_none = [df_temp.dropna(subset=[column1, column2]) for df_temp in dfs]

    # Flatten single-element lists in column1
    for df_i, df_temp in enumerate(df_list_no_none):
        df_temp[column1] = df_temp[column1].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)
        df_temp[column2] = df_temp[column2].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

        if color_column != None:
            if ctype == 'cont':
                scatter = I.plt.scatter(df_temp[column1], df_temp[column2], c=df_temp[color_column], cmap=cmap, edgecolor='black', label=labels[df_i] if df_i < len(labels) else f'df{df_i}')

                cbar = I.plt.colorbar(scatter)
                cbar.set_label(clabel if clabel!= None else color_column)
            elif ctype == 'type':
                color_types = df_temp[color_column].unique()
                colors = I.plt.cm.get_cmap('Set1', len(color_types))

                # Create a dictionary to map each type to a color
                color_mapping = {color_type: i for i, color_type in enumerate(color_types)}
                color_indices = df_temp[color_column].map(color_mapping)

                scatter = I.plt.scatter(df_temp[column1], df_temp[column2], c=color_indices, cmap=colors, edgecolor='black', label=labels[df_i] if df_i < len(labels) else f'df{df_i}')
                
                # Create a custom legend for the categories
                handles = [I.plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors(i), markersize=10) for i in range(len(color_types))]
                I.plt.legend(handles, color_types, title=clabel if clabel!= None else color_column)
            else:
                raise ValueError(f'ctype can only be "cont" or "type". Received {ctype}')
        else:
            scatter = I.plt.scatter(df_temp[column1], df_temp[column2], edgecolor='black', label=labels[df_i] if df_i < len(labels) else f'df{df_i}')

    I.plt.xlabel(xlabel if xlabel!=None else column1)
    I.plt.ylabel(ylabel if ylabel!=None else column2)

    I.plt.legend()

    # I.plt.xlim(max_range[0]-(max_range[1]-max_range[0])*0.1, max_range[1]+(max_range[1]-max_range[0])*0.1)
    # I.plt.ylim(max_range[0]-(max_range[1]-max_range[0])*0.1, max_range[1]+(max_range[1]-max_range[0])*0.1)

    %matplotlib inline
    I.plt.show()

In [ ]:
def plot_violin(dfs, column, ylabel=None, labels=[]):
    # Sort all DataFrames
    df_list_sorted = [df_temp.copy().sort_values(by=['presyn_cell_label', 'syn_id']) for df_temp in dfs]

    # Get valid indices (rows without NaN in all DataFrames)
    valid_indices = df_list_sorted[0].dropna(subset=[column]).index
    for df_temp in df_list_sorted[1:]:
        valid_indices = valid_indices.intersection(df_temp.dropna(subset=[column]).index)

    # Filter all DataFrames based on valid indices
    df_list_no_none = [df_temp.loc[valid_indices].reset_index(drop=True) for df_temp in df_list_sorted]

    # Flatten single-element lists in column1
    for df_temp in df_list_no_none:
        df_temp[column] = df_temp[column].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

    data = [df_temp[column] for df_temp in df_list_no_none]

    fig, ax = I.plt.subplots()
    ax.violinplot(data, showmeans=True, showmedians=False, showextrema=False)

    # Set custom categorical labels for the x-axis
    ax.set_xticks(I.np.arange(1, len(data)+1))
    ax.set_xticklabels(labels if len(labels) == len(data) else [f'df{i}' for i in range(len(data))], rotation=45)

    ax.set_ylabel(ylabel if ylabel!=None else column)

    %matplotlib inline
    I.plt.show()

In [ ]:
def plot_syn_v_scatter(df):   
    # plot_histogram(df, 'syn_max_peak_height', xlabel='Max Vm at synapse (mV)', ylabel='Num synapses')
    # plot_histogram(df, 'syn_auc_total', xlabel='Vm AUC at synapse', ylabel='Num synapses')

    plot_scatter(df, 'soma_dist', 'syn_max_peak_height', 'syn_weight', xlabel='Soma distance (µm)', ylabel='Max Vm at synapse (mV)', clabel='Synaptic weight')
    plot_scatter(df, 'soma_dist', 'syn_max_peak_height', 'presyn_cell_type', ctype='type', xlabel='Soma distance (µm)', ylabel='Max Vm at synapse (mV)', clabel='Presynaptic cell type')

    plot_scatter(df, 'soma_dist', 'syn_auc_total', 'syn_weight', xlabel='Soma distance (µm)', ylabel='Vm AUC at synapse', clabel='Synaptic weight')
    plot_scatter(df, 'soma_dist', 'syn_auc_total', 'presyn_cell_type', ctype='type', xlabel='Soma distance (µm)', ylabel='Vm AUC at synapse', clabel='Presynaptic cell type')

    plot_scatter(df, 'syn_max_peak_height', 'syn_auc_total', 'soma_dist', xlabel='Max Vm at synapse (mV)', ylabel='Vm AUC at synapse', clabel='Soma distance (µm)')

    # plot_scatter(df, 'soma_dist', 'syn_cai_max_peak', 'syn_max_peak_height', xlabel='Soma distance (µm)', ylabel='Peak calcium concentration', clabel='Max Vm at synapse (mV)')
    # plot_scatter(df, 'soma_dist', 'syn_cai_auc_total', 'syn_auc_total', xlabel='Soma distance (µm)', ylabel='Calcium concentration AUC', clabel='Vm AUC at synapse')

In [ ]:
def plot_syn_v_hist(dfs, labels=[]):
    plot_histogram(dfs, 'syn_max_peak_height', xlabel='Max Vm at synapse (mV)', ylabel='Num synapses', labels=labels)
    plot_histogram(dfs, 'syn_auc_total', xlabel='Vm AUC at synapse', ylabel='Num synapses', labels=labels)

In [ ]:
def plot_syn_v_violin(dfs, labels=[]):
    plot_violin(dfs, 'syn_max_peak_height', ylabel='Max Vm at synapse (mV)', labels=labels)
    plot_violin(dfs, 'syn_auc_total', ylabel='Vm AUC at synapse', labels=labels)

In [ ]:
def plot_syn_v_comparisons(dfs, labels=[]):
    plot_scatter_subtract(dfs, 'soma_dist', 'syn_max_peak_height', 'syn_weight', xlabel='Soma distance (µm)', ylabel='Max Vm at synapse (mV)', clabel='Synaptic weight', labels=labels)
    plot_scatter_subtract(dfs, 'soma_dist', 'syn_auc_total', 'syn_weight', xlabel='Soma distance (µm)', ylabel='Vm AUC at synapse', clabel='Synaptic weight', labels=labels)

In [ ]:
def plot_extracellular_syn_v(df, c_column='mean_syn_weight', c_label='Mean synaptic weight'):
    plot_scatter(df, 'mean_soma_dist', 'num_syns', c_column, xlabel='Soma distance (µm)', ylabel='Num synapses', clabel=c_label)

    # plot_scatter(df, 'mean_soma_dist', 'max_syn_peak_heights', xlabel='Soma distance (µm)', ylabel='Max Vm peak at synapse')
    plot_scatter(df, 'mean_soma_dist', 'mean_syn_peak_heights', c_column, xlabel='Soma distance (µm)', ylabel='Mean Vm peak at synapse', clabel=c_label)

    # plot_scatter(df, 'mean_soma_dist', 'max_syn_auc_total', xlabel='Soma distance (µm)', ylabel='Max Vm AUC at synapse')
    plot_scatter(df, 'mean_soma_dist', 'mean_syn_auc_total', c_column, xlabel='Soma distance (µm)', ylabel='Mean Vm AUC at synapse', clabel=c_label)

In [ ]:
def plot_extracellular_syn_v_compare(dfs, labels=[]):
    plot_scatter_compare(dfs, 'mean_soma_dist', 'num_syns', xlabel='Soma distance (µm)', ylabel='Num synapses', labels=labels)

    plot_scatter_compare(dfs, 'mean_soma_dist', 'mean_syn_peak_heights', xlabel='Soma distance (µm)', ylabel='Mean Vm peak at synapse', labels=labels)

    plot_scatter_compare(dfs, 'mean_soma_dist', 'mean_syn_auc_total', xlabel='Soma distance (µm)', ylabel='Mean Vm AUC at synapse', labels=labels)

## Sjöström et al., 2001

### 1 Hz

#### Synaptic stimulation only

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '1Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'GlutamateUncaging')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['GlutamateUncaging.stim.syn_stim_times'] = [400]
protocol_params['GlutamateUncaging.run.tStop'] = 600

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'GlutamateUncaging', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

synapses_only_1Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
synapses_only_1Hz_df

In [ ]:
plot_traces(
    synapses_only_1Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['GlutamateUncaging.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
    ],
    x_lim=(385, 460), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411)  # , y_lim_soma_inset=(-76, -74)
)

In [ ]:
# # Removing synapse segments without cai should only leave apical synapses
# synapses_only_1Hz_df_df_copy = synapses_only_1Hz_df.copy()

# synapses_only_1Hz_df = synapses_only_1Hz_df.dropna(subset=['peak_cai'])
# synapses_only_1Hz_df

In [ ]:
plot_histogram(synapses_only_1Hz_df, 'max_EPSP_amp', xlabel='EPSP amplitude (mV)', ylabel='Num synapses')
    
plot_scatter(synapses_only_1Hz_df, 'soma_dist', 'max_EPSP_amp', 'syn_weight', xlabel='Soma distance (µm)', ylabel='EPSP amplitude (mV)', clabel='Synaptic weight')
plot_scatter(synapses_only_1Hz_df, 'soma_dist', 'max_EPSP_amp', 'presyn_cell_type', ctype='type', xlabel='Soma distance (µm)', ylabel='EPSP amplitude (mV)', clabel='Presynaptic cell type')

plot_scatter(synapses_only_1Hz_df, 'syn_max_peak_height', 'max_EPSP_amp', 'soma_dist', xlabel='Max Vm at synapse (mV)', ylabel='EPSP amplitude (mV)', clabel='Soma distance (µm)')
plot_scatter(synapses_only_1Hz_df, 'syn_auc_total', 'max_EPSP_amp', 'soma_dist', xlabel='Vm AUC at synapse', ylabel='EPSP amplitude (mV)', clabel='Soma distance (µm)')

In [ ]:
plot_syn_v_scatter(synapses_only_1Hz_df)

#### Somatic current injection only

##### Find somatic current stimulation parameters

In [ ]:
from biophysics_fitting.ephys import find_crossing

amps = I.np.arange(1, 5.1, step=0.1)

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

breaker = False
for amp in amps:
    amp = round(amp, 1)

    cell_params = loaded_cell_params.loc[biophysics_id].copy()

    cell_params['SomaticCurrentInjection.stim.current_amp'] = amp
    cell_params['SomaticCurrentInjection.stim.current_inj_times'] = [400]
    cell_params['SomaticCurrentInjection.stim.current_duration'] = 2
    cell_params['SomaticCurrentInjection.run.tStop'] = 600

    simulated_cell, param = loaded_simulator.get_simulated_cell(cell_params, 'SomaticCurrentInjection')

    spikes = find_crossing(simulated_cell.soma.recVList[0], thresh=0)

    if len(spikes[0]) > len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        print('Multiple spikes')
        # I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        break
    elif len(spikes[0]) == len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        spike_amp = amp
        # time_to_spike = simulated_cell.tVec[spikes[0][0]] - cell_params['SomaticCurrentInjection.stim.current_inj_times'][0]
        print(spike_amp)
        print(simulated_cell.tVec[spikes[0][0]])
        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)

        # Run 1 extra increment of 0.1 when reaching "rheobase"
        if breaker:
            break
        breaker = True
    else:
        continue

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.xlim(left=390)
I.plt.ylabel('Membrane potential (mV)')
I.plt.legend()
I.plt.show()

##### Somatic current injection control

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '1Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'SomaticCurrentInjection')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['SomaticCurrentInjection.stim.current_amp'] = spike_amp
protocol_params['SomaticCurrentInjection.stim.current_inj_times'] = [400]
protocol_params['SomaticCurrentInjection.stim.current_duration'] = 2
protocol_params['SomaticCurrentInjection.run.tStop'] = 600

In [ ]:
run_protocol_soma_only(simulation_results_dir, loaded_simulator, loaded_cell_params.loc[biophysics_id], protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

soma_only_1Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
soma_only_1Hz_df

In [ ]:
plot_traces(
    soma_only_1Hz_df,
    inputs=[
        {
            'stim_times': cell_params['SomaticCurrentInjection.stim.current_inj_times'],
            'label': 'Somatic current (nA)',
            'amp': cell_params['SomaticCurrentInjection.stim.current_amp'],
            'duration': cell_params['SomaticCurrentInjection.stim.current_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, None),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
# from visualize.cell_morphology_visualizer import CellMorphologyVisualizer

# images_path = I.os.path.join(results_dir, 'soma_spike_animation_3d')

# # To remake the visualization, uncomment this code
# if I.os.path.exists(images_path):
#     I.shutil.rmtree(images_path)

# cmv = CellMorphologyVisualizer(simulated_cell)
# cmv.population_to_color_dict['inactive'] = "#f0f0f0"  # add color for inactive
# # cmv.population_to_color_dict['Generic'] = "red"

# cmv.animation(
#     images_path=images_path, 
#     color="voltage",
#     client=client, 
#     t_start=400-10, t_stop=440, t_step=0.2
# )

In [ ]:
filtered_df = soma_only_1Hz_df[soma_only_1Hz_df['syn_peak_heights'].apply(lambda x: len(x) > 1)]
filtered_df

In [ ]:
plot_traces(filtered_df, x_lim=(385, 460), y_lim_syn=(-80, None), x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74))

plot_histogram(filtered_df, 'soma_dist', xlabel='Soma distance (µm)', ylabel='Num synapses')

In [ ]:
plot_syn_v_scatter(soma_only_1Hz_df)

In [ ]:
plot_syn_v_comparisons([soma_only_1Hz_df, synapses_only_1Hz_df], labels=['Glutamate uncaging', 'Somatic current injection'])

In [ ]:
merged = soma_only_1Hz_df.merge(synapses_only_1Hz_df, on=['presyn_cell_label', 'syn_id'], suffixes=('_soma', '_synapses'))
merged['peak_height_diff'] = merged['syn_max_peak_height_soma'] - merged['syn_max_peak_height_synapses']

merged = merged[merged['peak_height_diff'].apply(lambda x: x <= 0)]
merged

In [ ]:
plot_scatter(merged, 'soma_dist_soma', 'peak_height_diff', 'syn_dendrite_type_soma', ctype='type', xlabel='Soma distance (µm)', ylabel='Max Vm at synapse (mV)', clabel='Dendritic type')

#### +10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '1Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'plus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [410]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
# protocol_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
# protocol_params['STDP.stim.pairing_delay'] = 10
protocol_params['STDP.run.tStop'] = 600

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus10_1Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus10_1Hz_df

In [ ]:
plot_traces(
    plus10_1Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
# # Removing synapse segments without cai should only leave apical synapses
# plus10_1Hz_df_copy = plus10_1Hz_df.copy()

# plus10_1Hz_df = plus10_1Hz_df.dropna(subset=['peak_cai'])
# plus10_1Hz_df

In [ ]:
# plot_traces(filtered_df, x_lim=(385, 460), y_lim_syn=(-80, -25), x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74))

In [ ]:
all_delays = [delay for delays in plus10_1Hz_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus10_1Hz_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_syn_v_scatter(plus10_1Hz_df)

In [ ]:
# plot_syn_v_comparisons([plus10_1Hz_df, synapses_only_1Hz_df], labels=['+10 ms delay', 'Glutamate uncaging'])

In [ ]:
# plot_syn_v_comparisons([plus10_1Hz_df, soma_only_1Hz_df], labels=['+10 ms delay', 'Somatic current injection'])

In [ ]:
plot_syn_v_comparisons([plus10_1Hz_df, soma_only_1Hz_df, synapses_only_1Hz_df], labels=['+10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

In [ ]:
# filtered_df = plus10_1Hz_df[plus10_1Hz_df['presyn_cell_type'].apply(lambda x: 'L5' in x)]
filtered_df = plus10_1Hz_df[plus10_1Hz_df['syn_dendrite_type'].apply(lambda x: x == 'apical')]
# # filtered_df = plus10_1Hz_df[plus10_1Hz_df['EPSP_amps'].apply(lambda x: x[0] == 0.0)]
# # filtered_df = plus10_1Hz_df[plus10_1Hz_df['soma_spike_heights'].apply(lambda x: x > 60)]
# # filtered_df = plus10_1Hz_df[plus10_1Hz_df['soma_spike_times'].apply(lambda x: len(x) > 1)]
filtered_df

In [ ]:
plot_syn_v_scatter(filtered_df)

#### -10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '1Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'minus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [390]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
# protocol_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
# protocol_params['STDP.stim.pairing_delay'] = -10
protocol_params['STDP.run.tStop'] = 600

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus10_1Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus10_1Hz_df

In [ ]:
plot_traces(
    minus10_1Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-70, -55)
)

In [ ]:
all_delays = [delay for delays in minus10_1Hz_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

# df = minus10_1Hz_df.copy()

# df = df.dropna(subset=['actual_delay'])
# df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

# I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

# I.plt.xlabel('Delay between somatic and synaptic spike')
# I.plt.ylabel('Num synapses')

# %matplotlib inline
# I.plt.show()

In [ ]:
plot_syn_v_scatter(minus10_1Hz_df)

In [ ]:
plot_syn_v_comparisons([minus10_1Hz_df, soma_only_1Hz_df, synapses_only_1Hz_df], labels=['-10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

#### Analysis

In [ ]:
plot_syn_v_hist([synapses_only_1Hz_df, soma_only_1Hz_df, plus10_1Hz_df, minus10_1Hz_df], labels=['Glutamate uncaging', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_comparisons([plus10_1Hz_df, minus10_1Hz_df], labels=['+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_violin([synapses_only_1Hz_df, soma_only_1Hz_df, plus10_1Hz_df, minus10_1Hz_df], labels=['Glutamate uncaging', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

##### Sjöström (only L5TT-L5TT)

In [ ]:
filtered_soma_only_1Hz_df = soma_only_1Hz_df[soma_only_1Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_soma_only_1Hz_df = filtered_soma_only_1Hz_df[filtered_soma_only_1Hz_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
filtered_synapses_only_1Hz_df = synapses_only_1Hz_df[synapses_only_1Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_synapses_only_1Hz_df = filtered_synapses_only_1Hz_df[filtered_synapses_only_1Hz_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
filtered_plus10_1Hz_df = plus10_1Hz_df[plus10_1Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_plus10_1Hz_df = filtered_plus10_1Hz_df[filtered_plus10_1Hz_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
filtered_minus10_1Hz_df = minus10_1Hz_df[minus10_1Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_minus10_1Hz_df = filtered_minus10_1Hz_df[filtered_minus10_1Hz_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_plus10_1Hz_df

In [ ]:
plot_syn_v_hist([filtered_synapses_only_1Hz_df, filtered_soma_only_1Hz_df, filtered_plus10_1Hz_df, filtered_minus10_1Hz_df], labels=['Synaptic stimulation', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
plot_traces(
    filtered_plus10_1Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(395, 460), y_lim_syn=(-80, 20),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74),
    num_traces=10
)

In [ ]:
plot_syn_v_scatter(filtered_plus10_1Hz_df)

In [ ]:
plot_syn_v_scatter(filtered_minus10_1Hz_df)

In [ ]:
plot_syn_v_comparisons([filtered_plus10_1Hz_df, filtered_soma_only_1Hz_df, filtered_synapses_only_1Hz_df], labels=['+10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

In [ ]:
plot_syn_v_comparisons([filtered_plus10_1Hz_df, filtered_minus10_1Hz_df], labels=['+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_violin([filtered_synapses_only_1Hz_df, filtered_soma_only_1Hz_df, filtered_plus10_1Hz_df, filtered_minus10_1Hz_df], labels=['Synaptic stimulation', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

### 10 Hz

#### Synaptic stimulation only

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '10Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'GlutamateUncaging')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['GlutamateUncaging.stim.syn_stim_times'] = I.np.arange(start=400, stop=900, step=100)
protocol_params['GlutamateUncaging.run.tStop'] = 1000

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'GlutamateUncaging', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

synapses_only_10Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
synapses_only_10Hz_df

In [ ]:
plot_traces(
    synapses_only_10Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['GlutamateUncaging.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
    ],
    x_lim=(385, 460), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411)  # , y_lim_soma_inset=(-76, -74)
)

In [ ]:
plot_syn_v_scatter(synapses_only_10Hz_df)

#### Somatic current injection only

##### Find somatic current stimulation parameters

In [ ]:
from biophysics_fitting.ephys import find_crossing

amps = I.np.arange(1, 5.1, step=0.1)

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

breaker = False
for amp in amps:
    amp = round(amp, 1)

    cell_params = loaded_cell_params.loc[biophysics_id].copy()

    cell_params['SomaticCurrentInjection.stim.current_amp'] = amp
    cell_params['SomaticCurrentInjection.stim.current_inj_times'] = I.np.arange(start=400, stop=900, step=100)
    cell_params['SomaticCurrentInjection.stim.current_duration'] = 2
    cell_params['SomaticCurrentInjection.run.tStop'] = 1000

    simulated_cell, param = loaded_simulator.get_simulated_cell(cell_params, 'SomaticCurrentInjection')

    spikes = find_crossing(simulated_cell.soma.recVList[0], thresh=0)

    if len(spikes[0]) > len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        print('Multiple spikes')
        # I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        break
    elif len(spikes[0]) == len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        spike_amp = amp
        # time_to_spike = simulated_cell.tVec[spikes[0][0]] - cell_params['SomaticCurrentInjection.stim.current_inj_times'][0]
        print(spike_amp)
        print(simulated_cell.tVec[spikes[0][0]])
        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)

        # Run 1 extra increment of 0.1 when reaching "rheobase"
        if breaker:
            break
        breaker = True
    else:
        continue

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.xlim(left=390)
I.plt.ylabel('Membrane potential (mV)')
I.plt.legend()
I.plt.show()

##### Somatic current injection control

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '10Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'SomaticCurrentInjection')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['SomaticCurrentInjection.stim.current_amp'] = spike_amp
protocol_params['SomaticCurrentInjection.stim.current_inj_times'] = I.np.arange(start=400, stop=900, step=100)
protocol_params['SomaticCurrentInjection.stim.current_duration'] = 2
protocol_params['SomaticCurrentInjection.run.tStop'] = 1000

In [ ]:
run_protocol_soma_only(simulation_results_dir, loaded_simulator, loaded_cell_params.loc[biophysics_id], protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

soma_only_10Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
soma_only_10Hz_df

In [ ]:
plot_traces(
    soma_only_10Hz_df,
    inputs=[
        {
            'stim_times': cell_params['SomaticCurrentInjection.stim.current_inj_times'],
            'label': 'Somatic current (nA)',
            'amp': cell_params['SomaticCurrentInjection.stim.current_amp'],
            'duration': cell_params['SomaticCurrentInjection.stim.current_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, None),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
# from visualize.cell_morphology_visualizer import CellMorphologyVisualizer

# images_path = I.os.path.join(results_dir, 'soma_spike_animation_3d')

# # To remake the visualization, uncomment this code
# if I.os.path.exists(images_path):
#     I.shutil.rmtree(images_path)

# cmv = CellMorphologyVisualizer(simulated_cell)
# cmv.population_to_color_dict['inactive'] = "#f0f0f0"  # add color for inactive
# # cmv.population_to_color_dict['Generic'] = "red"

# cmv.animation(
#     images_path=images_path, 
#     color="voltage",
#     client=client, 
#     t_start=400-10, t_stop=440, t_step=0.2
# )

In [ ]:
plot_syn_v_scatter(soma_only_10Hz_df)

#### +10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '10Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'plus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = I.np.arange(start=400, stop=900, step=100)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = I.np.arange(start=410, stop=910, step=100)
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
# protocol_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
# protocol_params['STDP.stim.pairing_delay'] = 10
protocol_params['STDP.run.tStop'] = 1000

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus10_10Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus10_10Hz_df

In [ ]:
plot_traces(
    plus10_10Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
all_delays = [delay for delays in plus10_10Hz_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus10_10Hz_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_syn_v_scatter(plus10_10Hz_df)

In [ ]:
plot_syn_v_comparisons([plus10_10Hz_df, soma_only_10Hz_df, synapses_only_10Hz_df], labels=['+10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

#### -10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '10Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'minus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = I.np.arange(start=400, stop=900, step=100)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = I.np.arange(start=390, stop=890, step=100)
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
# protocol_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
# protocol_params['STDP.stim.pairing_delay'] = -10
protocol_params['STDP.run.tStop'] = 1000

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus10_10Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus10_10Hz_df

In [ ]:
plot_traces(
    minus10_10Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-70, -55)
)

In [ ]:
all_delays = [delay for delays in minus10_10Hz_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = minus10_10Hz_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_syn_v_scatter(minus10_10Hz_df)

In [ ]:
plot_syn_v_comparisons([minus10_10Hz_df, soma_only_10Hz_df, synapses_only_10Hz_df], labels=['-10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

#### Analysis

In [ ]:
plot_syn_v_hist([synapses_only_10Hz_df, soma_only_10Hz_df, plus10_10Hz_df, minus10_10Hz_df], labels=['Glutamate uncaging', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_comparisons([plus10_10Hz_df, minus10_10Hz_df], labels=['+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_violin([synapses_only_10Hz_df, soma_only_10Hz_df, plus10_10Hz_df, minus10_10Hz_df], labels=['Glutamate uncaging', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

##### Sjöström (only L5TT-L5TT)

In [ ]:
filtered_soma_only_10Hz_df = soma_only_10Hz_df[soma_only_10Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_synapses_only_10Hz_df = synapses_only_10Hz_df[synapses_only_10Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_plus10_10Hz_df = plus10_10Hz_df[plus10_10Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_minus10_10Hz_df = minus10_10Hz_df[minus10_10Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]

filtered_plus10_10Hz_df

In [ ]:
plot_syn_v_hist([filtered_synapses_only_10Hz_df, filtered_soma_only_10Hz_df, filtered_plus10_10Hz_df, filtered_minus10_10Hz_df], labels=['Synaptic stimulation', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_scatter(filtered_plus10_10Hz_df)

In [ ]:
plot_syn_v_scatter(filtered_plus10_10Hz_df)

In [ ]:
plot_syn_v_scatter(filtered_minus10_10Hz_df)

In [ ]:
plot_syn_v_comparisons([filtered_plus10_10Hz_df, filtered_minus10_10Hz_df], labels=['+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_violin([filtered_synapses_only_10Hz_df, filtered_soma_only_10Hz_df, filtered_plus10_10Hz_df, filtered_minus10_10Hz_df], labels=['Synaptic stimulation', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

### 20 Hz

#### Synaptic stimulation only

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '20Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'GlutamateUncaging')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['GlutamateUncaging.stim.syn_stim_times'] = I.np.arange(start=400, stop=650, step=50)
protocol_params['GlutamateUncaging.run.tStop'] = 800

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'GlutamateUncaging', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

synapses_only_20Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
synapses_only_20Hz_df

In [ ]:
plot_traces(
    synapses_only_20Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['GlutamateUncaging.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
    ],
    x_lim=(385, 460), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411)  # , y_lim_soma_inset=(-76, -74)
)

In [ ]:
plot_syn_v_scatter(synapses_only_20Hz_df)

#### Somatic current injection only

##### Find somatic current stimulation parameters

In [ ]:
from biophysics_fitting.ephys import find_crossing

amps = I.np.arange(1, 5.1, step=0.1)

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

breaker = False
for amp in amps:
    amp = round(amp, 1)

    cell_params = loaded_cell_params.loc[biophysics_id].copy()

    cell_params['SomaticCurrentInjection.stim.current_amp'] = amp
    cell_params['SomaticCurrentInjection.stim.current_inj_times'] = I.np.arange(start=400, stop=650, step=50)
    cell_params['SomaticCurrentInjection.stim.current_duration'] = 2
    cell_params['SomaticCurrentInjection.run.tStop'] = 800

    simulated_cell, param = loaded_simulator.get_simulated_cell(cell_params, 'SomaticCurrentInjection')

    spikes = find_crossing(simulated_cell.soma.recVList[0], thresh=0)

    if len(spikes[0]) > len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        print('Multiple spikes')
        # I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        break
    elif len(spikes[0]) == len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        spike_amp = amp
        # time_to_spike = simulated_cell.tVec[spikes[0][0]] - cell_params['SomaticCurrentInjection.stim.current_inj_times'][0]
        print(spike_amp)
        print(simulated_cell.tVec[spikes[0][0]])
        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)

        # Run 1 extra increment of 0.1 when reaching "rheobase"
        if breaker:
            break
        breaker = True
    else:
        continue

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.xlim(left=390)
I.plt.ylabel('Membrane potential (mV)')
I.plt.legend()
I.plt.show()

##### Somatic current injection control

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '20Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'SomaticCurrentInjection')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['SomaticCurrentInjection.stim.current_amp'] = spike_amp
protocol_params['SomaticCurrentInjection.stim.current_inj_times'] = I.np.arange(start=400, stop=650, step=50)
protocol_params['SomaticCurrentInjection.stim.current_duration'] = 2
protocol_params['SomaticCurrentInjection.run.tStop'] = 800

In [ ]:
run_protocol_soma_only(simulation_results_dir, loaded_simulator, loaded_cell_params.loc[biophysics_id], protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

soma_only_20Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
soma_only_20Hz_df

In [ ]:
plot_traces(
    soma_only_20Hz_df,
    inputs=[
        {
            'stim_times': cell_params['SomaticCurrentInjection.stim.current_inj_times'],
            'label': 'Somatic current (nA)',
            'amp': cell_params['SomaticCurrentInjection.stim.current_amp'],
            'duration': cell_params['SomaticCurrentInjection.stim.current_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, None),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
# from visualize.cell_morphology_visualizer import CellMorphologyVisualizer

# images_path = I.os.path.join(results_dir, 'soma_spike_animation_3d')

# # To remake the visualization, uncomment this code
# if I.os.path.exists(images_path):
#     I.shutil.rmtree(images_path)

# cmv = CellMorphologyVisualizer(simulated_cell)
# cmv.population_to_color_dict['inactive'] = "#f0f0f0"  # add color for inactive
# # cmv.population_to_color_dict['Generic'] = "red"

# cmv.animation(
#     images_path=images_path, 
#     color="voltage",
#     client=client, 
#     t_start=400-10, t_stop=440, t_step=0.2
# )

In [ ]:
plot_syn_v_scatter(soma_only_20Hz_df)

#### +10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '20Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'plus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = I.np.arange(start=400, stop=650, step=50)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = I.np.arange(start=410, stop=660, step=50)
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
# protocol_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
# protocol_params['STDP.stim.pairing_delay'] = 10
protocol_params['STDP.run.tStop'] = 800

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus10_20Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus10_20Hz_df

In [ ]:
plot_traces(
    plus10_20Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(395, 470), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
# all_delays = [delay for delays in plus10_20Hz_df['actual_delay'].dropna() for delay in delays]
# print(all_delays)

# # Calculate the average of the time differences
# mean_delay = I.np.mean(all_delays)
# print("Average delay:", mean_delay)

# df = plus10_20Hz_df.copy()

# df = df.dropna(subset=['actual_delay'])
# df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

# I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

# I.plt.xlabel('Delay between somatic and synaptic spike')
# I.plt.ylabel('Num synapses')

# %matplotlib inline
# I.plt.show()

In [ ]:
plot_syn_v_scatter(plus10_20Hz_df)

In [ ]:
plot_syn_v_comparisons([plus10_20Hz_df, soma_only_20Hz_df, synapses_only_20Hz_df], labels=['+10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

#### -10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '20Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'minus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = I.np.arange(start=400, stop=650, step=50)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = I.np.arange(start=390, stop=640, step=50)
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
# protocol_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
# protocol_params['STDP.stim.pairing_delay'] = -10
protocol_params['STDP.run.tStop'] = 800

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus10_20Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus10_20Hz_df

In [ ]:
plot_traces(
    minus10_20Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-70, -55)
)

In [ ]:
all_delays = [delay for delays in minus10_20Hz_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = minus10_20Hz_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_syn_v_scatter(minus10_20Hz_df)

In [ ]:
plot_syn_v_comparisons([minus10_20Hz_df, soma_only_20Hz_df, synapses_only_20Hz_df], labels=['-10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

#### Analysis

In [ ]:
plot_syn_v_hist([synapses_only_20Hz_df, soma_only_20Hz_df, plus10_20Hz_df, minus10_20Hz_df], labels=['Glutamate uncaging', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_comparisons([plus10_20Hz_df, minus10_20Hz_df], labels=['+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_violin([synapses_only_20Hz_df, soma_only_20Hz_df, plus10_20Hz_df, minus10_20Hz_df], labels=['Glutamate uncaging', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

##### Sjöström (only L5TT-L5TT)

In [ ]:
filtered_soma_only_20Hz_df = soma_only_20Hz_df[soma_only_20Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_synapses_only_20Hz_df = synapses_only_20Hz_df[synapses_only_20Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_plus10_20Hz_df = plus10_20Hz_df[plus10_20Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_minus10_20Hz_df = minus10_20Hz_df[minus10_20Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]

filtered_plus10_20Hz_df

In [ ]:
plot_syn_v_hist([filtered_synapses_only_20Hz_df, filtered_soma_only_20Hz_df, filtered_plus10_20Hz_df, filtered_minus10_20Hz_df], labels=['Synaptic stimulation', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_scatter(filtered_plus10_20Hz_df)

In [ ]:
plot_syn_v_scatter(filtered_minus10_20Hz_df)

In [ ]:
plot_syn_v_comparisons([filtered_plus10_20Hz_df, filtered_minus10_20Hz_df], labels=['+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_violin([filtered_synapses_only_20Hz_df, filtered_soma_only_20Hz_df, filtered_plus10_20Hz_df, filtered_minus10_20Hz_df], labels=['Synaptic stimulation', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

### 40 Hz

#### Synaptic stimulation only

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '40Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'GlutamateUncaging')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['GlutamateUncaging.stim.syn_stim_times'] = I.np.arange(start=400, stop=525, step=25)
protocol_params['GlutamateUncaging.run.tStop'] = 700

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'GlutamateUncaging', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

synapses_only_40Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
synapses_only_40Hz_df

In [ ]:
plot_traces(
    synapses_only_40Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['GlutamateUncaging.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
    ],
    x_lim=(385, 460), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411)  # , y_lim_soma_inset=(-76, -74)
)

In [ ]:
plot_syn_v_scatter(synapses_only_40Hz_df)

#### Somatic current injection only

##### Find somatic current stimulation parameters

In [ ]:
from biophysics_fitting.ephys import find_crossing

amps = I.np.arange(1, 5.1, step=0.1)

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

breaker = False
for amp in amps:
    amp = round(amp, 1)

    cell_params = loaded_cell_params.loc[biophysics_id].copy()

    cell_params['SomaticCurrentInjection.stim.current_amp'] = amp
    cell_params['SomaticCurrentInjection.stim.current_inj_times'] = I.np.arange(start=400, stop=525, step=25)
    cell_params['SomaticCurrentInjection.stim.current_duration'] = 2
    cell_params['SomaticCurrentInjection.run.tStop'] = 700

    simulated_cell, param = loaded_simulator.get_simulated_cell(cell_params, 'SomaticCurrentInjection')

    spikes = find_crossing(simulated_cell.soma.recVList[0], thresh=0)

    if len(spikes[0]) > len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        print('Multiple spikes')
        # I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        break
    elif len(spikes[0]) == len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        spike_amp = amp
        # time_to_spike = simulated_cell.tVec[spikes[0][0]] - cell_params['SomaticCurrentInjection.stim.current_inj_times'][0]
        print(spike_amp)
        print(simulated_cell.tVec[spikes[0][0]])
        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)

        # Run 1 extra increment of 0.1 when reaching "rheobase"
        if breaker:
            break
        breaker = True
    else:
        continue

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.xlim(left=390)
I.plt.ylabel('Membrane potential (mV)')
I.plt.legend()
I.plt.show()

##### Somatic current injection control

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '40Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'SomaticCurrentInjection')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['SomaticCurrentInjection.stim.current_amp'] = spike_amp
protocol_params['SomaticCurrentInjection.stim.current_inj_times'] = I.np.arange(start=400, stop=525, step=25)
protocol_params['SomaticCurrentInjection.stim.current_duration'] = 2
protocol_params['SomaticCurrentInjection.run.tStop'] = 700

In [ ]:
run_protocol_soma_only(simulation_results_dir, loaded_simulator, loaded_cell_params.loc[biophysics_id], protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

soma_only_40Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
soma_only_40Hz_df

In [ ]:
plot_traces(
    soma_only_40Hz_df,
    inputs=[
        {
            'stim_times': cell_params['SomaticCurrentInjection.stim.current_inj_times'],
            'label': 'Somatic current (nA)',
            'amp': cell_params['SomaticCurrentInjection.stim.current_amp'],
            'duration': cell_params['SomaticCurrentInjection.stim.current_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, None),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
# from visualize.cell_morphology_visualizer import CellMorphologyVisualizer

# images_path = I.os.path.join(results_dir, 'soma_spike_animation_3d')

# # To remake the visualization, uncomment this code
# if I.os.path.exists(images_path):
#     I.shutil.rmtree(images_path)

# cmv = CellMorphologyVisualizer(simulated_cell)
# cmv.population_to_color_dict['inactive'] = "#f0f0f0"  # add color for inactive
# # cmv.population_to_color_dict['Generic'] = "red"

# cmv.animation(
#     images_path=images_path, 
#     color="voltage",
#     client=client, 
#     t_start=400-10, t_stop=440, t_step=0.2
# )

In [ ]:
plot_syn_v_scatter(soma_only_40Hz_df)

#### +10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '40Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'plus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = I.np.arange(start=400, stop=525, step=25)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = I.np.arange(start=410, stop=535, step=25)
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
# protocol_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
# protocol_params['STDP.stim.pairing_delay'] = 10
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus10_40Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus10_40Hz_df

In [ ]:
plot_traces(
    plus10_40Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(395, 470), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
all_delays = [delay for delays in plus10_40Hz_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus10_40Hz_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_syn_v_scatter(plus10_40Hz_df)

In [ ]:
plot_syn_v_comparisons([plus10_40Hz_df, soma_only_40Hz_df, synapses_only_40Hz_df], labels=['+10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

#### -10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '40Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'minus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = I.np.arange(start=400, stop=525, step=25)  # 40 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = I.np.arange(start=390, stop=515, step=25)
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
# protocol_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
# protocol_params['STDP.stim.pairing_delay'] = -10
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus10_40Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus10_40Hz_df

In [ ]:
plot_traces(
    minus10_40Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-70, -55)
)

In [ ]:
all_delays = [delay for delays in minus10_40Hz_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = minus10_40Hz_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_syn_v_scatter(minus10_40Hz_df)

In [ ]:
plot_syn_v_comparisons([minus10_40Hz_df, soma_only_40Hz_df, synapses_only_40Hz_df], labels=['-10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

#### Analysis

In [ ]:
plot_syn_v_hist([synapses_only_40Hz_df, soma_only_40Hz_df, plus10_40Hz_df, minus10_40Hz_df], labels=['Glutamate uncaging', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_comparisons([plus10_40Hz_df, minus10_40Hz_df], labels=['+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_violin([synapses_only_40Hz_df, soma_only_40Hz_df, plus10_40Hz_df, minus10_40Hz_df], labels=['Glutamate uncaging', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

##### Sjöström (only L5TT-L5TT)

In [ ]:
filtered_soma_only_40Hz_df = soma_only_40Hz_df[soma_only_40Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_synapses_only_40Hz_df = synapses_only_40Hz_df[synapses_only_40Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_plus10_40Hz_df = plus10_40Hz_df[plus10_40Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_minus10_40Hz_df = minus10_40Hz_df[minus10_40Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]

filtered_plus10_40Hz_df

In [ ]:
plot_syn_v_hist([filtered_synapses_only_40Hz_df, filtered_soma_only_40Hz_df, filtered_plus10_40Hz_df, filtered_minus10_40Hz_df], labels=['Synaptic stimulation', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_scatter(filtered_plus10_40Hz_df)

In [ ]:
plot_syn_v_scatter(filtered_minus10_40Hz_df)

In [ ]:
plot_syn_v_comparisons([filtered_plus10_40Hz_df, filtered_minus10_40Hz_df], labels=['+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_violin([filtered_synapses_only_40Hz_df, filtered_soma_only_40Hz_df, filtered_plus10_40Hz_df, filtered_minus10_40Hz_df], labels=['Synaptic stimulation', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
# data = [soma_only_1Hz_df['syn_max_peak_height'], synapses_only_1Hz_df['syn_max_peak_height'], plus10_1Hz_df['syn_max_peak_height'], minus10_1Hz_df['syn_max_peak_height'], plus10_10Hz_df['syn_max_peak_height'], minus10_10Hz_df['syn_max_peak_height'], plus10_20Hz_df['syn_max_peak_height'], minus10_20Hz_df['syn_max_peak_height'], plus10_40Hz_df['syn_max_peak_height'], minus10_40Hz_df['syn_max_peak_height']]

# fig, ax = I.plt.subplots()
# ax.violinplot(data, showmeans=True, showmedians=False, showextrema=False)

# # Set custom categorical labels for the x-axis
# ax.set_xticks(I.np.arange(1, len(data)+1))
# ax.set_xticklabels(['Somatic spike (1 Hz)', 'Glutamate uncaging (1 Hz)', '+10 ms delay (1 Hz)', '-10 ms delay (1 Hz)', '+10 ms delay (5x10 Hz)', '-10 ms delay x 5 (5x10 Hz)', '+10 ms delay (5x20 Hz)', '-10 ms delay x 5 (5x20 Hz)', '+10 ms delay (5x40 Hz)', '-10 ms delay x 5 (5x40 Hz)'], rotation=45)

# ax.set_ylabel('Max Vm at synapse (mV)')

# %matplotlib inline
# I.plt.show()

### 50 Hz

#### Synaptic stimulation only

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '50Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'GlutamateUncaging')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['GlutamateUncaging.stim.syn_stim_times'] = I.np.arange(start=400, stop=500, step=20)
protocol_params['GlutamateUncaging.run.tStop'] = 700

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'GlutamateUncaging', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

synapses_only_50Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
synapses_only_50Hz_df

In [ ]:
plot_traces(
    synapses_only_50Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['GlutamateUncaging.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
    ],
    x_lim=(385, 460), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411)  # , y_lim_soma_inset=(-76, -74)
)

In [ ]:
plot_syn_v_scatter(synapses_only_50Hz_df)

#### Somatic current injection only

##### Find somatic current stimulation parameters

In [ ]:
from biophysics_fitting.ephys import find_crossing

amps = I.np.arange(1, 5.1, step=0.1)

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

breaker = False
for amp in tqdm(amps):
    amp = round(amp, 1)

    cell_params = loaded_cell_params.loc[biophysics_id].copy()

    cell_params['SomaticCurrentInjection.stim.current_amp'] = amp
    cell_params['SomaticCurrentInjection.stim.current_inj_times'] = I.np.arange(start=400, stop=500, step=20)
    cell_params['SomaticCurrentInjection.stim.current_duration'] = 2
    cell_params['SomaticCurrentInjection.run.tStop'] = 700

    simulated_cell, param = loaded_simulator.get_simulated_cell(cell_params, 'SomaticCurrentInjection')

    spikes = find_crossing(simulated_cell.soma.recVList[0], thresh=0)

    if len(spikes[0]) > len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        print('Multiple spikes')
        # I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        break
    elif len(spikes[0]) == len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        spike_amp = amp
        # time_to_spike = simulated_cell.tVec[spikes[0][0]] - cell_params['SomaticCurrentInjection.stim.current_inj_times'][0]
        print(spike_amp)
        print(simulated_cell.tVec[spikes[0][0]])
        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)

        # Run 1 extra increment of 0.1 when reaching "rheobase"
        if breaker:
            break
        breaker = True
    else:
        continue

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.xlim(left=390)
I.plt.ylabel('Membrane potential (mV)')
I.plt.legend()
I.plt.show()

##### Somatic current injection control

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '50Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'SomaticCurrentInjection')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['SomaticCurrentInjection.stim.current_amp'] = spike_amp
protocol_params['SomaticCurrentInjection.stim.current_inj_times'] = I.np.arange(start=400, stop=500, step=20)
protocol_params['SomaticCurrentInjection.stim.current_duration'] = 2
protocol_params['SomaticCurrentInjection.run.tStop'] = 700

In [ ]:
run_protocol_soma_only(simulation_results_dir, loaded_simulator, loaded_cell_params.loc[biophysics_id], protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

soma_only_50Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
soma_only_50Hz_df

In [ ]:
plot_traces(
    soma_only_50Hz_df,
    inputs=[
        {
            'stim_times': cell_params['SomaticCurrentInjection.stim.current_inj_times'],
            'label': 'Somatic current (nA)',
            'amp': cell_params['SomaticCurrentInjection.stim.current_amp'],
            'duration': cell_params['SomaticCurrentInjection.stim.current_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, None),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
# from visualize.cell_morphology_visualizer import CellMorphologyVisualizer

# images_path = I.os.path.join(results_dir, 'soma_spike_animation_3d')

# # To remake the visualization, uncomment this code
# if I.os.path.exists(images_path):
#     I.shutil.rmtree(images_path)

# cmv = CellMorphologyVisualizer(simulated_cell)
# cmv.population_to_color_dict['inactive'] = "#f0f0f0"  # add color for inactive
# # cmv.population_to_color_dict['Generic'] = "red"

# cmv.animation(
#     images_path=images_path, 
#     color="voltage",
#     client=client, 
#     t_start=400-10, t_stop=440, t_step=0.2
# )

In [ ]:
plot_syn_v_scatter(soma_only_50Hz_df)

#### +10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '50Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'plus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = I.np.arange(start=400, stop=500, step=20)  # 50 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = I.np.arange(start=410, stop=510, step=20)
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
# protocol_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
# protocol_params['STDP.stim.pairing_delay'] = 10
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus10_50Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus10_50Hz_df

In [ ]:
plot_traces(
    plus10_50Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(395, 470), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
# all_delays = [delay for delays in plus10_50Hz_df['actual_delay'].dropna() for delay in delays]

# # Calculate the average of the time differences
# mean_delay = I.np.mean(all_delays)
# print("Average delay:", mean_delay)

# df = plus10_50Hz_df.copy()

# df = df.dropna(subset=['actual_delay'])
# df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

# I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

# I.plt.xlabel('Delay between somatic and synaptic spike')
# I.plt.ylabel('Num synapses')

# %matplotlib inline
# I.plt.show()

In [ ]:
plot_syn_v_scatter(plus10_50Hz_df)

In [ ]:
plot_syn_v_comparisons([plus10_50Hz_df, soma_only_50Hz_df, synapses_only_50Hz_df], labels=['+10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

#### -10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '50Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'minus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = I.np.arange(start=400, stop=500, step=20)  # 40 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = I.np.arange(start=390, stop=490, step=20)
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
# protocol_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
# protocol_params['STDP.stim.pairing_delay'] = -10
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus10_50Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus10_50Hz_df

In [ ]:
plot_traces(
    minus10_50Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-70, -55)
)

In [ ]:
all_delays = [delay for delays in minus10_50Hz_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = minus10_50Hz_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_syn_v_scatter(minus10_50Hz_df)

In [ ]:
plot_syn_v_comparisons([minus10_50Hz_df, soma_only_50Hz_df, synapses_only_50Hz_df], labels=['-10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

#### Analysis

In [ ]:
plot_syn_v_hist([synapses_only_50Hz_df, soma_only_50Hz_df, plus10_50Hz_df, minus10_50Hz_df], labels=['Glutamate uncaging', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_comparisons([plus10_50Hz_df, minus10_50Hz_df], labels=['+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_violin([synapses_only_50Hz_df, soma_only_50Hz_df, plus10_50Hz_df, minus10_50Hz_df], labels=['Glutamate uncaging', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

##### Sjöström (only L5TT-L5TT)

In [ ]:
filtered_soma_only_50Hz_df = soma_only_50Hz_df[soma_only_50Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_synapses_only_50Hz_df = synapses_only_50Hz_df[synapses_only_50Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_plus10_50Hz_df = plus10_50Hz_df[plus10_50Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]
filtered_minus10_50Hz_df = minus10_50Hz_df[minus10_50Hz_df['presyn_cell_type'].apply(lambda x: 'L5tt' in x)]

filtered_plus10_50Hz_df

In [ ]:
plot_syn_v_hist([filtered_synapses_only_50Hz_df, filtered_soma_only_50Hz_df, filtered_plus10_50Hz_df, filtered_minus10_50Hz_df], labels=['Synaptic stimulation', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_scatter(filtered_plus10_50Hz_df)

In [ ]:
plot_syn_v_scatter(filtered_minus10_50Hz_df)

In [ ]:
plot_syn_v_comparisons([filtered_plus10_50Hz_df, filtered_minus10_50Hz_df], labels=['+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_violin([filtered_synapses_only_50Hz_df, filtered_soma_only_50Hz_df, filtered_plus10_50Hz_df, filtered_minus10_50Hz_df], labels=['Synaptic stimulation', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
# data = [soma_only_1Hz_df['syn_max_peak_height'], synapses_only_1Hz_df['syn_max_peak_height'], plus10_1Hz_df['syn_max_peak_height'], minus10_1Hz_df['syn_max_peak_height'], plus10_10Hz_df['syn_max_peak_height'], minus10_10Hz_df['syn_max_peak_height'], plus10_20Hz_df['syn_max_peak_height'], minus10_20Hz_df['syn_max_peak_height'], plus10_40Hz_df['syn_max_peak_height'], minus10_40Hz_df['syn_max_peak_height']]

# fig, ax = I.plt.subplots()
# ax.violinplot(data, showmeans=True, showmedians=False, showextrema=False)

# # Set custom categorical labels for the x-axis
# ax.set_xticks(I.np.arange(1, len(data)+1))
# ax.set_xticklabels(['Somatic spike (1 Hz)', 'Glutamate uncaging (1 Hz)', '+10 ms delay (1 Hz)', '-10 ms delay (1 Hz)', '+10 ms delay (5x10 Hz)', '-10 ms delay x 5 (5x10 Hz)', '+10 ms delay (5x20 Hz)', '-10 ms delay x 5 (5x20 Hz)', '+10 ms delay (5x40 Hz)', '-10 ms delay x 5 (5x40 Hz)'], rotation=45)

# ax.set_ylabel('Max Vm at synapse (mV)')

# %matplotlib inline
# I.plt.show()

### Frequency analysis

In [ ]:
plot_syn_v_hist([plus10_1Hz_df, plus10_10Hz_df, plus10_20Hz_df, plus10_40Hz_df], labels=['1Hz; +10 ms delay', '10Hz; +10 ms delay', '20Hz; +10 ms delay', '40Hz; +10 ms delay'])

In [ ]:
plot_syn_v_violin([plus10_1Hz_df, plus10_10Hz_df, plus10_20Hz_df, plus10_40Hz_df], labels=['1Hz; +10 ms delay', '10Hz; +10 ms delay', '20Hz; +10 ms delay', '40Hz; +10 ms delay'])

In [ ]:
plot_syn_v_hist([minus10_1Hz_df, minus10_10Hz_df, minus10_20Hz_df, minus10_40Hz_df], labels=['1Hz; -10 ms delay', '10Hz; -10 ms delay', '20Hz; -10 ms delay', '40Hz; -10 ms delay'])

In [ ]:
plot_syn_v_violin([minus10_1Hz_df, minus10_10Hz_df, minus10_20Hz_df, minus10_40Hz_df], labels=['1Hz; -10 ms delay', '10Hz; -10 ms delay', '20Hz; -10 ms delay', '40Hz; -10 ms delay'])

In [ ]:
from biophysics_fitting.ephys import find_crossing

amps = I.np.arange(1, 5.1, step=0.1)
current_inj_timesteps = [1000, 100, 50, 25]
recording_dists = [loaded_fixed_params['BAC.stim.dist'], 1200]

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

current_inj_amps = []
for timestep in current_inj_timesteps:
    print(timestep)

    cell_params = loaded_cell_params.loc[biophysics_id].copy()

    if timestep == 1000:
        cell_params['SomaticCurrentInjection.stim.current_inj_times'] = [400]
    else:
        cell_params['SomaticCurrentInjection.stim.current_inj_times'] = I.np.arange(start=400, stop=400+(5*timestep), step=timestep)
    cell_params['SomaticCurrentInjection.stim.current_duration'] = 2
    cell_params['SomaticCurrentInjection.run.tStop'] = cell_params['SomaticCurrentInjection.stim.current_inj_times'][-1] + 100

    print(cell_params['SomaticCurrentInjection.stim.current_inj_times'])

    breaker = False
    for amp in amps:
        amp = round(amp, 1)

        cell_params['SomaticCurrentInjection.stim.current_amp'] = amp

        simulated_cell, param = loaded_simulator.get_simulated_cell(cell_params, 'SomaticCurrentInjection')

        spikes = find_crossing(simulated_cell.soma.recVList[0], thresh=0)

        if len(spikes[0]) == len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
            # Run 1 extra increment of 0.1 when reaching "rheobase"
            if breaker:
                print(amp)
                current_inj_amps.append(amp)

                fig, axes = I.plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(8, 10), height_ratios=[1, 5])

                y_input = I.np.zeros_like(simulated_cell.tVec)
                for time in cell_params['SomaticCurrentInjection.stim.current_inj_times']:
                    i = I.np.argmin(I.np.abs(simulated_cell.tVec - time))  # Find index closest to time
                    j = I.np.argmin(I.np.abs(simulated_cell.tVec - (time + cell_params['SomaticCurrentInjection.stim.current_duration'])))
                    y_input[i:j] = amp
                axes[0].plot(simulated_cell.tVec, y_input)

                # Soma
                axes[1].plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label='Soma')

                # Dendrite
                for dist in recording_dists:
                    apical_sect = _get_apical_sec_and_i_at_distance(simulated_cell, dist)
                    vm_dend = I.np.array(apical_sect[0].recVList)
                    axes[1].plot(simulated_cell.tVec, vm_dend[0], label=f'{round(dist, 1)} dist')

                I.plt.xlabel('Time (ms)')
                I.plt.xlim((390, cell_params['SomaticCurrentInjection.run.tStop']))
                axes[0].set_ylabel('Somatic current (nA)')
                axes[1].set_ylabel('Membrane potential (mV)')
                I.plt.legend()

                %matplotlib inline
                I.plt.show()

                break
            breaker = True
        else:
            continue

In [ ]:
# Step 1: Filter by soma_dist
filtered_df = plus10_40Hz_df[plus10_40Hz_df['soma_dist'] >= 1000]

# Step 2: Get rows with the highest syn_max_peak_height per presyn_cell_type
idx_max = filtered_df.groupby('presyn_cell_type')['syn_max_peak_height'].idxmax()

# Step 3: Use those indices to get the final filtered DataFrame
filtered_df = filtered_df.loc[idx_max].reset_index(drop=True)

filtered_df = filtered_df.sort_values(by='presyn_cell_type').reset_index(drop=True)
filtered_df

In [ ]:
current_inj_timesteps = [1000, 100, 50, 25]

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

# Map synapses
syn_dist = read_synapse_realization(syn_file_path)
synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
synapse_mapper.map_synapse_realization()

stim_synapses = []
for _, stim_syn in filtered_df.iterrows():
    for presyn_cell_type in cell.synapses.keys():
        for syn_id, syn in enumerate(cell.synapses[presyn_cell_type]):
            if presyn_cell_type == stim_syn['presyn_cell_label'] and syn_id == stim_syn['syn_id']:
                synapse_strength_celltype = [x for x in loaded_syn_weights.keys() if x in presyn_cell_type]
                assert(len(synapse_strength_celltype) == 1)
                synapse_strength_celltype = synapse_strength_celltype[0]
                weight = loaded_syn_weights[synapse_strength_celltype]

                syn.weight = {'glutamate_syn': [weight, weight]}
                syn.receptors = EXC_RECEPTOR_DICT

                stim_synapses.append(syn)

print(stim_synapses)

for i, timestep in enumerate(current_inj_timesteps):
    print(timestep)

    cell_params = loaded_cell_params.loc[biophysics_id].copy()

    if timestep == 1000:
        cell_params['STDP.stim.syn_stim_times'] = [400]
        cell_params['STDP.stim.postsyn_stim_times'] = [410]
    else:
        cell_params['STDP.stim.syn_stim_times'] = I.np.arange(start=400, stop=400+(5*timestep), step=timestep)
        cell_params['STDP.stim.postsyn_stim_times'] = I.np.arange(start=410, stop=410+(5*timestep), step=timestep)
    cell_params['STDP.stim.postsyn_stim_amp'] = current_inj_amps[i]
    cell_params['STDP.stim.postsyn_stim_duration'] = 2
    cell_params['STDP.run.tStop'] = cell_params['STDP.stim.postsyn_stim_times'][-1] + 100

    print(cell_params['STDP.stim.postsyn_stim_times'])
    print(cell_params['STDP.stim.postsyn_stim_amp'])

    fig, axes = I.plt.subplots(nrows=3, ncols=1, sharex=True, figsize=(8, 11), height_ratios=[1, 1, 4])

    for syn_i, syn in enumerate(stim_synapses):
        cell_params['STDP.stim.synapses'] = [syn]

        simulated_cell, param = loaded_simulator.get_simulated_cell(cell_params, 'STDP')

        # Soma
        axes[2].plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], c='k', alpha=0.5)

        # Dendrite
        syn_seg_id = int(syn.x * simulated_cell.sections[syn.secID].nseg)
        if syn_seg_id == simulated_cell.sections[syn.secID].nseg:
            syn_seg_id -= 1

        axes[2].plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recVList[syn_seg_id].as_numpy(), label=f'{filtered_df.iloc[syn_i]["presyn_cell_type"]}')

    # Synaptic input
    y_input = I.np.zeros_like(simulated_cell.tVec)
    for time in cell_params['STDP.stim.syn_stim_times']:
        i = I.np.argmin(I.np.abs(simulated_cell.tVec - time))  # Find index closest to time
        y_input[i] = 1
    axes[0].plot(simulated_cell.tVec, y_input)

    # Somatic input
    y_input = I.np.zeros_like(simulated_cell.tVec)
    for time in cell_params['STDP.stim.postsyn_stim_times']:
        i = I.np.argmin(I.np.abs(simulated_cell.tVec - time))  # Find index closest to time
        j = I.np.argmin(I.np.abs(simulated_cell.tVec - (time + cell_params['STDP.stim.postsyn_stim_duration'])))
        y_input[i:j] = cell_params['STDP.stim.postsyn_stim_amp']
    axes[1].plot(simulated_cell.tVec, y_input)

    I.plt.xlabel('Time (ms)')
    I.plt.xlim((390, cell_params['STDP.run.tStop']))
    axes[0].set_ylabel('Synaptic input')
    axes[1].set_ylabel('Somatic current (nA)')
    axes[2].set_ylabel('Membrane potential (mV)')
    I.plt.legend()

    %matplotlib inline
    I.plt.show()

#### Sjöström (only L5TT-L5TT)

In [ ]:
plot_syn_v_hist([filtered_plus10_1Hz_df, filtered_plus10_10Hz_df, filtered_plus10_20Hz_df, filtered_plus10_40Hz_df], labels=['1Hz; +10 ms delay', '10Hz; +10 ms delay', '20Hz; +10 ms delay', '40Hz; +10 ms delay'])

In [ ]:
plot_syn_v_violin([filtered_plus10_1Hz_df, filtered_plus10_10Hz_df, filtered_plus10_20Hz_df, filtered_plus10_40Hz_df], labels=['1Hz; +10 ms delay', '10Hz; +10 ms delay', '20Hz; +10 ms delay', '40Hz; +10 ms delay'])

In [ ]:
plot_syn_v_hist([filtered_minus10_1Hz_df, filtered_minus10_10Hz_df, filtered_minus10_20Hz_df, filtered_minus10_40Hz_df], labels=['1Hz; -10 ms delay', '10Hz; -10 ms delay', '20Hz; -10 ms delay', '40Hz; -10 ms delay'])

In [ ]:
plot_syn_v_violin([filtered_minus10_1Hz_df, filtered_minus10_10Hz_df, filtered_minus10_20Hz_df, filtered_minus10_40Hz_df], labels=['1Hz; -10 ms delay', '10Hz; -10 ms delay', '20Hz; -10 ms delay', '40Hz; -10 ms delay'])

### Timing (extracellular stimulation)

#### Synaptic stimulation only

##### 1 < EPSP amplitude < 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['GlutamateUncaging.stim.syn_stim_times'] = [400]
protocol_params['GlutamateUncaging.run.tStop'] = 600

In [ ]:
find_extracellular_stim_syns(simulation_results_dir,
    'GlutamateUncaging',
    loaded_simulator,
    loaded_cell_params.loc[biophysics_id],
    loaded_syn_weights,
    protocol_params,
    max_distance=max_soma_dist,
    distance_bin_size=10,
    min_epsp_amp=1,
)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

synapses_only_smallEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
synapses_only_smallEPSPs_df

In [ ]:
# plot_traces(
#     synapses_only_1Hz_df,
#     inputs=[
#         {
#             'stim_times': protocol_params['GlutamateUncaging.stim.syn_stim_times'],
#             'label': 'Synaptic input',
#             'amp': 1
#         },
#     ],
#     x_lim=(385, 460), y_lim_syn=(-80, -5),
#     x_lim_soma_inset=(400, 411)  # , y_lim_soma_inset=(-76, -74)
# )

In [ ]:
plot_scatter(synapses_only_smallEPSPs_df, 'mean_soma_dist', 'max_EPSP_amp', color_column='num_syns', xlabel='Soma distance (µm)', ylabel='EPSP amplitude (mV)', clabel='Num synapses')

plot_extracellular_syn_v(synapses_only_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
apical_synapses_only_smallEPSPs_df = synapses_only_smallEPSPs_df[synapses_only_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_synapses_only_smallEPSPs_df

In [ ]:
plot_scatter(apical_synapses_only_smallEPSPs_df, 'mean_soma_dist', 'max_EPSP_amp', color_column='num_syns', xlabel='Soma distance (µm)', ylabel='EPSP amplitude (mV)', clabel='Num synapses')

plot_extracellular_syn_v(apical_synapses_only_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### EPSP amplitude > 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['GlutamateUncaging.stim.syn_stim_times'] = [400]
protocol_params['GlutamateUncaging.run.tStop'] = 700

In [ ]:
find_extracellular_stim_syns(simulation_results_dir, 'GlutamateUncaging', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params, max_distance=max_soma_dist, min_epsp_amp=2.3, repetions=3)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

synapses_only_largeEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
synapses_only_largeEPSPs_df

In [ ]:
# plot_traces(
#     synapses_only_1Hz_df,
#     inputs=[
#         {
#             'stim_times': protocol_params['GlutamateUncaging.stim.syn_stim_times'],
#             'label': 'Synaptic input',
#             'amp': 1
#         },
#     ],
#     x_lim=(385, 460), y_lim_syn=(-80, -5),
#     x_lim_soma_inset=(400, 411)  # , y_lim_soma_inset=(-76, -74)
# )

In [ ]:
plot_scatter(synapses_only_largeEPSPs_df, 'mean_soma_dist', 'max_EPSP_amp', color_column='num_syns', xlabel='Soma distance (µm)', ylabel='EPSP amplitude (mV)', clabel='Num synapses')

plot_extracellular_syn_v(synapses_only_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
apical_synapses_only_largeEPSPs_df = synapses_only_largeEPSPs_df[synapses_only_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_synapses_only_largeEPSPs_df

In [ ]:
plot_scatter(apical_synapses_only_largeEPSPs_df, 'mean_soma_dist', 'max_EPSP_amp', color_column='num_syns', xlabel='Soma distance (µm)', ylabel='EPSP amplitude (mV)', clabel='Num synapses')

plot_extracellular_syn_v(apical_synapses_only_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

#### Find somatic current stimulation parameters

In [ ]:
from biophysics_fitting.ephys import find_crossing

amps = I.np.arange(1, 5.1, step=0.1)

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

breaker = False
for amp in amps:
    amp = round(amp, 1)

    cell_params = loaded_cell_params.loc[biophysics_id].copy()

    cell_params['SomaticCurrentInjection.stim.current_amp'] = amp
    cell_params['SomaticCurrentInjection.stim.current_inj_times'] = [400]
    cell_params['SomaticCurrentInjection.stim.current_duration'] = 2
    cell_params['SomaticCurrentInjection.run.tStop'] = 700

    simulated_cell, param = loaded_simulator.get_simulated_cell(cell_params, 'SomaticCurrentInjection')

    spikes = find_crossing(simulated_cell.soma.recVList[0], thresh=0)

    if len(spikes[0]) > len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        print('Multiple spikes')
        # I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        break
    elif len(spikes[0]) == len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        spike_amp = amp
        # time_to_spike = simulated_cell.tVec[spikes[0][0]] - cell_params['SomaticCurrentInjection.stim.current_inj_times'][0]
        print(spike_amp)
        print(simulated_cell.tVec[spikes[0][0]])
        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)

        # Run 1 extra increment of 0.1 when reaching "rheobase"
        if breaker:
            break
        breaker = True
    else:
        continue

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.xlim(left=390)
I.plt.ylabel('Membrane potential (mV)')
I.plt.legend()
I.plt.show()

#### +10 ms delay

##### 1 < EPSP amplitude < 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'plus10')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [410]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus10_smallEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus10_smallEPSPs_df

In [ ]:
all_delays = [delay for delays in plus10_smallEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus10_smallEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(plus10_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_plus10_smallEPSPs_df = plus10_smallEPSPs_df[plus10_smallEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
# filtered_df = plus10_smallEPSPs_df[plus10_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
filtered_plus10_smallEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(plus10_smallEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_plus10_smallEPSPs_df = plus10_smallEPSPs_df[plus10_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_plus10_smallEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_plus10_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### EPSP amplitude > 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'plus10')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [410]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus10_largeEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus10_largeEPSPs_df

In [ ]:
all_delays = [delay for delays in plus10_largeEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus10_largeEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(plus10_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_plus10_largeEPSPs_df = plus10_largeEPSPs_df[plus10_largeEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
filtered_plus10_largeEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(plus10_largeEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_plus10_largeEPSPs_df = plus10_largeEPSPs_df[plus10_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_plus10_largeEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_plus10_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### Analysis

In [ ]:
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, plus10_smallEPSPs_df, plus10_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+10 ms (<2.3 mV)', '+10 ms (>2.3 mV)'])
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, plus10_smallEPSPs_df, plus10_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+10 ms (<2.3 mV)', '+10 ms (>2.3 mV)'])

In [ ]:
plot_histogram([plus10_smallEPSPs_df, plus10_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['+10 ms (<2.3 mV)', '+10 ms (>2.3 mV)'])
plot_histogram([plus10_smallEPSPs_df, plus10_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['+10 ms (<2.3 mV)', '+10 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([plus10_smallEPSPs_df, plus10_largeEPSPs_df], labels=['+10 ms (<2.3 mV)', '+10 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([apical_only_plus10_smallEPSPs_df, apical_only_plus10_largeEPSPs_df], labels=['+10 ms (<2.3 mV)', '+10 ms (>2.3 mV)'])

###### Sjöström (only lower L4)

In [ ]:
filtered_synapses_only_smallEPSPs_df = synapses_only_smallEPSPs_df[synapses_only_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_smallEPSPs_df = filtered_synapses_only_smallEPSPs_df[filtered_synapses_only_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_synapses_only_largeEPSPs_df = synapses_only_largeEPSPs_df[synapses_only_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_largeEPSPs_df = filtered_synapses_only_largeEPSPs_df[filtered_synapses_only_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_plus10_smallEPSPs_df = plus10_smallEPSPs_df[plus10_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_plus10_smallEPSPs_df = filtered_plus10_smallEPSPs_df[filtered_plus10_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_plus10_largeEPSPs_df = plus10_largeEPSPs_df[plus10_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_plus10_largeEPSPs_df = filtered_plus10_largeEPSPs_df[filtered_plus10_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_plus10_largeEPSPs_df

In [ ]:
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_plus10_smallEPSPs_df, filtered_plus10_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+10 ms (<2.3 mV)', '+10 ms (>2.3 mV)'])
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_plus10_smallEPSPs_df, filtered_plus10_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+10 ms (<2.3 mV)', '+10 ms (>2.3 mV)'])

In [ ]:
plot_histogram([filtered_plus10_smallEPSPs_df, filtered_plus10_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['+10 ms (<2.3 mV)', '+10 ms (>2.3 mV)'])
plot_histogram([filtered_plus10_smallEPSPs_df, filtered_plus10_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['+10 ms (<2.3 mV)', '+10 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([filtered_plus10_smallEPSPs_df, filtered_plus10_largeEPSPs_df], labels=['+10 ms (<2.3 mV)', '+10 ms (>2.3 mV)'])

#### -10 ms delay

##### 1 < EPSP amplitude < 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'minus10')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [390]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus10_smallEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus10_smallEPSPs_df

In [ ]:
all_delays = [delay for delays in minus10_smallEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = minus10_smallEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(minus10_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_minus10_smallEPSPs_df = minus10_smallEPSPs_df[minus10_smallEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
# filtered_df = minus10_smallEPSPs_df[minus10_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
filtered_minus10_smallEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(minus10_smallEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_minus10_smallEPSPs_df = minus10_smallEPSPs_df[minus10_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_minus10_smallEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_minus10_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### EPSP amplitude > 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'minus10')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [390]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus10_largeEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus10_largeEPSPs_df

In [ ]:
all_delays = [delay for delays in minus10_largeEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = minus10_largeEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(minus10_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_minus10_largeEPSPs_df = minus10_largeEPSPs_df[minus10_largeEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
filtered_minus10_largeEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(minus10_largeEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_minus10_largeEPSPs_df = minus10_largeEPSPs_df[minus10_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_minus10_largeEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_minus10_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### Analysis

In [ ]:
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, minus10_smallEPSPs_df, minus10_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-10 ms (<2.3 mV)', '-10 ms (>2.3 mV)'])
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, minus10_smallEPSPs_df, minus10_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-10 ms (<2.3 mV)', '-10 ms (>2.3 mV)'])

In [ ]:
plot_histogram([minus10_smallEPSPs_df, minus10_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['-10 ms (<2.3 mV)', '-10 ms (>2.3 mV)'])
plot_histogram([minus10_smallEPSPs_df, minus10_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['-10 ms (<2.3 mV)', '-10 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([minus10_smallEPSPs_df, minus10_largeEPSPs_df], labels=['-10 ms (<2.3 mV)', '-10 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([apical_only_minus10_smallEPSPs_df, apical_only_minus10_largeEPSPs_df], labels=['-10 ms (<2.3 mV)', '-10 ms (>2.3 mV)'])

###### Sjöström (only lower L4)

In [ ]:
filtered_synapses_only_smallEPSPs_df = synapses_only_smallEPSPs_df[synapses_only_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_smallEPSPs_df = filtered_synapses_only_smallEPSPs_df[filtered_synapses_only_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_synapses_only_largeEPSPs_df = synapses_only_largeEPSPs_df[synapses_only_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_largeEPSPs_df = filtered_synapses_only_largeEPSPs_df[filtered_synapses_only_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_minus10_smallEPSPs_df = minus10_smallEPSPs_df[minus10_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_minus10_smallEPSPs_df = filtered_minus10_smallEPSPs_df[filtered_minus10_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_minus10_largeEPSPs_df = minus10_largeEPSPs_df[minus10_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_minus10_largeEPSPs_df = filtered_minus10_largeEPSPs_df[filtered_minus10_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_minus10_largeEPSPs_df

In [ ]:
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_minus10_smallEPSPs_df, filtered_minus10_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-10 ms (<2.3 mV)', '-10 ms (>2.3 mV)'])
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_minus10_smallEPSPs_df, filtered_minus10_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-10 ms (<2.3 mV)', '-10 ms (>2.3 mV)'])

In [ ]:
plot_histogram([filtered_minus10_smallEPSPs_df, filtered_minus10_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['-10 ms (<2.3 mV)', '-10 ms (>2.3 mV)'])
plot_histogram([filtered_minus10_smallEPSPs_df, filtered_minus10_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['-10 ms (<2.3 mV)', '-10 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([filtered_minus10_smallEPSPs_df, filtered_minus10_largeEPSPs_df], labels=['-10 ms (<2.3 mV)', '-10 ms (>2.3 mV)'])

#### Analysis

In [ ]:
# plot_extracellular_syn_v_compare([plus10_smallEPSPs_df, minus10_smallEPSPs_df], labels=['+10 ms (<2.3 mV)', '-10 ms (<2.3 mV)'])

In [ ]:
# plot_extracellular_syn_v_compare([plus10_largeEPSPs_df, minus10_largeEPSPs_df], labels=['+10 ms (>2.3 mV)', '-10 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([plus10_smallEPSPs_df, minus10_smallEPSPs_df, plus10_largeEPSPs_df, minus10_largeEPSPs_df], labels=['+10 ms (<2.3 mV)', '-10 ms (<2.3 mV)', '+10 ms (>2.3 mV)', '-10 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([apical_only_plus10_smallEPSPs_df, apical_only_minus10_smallEPSPs_df, apical_only_plus10_largeEPSPs_df, apical_only_minus10_largeEPSPs_df], labels=['+10 ms (<2.3 mV)', '-10 ms (<2.3 mV)', '+10 ms (>2.3 mV)', '-10 ms (>2.3 mV)'])

#### +25 ms delay

##### 1 < EPSP amplitude < 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'plus25')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [425]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus25_smallEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus25_smallEPSPs_df

In [ ]:
all_delays = [delay for delays in plus25_smallEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus25_smallEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(plus25_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_plus25_smallEPSPs_df = plus25_smallEPSPs_df[plus25_smallEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
# filtered_df = plus25_smallEPSPs_df[plus25_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
filtered_plus25_smallEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(plus25_smallEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_plus25_smallEPSPs_df = plus25_smallEPSPs_df[plus25_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_plus25_smallEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_plus25_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### EPSP amplitude > 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'plus25')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [425]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus25_largeEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus25_largeEPSPs_df

In [ ]:
all_delays = [delay for delays in plus25_largeEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus25_largeEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(plus25_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_plus25_largeEPSPs_df = plus25_largeEPSPs_df[plus25_largeEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
filtered_plus25_largeEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(plus25_largeEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_plus25_largeEPSPs_df = plus25_largeEPSPs_df[plus25_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_plus25_largeEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_plus25_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### Analysis

In [ ]:
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, plus25_smallEPSPs_df, plus25_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+25 ms (<2.3 mV)', '+25 ms (>2.3 mV)'])
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, plus25_smallEPSPs_df, plus25_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+25 ms (<2.3 mV)', '+25 ms (>2.3 mV)'])

In [ ]:
plot_histogram([plus25_smallEPSPs_df, plus25_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['+25 ms (<2.3 mV)', '+25 ms (>2.3 mV)'])
plot_histogram([plus25_smallEPSPs_df, plus25_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['+25 ms (<2.3 mV)', '+25 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([plus25_smallEPSPs_df, plus25_largeEPSPs_df], labels=['+25 ms (<2.3 mV)', '+25 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([apical_only_plus25_smallEPSPs_df, apical_only_plus25_largeEPSPs_df], labels=['+25 ms (<2.3 mV)', '+25 ms (>2.3 mV)'])

###### Sjöström (only lower L4)

In [ ]:
filtered_synapses_only_smallEPSPs_df = synapses_only_smallEPSPs_df[synapses_only_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_smallEPSPs_df = filtered_synapses_only_smallEPSPs_df[filtered_synapses_only_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_synapses_only_largeEPSPs_df = synapses_only_largeEPSPs_df[synapses_only_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_largeEPSPs_df = filtered_synapses_only_largeEPSPs_df[filtered_synapses_only_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_plus25_smallEPSPs_df = plus25_smallEPSPs_df[plus25_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_plus25_smallEPSPs_df = filtered_plus25_smallEPSPs_df[filtered_plus25_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_plus25_largeEPSPs_df = plus25_largeEPSPs_df[plus25_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_plus25_largeEPSPs_df = filtered_plus25_largeEPSPs_df[filtered_plus25_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_plus25_largeEPSPs_df

In [ ]:
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_plus25_smallEPSPs_df, filtered_plus25_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+25 ms (<2.3 mV)', '+25 ms (>2.3 mV)'])
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_plus25_smallEPSPs_df, filtered_plus25_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+25 ms (<2.3 mV)', '+25 ms (>2.3 mV)'])

In [ ]:
plot_histogram([filtered_plus25_smallEPSPs_df, filtered_plus25_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['+25 ms (<2.3 mV)', '+25 ms (>2.3 mV)'])
plot_histogram([filtered_plus25_smallEPSPs_df, filtered_plus25_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['+25 ms (<2.3 mV)', '+25 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([filtered_plus25_smallEPSPs_df, filtered_plus25_largeEPSPs_df], labels=['+25 ms (<2.3 mV)', '+25 ms (>2.3 mV)'])

#### -25 ms delay

##### 1 < EPSP amplitude < 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'minus25')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [425]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [400]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus25_smallEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus25_smallEPSPs_df

In [ ]:
all_delays = [delay for delays in minus25_smallEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = minus25_smallEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(minus25_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_minus25_smallEPSPs_df = minus25_smallEPSPs_df[minus25_smallEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
# filtered_df = minus25_smallEPSPs_df[minus25_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
filtered_minus25_smallEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(minus25_smallEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_minus25_smallEPSPs_df = minus25_smallEPSPs_df[minus25_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_minus25_smallEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_minus25_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### EPSP amplitude > 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'minus25')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [425]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [400]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus25_largeEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus25_largeEPSPs_df

In [ ]:
all_delays = [delay for delays in minus25_largeEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = minus25_largeEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(minus25_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_minus25_largeEPSPs_df = minus25_largeEPSPs_df[minus25_largeEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
filtered_minus25_largeEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(minus25_largeEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_minus25_largeEPSPs_df = minus25_largeEPSPs_df[minus25_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_minus25_largeEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_minus25_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### Analysis

In [ ]:
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, minus25_smallEPSPs_df, minus25_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-25 ms (<2.3 mV)', '-25 ms (>2.3 mV)'])
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, minus25_smallEPSPs_df, minus25_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-25 ms (<2.3 mV)', '-25 ms (>2.3 mV)'])

In [ ]:
plot_histogram([minus25_smallEPSPs_df, minus25_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['-25 ms (<2.3 mV)', '-25 ms (>2.3 mV)'])
plot_histogram([minus25_smallEPSPs_df, minus25_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['-25 ms (<2.3 mV)', '-25 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([minus25_smallEPSPs_df, minus25_largeEPSPs_df], labels=['-25 ms (<2.3 mV)', '-25 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([apical_only_minus25_smallEPSPs_df, apical_only_minus25_largeEPSPs_df], labels=['-25 ms (<2.3 mV)', '-25 ms (>2.3 mV)'])

###### Sjöström (only lower L4)

In [ ]:
filtered_synapses_only_smallEPSPs_df = synapses_only_smallEPSPs_df[synapses_only_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_smallEPSPs_df = filtered_synapses_only_smallEPSPs_df[filtered_synapses_only_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_synapses_only_largeEPSPs_df = synapses_only_largeEPSPs_df[synapses_only_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_largeEPSPs_df = filtered_synapses_only_largeEPSPs_df[filtered_synapses_only_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_minus25_smallEPSPs_df = minus25_smallEPSPs_df[minus25_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_minus25_smallEPSPs_df = filtered_minus25_smallEPSPs_df[filtered_minus25_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_minus25_largeEPSPs_df = minus25_largeEPSPs_df[minus25_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_minus25_largeEPSPs_df = filtered_minus25_largeEPSPs_df[filtered_minus25_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_minus25_largeEPSPs_df

In [ ]:
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_minus25_smallEPSPs_df, filtered_minus25_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-25 ms (<2.3 mV)', '-25 ms (>2.3 mV)'])
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_minus25_smallEPSPs_df, filtered_minus25_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-25 ms (<2.3 mV)', '-25 ms (>2.3 mV)'])

In [ ]:
plot_histogram([filtered_minus25_smallEPSPs_df, filtered_minus25_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['-25 ms (<2.3 mV)', '-25 ms (>2.3 mV)'])
plot_histogram([filtered_minus25_smallEPSPs_df, filtered_minus25_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['-25 ms (<2.3 mV)', '-25 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([filtered_minus25_smallEPSPs_df, filtered_minus25_largeEPSPs_df], labels=['-25 ms (<2.3 mV)', '-25 ms (>2.3 mV)'])

#### Analysis

In [ ]:
# plot_extracellular_syn_v_compare([plus25_smallEPSPs_df, minus25_smallEPSPs_df], labels=['+25 ms (<2.3 mV)', '-25 ms (<2.3 mV)'])

In [ ]:
# plot_extracellular_syn_v_compare([plus25_largeEPSPs_df, minus25_largeEPSPs_df], labels=['+25 ms (>2.3 mV)', '-25 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([plus25_smallEPSPs_df, minus25_smallEPSPs_df, plus25_largeEPSPs_df, minus25_largeEPSPs_df], labels=['+25 ms (<2.3 mV)', '-25 ms (<2.3 mV)', '+25 ms (>2.3 mV)', '-25 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([apical_only_plus25_smallEPSPs_df, apical_only_minus25_smallEPSPs_df, apical_only_plus25_largeEPSPs_df, apical_only_minus25_largeEPSPs_df], labels=['+25 ms (<2.3 mV)', '-25 ms (<2.3 mV)', '+25 ms (>2.3 mV)', '-25 ms (>2.3 mV)'])

#### +50 ms delay

##### 1 < EPSP amplitude < 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'plus50')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [450]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus50_smallEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus50_smallEPSPs_df

In [ ]:
all_delays = [delay for delays in plus50_smallEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus50_smallEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(plus50_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_plus50_smallEPSPs_df = plus50_smallEPSPs_df[plus50_smallEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
# filtered_df = plus50_smallEPSPs_df[plus50_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
filtered_plus50_smallEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(plus50_smallEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_plus50_smallEPSPs_df = plus50_smallEPSPs_df[plus50_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_plus50_smallEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_plus50_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### EPSP amplitude > 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'plus50')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [450]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus50_largeEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus50_largeEPSPs_df

In [ ]:
all_delays = [delay for delays in plus50_largeEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus50_largeEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(plus50_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_plus50_largeEPSPs_df = plus50_largeEPSPs_df[plus50_largeEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
filtered_plus50_largeEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(plus50_largeEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_plus50_largeEPSPs_df = plus50_largeEPSPs_df[plus50_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_plus50_largeEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_plus50_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### Analysis

In [ ]:
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, plus50_smallEPSPs_df, plus50_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+50 ms (<2.3 mV)', '+50 ms (>2.3 mV)'])
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, plus50_smallEPSPs_df, plus50_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+50 ms (<2.3 mV)', '+50 ms (>2.3 mV)'])

In [ ]:
plot_histogram([plus50_smallEPSPs_df, plus50_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['+50 ms (<2.3 mV)', '+50 ms (>2.3 mV)'])
plot_histogram([plus50_smallEPSPs_df, plus50_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['+50 ms (<2.3 mV)', '+50 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([plus50_smallEPSPs_df, plus50_largeEPSPs_df], labels=['+50 ms (<2.3 mV)', '+50 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([apical_only_plus50_smallEPSPs_df, apical_only_plus50_largeEPSPs_df], labels=['+50 ms (<2.3 mV)', '+50 ms (>2.3 mV)'])

###### Sjöström (only lower L4)

In [ ]:
filtered_synapses_only_smallEPSPs_df = synapses_only_smallEPSPs_df[synapses_only_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_smallEPSPs_df = filtered_synapses_only_smallEPSPs_df[filtered_synapses_only_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_synapses_only_largeEPSPs_df = synapses_only_largeEPSPs_df[synapses_only_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_largeEPSPs_df = filtered_synapses_only_largeEPSPs_df[filtered_synapses_only_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_plus50_smallEPSPs_df = plus50_smallEPSPs_df[plus50_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_plus50_smallEPSPs_df = filtered_plus50_smallEPSPs_df[filtered_plus50_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_plus50_largeEPSPs_df = plus50_largeEPSPs_df[plus50_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_plus50_largeEPSPs_df = filtered_plus50_largeEPSPs_df[filtered_plus50_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_plus50_largeEPSPs_df

In [ ]:
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_plus50_smallEPSPs_df, filtered_plus50_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+50 ms (<2.3 mV)', '+50 ms (>2.3 mV)'])
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_plus50_smallEPSPs_df, filtered_plus50_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+50 ms (<2.3 mV)', '+50 ms (>2.3 mV)'])

In [ ]:
plot_histogram([filtered_plus50_smallEPSPs_df, filtered_plus50_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['+50 ms (<2.3 mV)', '+50 ms (>2.3 mV)'])
plot_histogram([filtered_plus50_smallEPSPs_df, filtered_plus50_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['+50 ms (<2.3 mV)', '+50 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([filtered_plus50_smallEPSPs_df, filtered_plus50_largeEPSPs_df], labels=['+50 ms (<2.3 mV)', '+50 ms (>2.3 mV)'])

#### -50 ms delay

##### 1 < EPSP amplitude < 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'minus50')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [450]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [400]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus50_smallEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus50_smallEPSPs_df

In [ ]:
all_delays = [delay for delays in minus50_smallEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = minus50_smallEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(minus50_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_minus50_smallEPSPs_df = minus50_smallEPSPs_df[minus50_smallEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
# filtered_df = minus50_smallEPSPs_df[minus50_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
filtered_minus50_smallEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(minus50_smallEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_minus50_smallEPSPs_df = minus50_smallEPSPs_df[minus50_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_minus50_smallEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_minus50_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### EPSP amplitude > 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'minus50')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [450]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [400]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 700

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus50_largeEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus50_largeEPSPs_df

In [ ]:
all_delays = [delay for delays in minus50_largeEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = minus50_largeEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(minus50_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_minus50_largeEPSPs_df = minus50_largeEPSPs_df[minus50_largeEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
filtered_minus50_largeEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(minus50_largeEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_minus50_largeEPSPs_df = minus50_largeEPSPs_df[minus50_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_minus50_largeEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_minus50_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### Analysis

In [ ]:
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, minus50_smallEPSPs_df, minus50_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+50 ms (<2.3 mV)', '+50 ms (>2.3 mV)'])
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, minus50_smallEPSPs_df, minus50_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+50 ms (<2.3 mV)', '+50 ms (>2.3 mV)'])

In [ ]:
plot_histogram([minus50_smallEPSPs_df, minus50_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['-50 ms (<2.3 mV)', '-50 ms (>2.3 mV)'])
plot_histogram([minus50_smallEPSPs_df, minus50_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['-50 ms (<2.3 mV)', '-50 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([minus50_smallEPSPs_df, minus50_largeEPSPs_df], labels=['-50 ms (<2.3 mV)', '-50 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([apical_only_minus50_smallEPSPs_df, apical_only_minus50_largeEPSPs_df], labels=['-50 ms (<2.3 mV)', '-50 ms (>2.3 mV)'])

###### Sjöström (only lower L4)

In [ ]:
filtered_synapses_only_smallEPSPs_df = synapses_only_smallEPSPs_df[synapses_only_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_smallEPSPs_df = filtered_synapses_only_smallEPSPs_df[filtered_synapses_only_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_synapses_only_largeEPSPs_df = synapses_only_largeEPSPs_df[synapses_only_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_largeEPSPs_df = filtered_synapses_only_largeEPSPs_df[filtered_synapses_only_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_minus50_smallEPSPs_df = minus50_smallEPSPs_df[minus50_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_minus50_smallEPSPs_df = filtered_minus50_smallEPSPs_df[filtered_minus50_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_minus50_largeEPSPs_df = minus50_largeEPSPs_df[minus50_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_minus50_largeEPSPs_df = filtered_minus50_largeEPSPs_df[filtered_minus50_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_minus50_largeEPSPs_df

In [ ]:
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_minus50_smallEPSPs_df, filtered_minus50_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-50 ms (<2.3 mV)', '-50 ms (>2.3 mV)'])
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_minus50_smallEPSPs_df, filtered_minus50_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-50 ms (<2.3 mV)', '-50 ms (>2.3 mV)'])

In [ ]:
plot_histogram([filtered_minus50_smallEPSPs_df, filtered_minus50_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['-50 ms (<2.3 mV)', '-50 ms (>2.3 mV)'])
plot_histogram([filtered_minus50_smallEPSPs_df, filtered_minus50_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['-50 ms (<2.3 mV)', '-50 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([filtered_minus50_smallEPSPs_df, filtered_minus50_largeEPSPs_df], labels=['-50 ms (<2.3 mV)', '-50 ms (>2.3 mV)'])

#### Analysis

In [ ]:
# plot_extracellular_syn_v_compare([plus50_smallEPSPs_df, minus10_smallEPSPs_df], labels=['+50 ms (<2.3 mV)', '-50 ms (<2.3 mV)'])

In [ ]:
# plot_extracellular_syn_v_compare([plus50_largeEPSPs_df, minus50_largeEPSPs_df], labels=['+50 ms (>2.3 mV)', '-50 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([plus50_smallEPSPs_df, minus50_smallEPSPs_df, plus50_largeEPSPs_df, minus50_largeEPSPs_df], labels=['+50 ms (<2.3 mV)', '-50 ms (<2.3 mV)', '+50 ms (>2.3 mV)', '-50 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([apical_only_plus50_smallEPSPs_df, apical_only_minus50_smallEPSPs_df, apical_only_plus50_largeEPSPs_df, apical_only_minus50_largeEPSPs_df], labels=['+50 ms (<2.3 mV)', '-50 ms (<2.3 mV)', '+50 ms (>2.3 mV)', '-50 ms (>2.3 mV)'])

#### +100 ms delay

##### 1 < EPSP amplitude < 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'plus100')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [500]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 800

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus100_smallEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus100_smallEPSPs_df

In [ ]:
all_delays = [delay for delays in plus100_smallEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus100_smallEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(plus100_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_plus100_smallEPSPs_df = plus100_smallEPSPs_df[plus100_smallEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
# filtered_df = plus100_smallEPSPs_df[plus100_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
filtered_plus100_smallEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(plus100_smallEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_plus100_smallEPSPs_df = plus100_smallEPSPs_df[plus100_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_plus100_smallEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_plus100_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### EPSP amplitude > 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'plus100')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [500]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 800

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus100_largeEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus100_largeEPSPs_df

In [ ]:
all_delays = [delay for delays in plus100_largeEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus100_largeEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(plus100_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_plus100_largeEPSPs_df = plus100_largeEPSPs_df[plus100_largeEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
filtered_plus100_largeEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(plus100_largeEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_plus100_largeEPSPs_df = plus100_largeEPSPs_df[plus100_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_plus100_largeEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_plus100_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### Analysis

In [ ]:
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, plus100_smallEPSPs_df, plus100_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+100 ms (<2.3 mV)', '+100 ms (>2.3 mV)'])
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, plus100_smallEPSPs_df, plus100_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+100 ms (<2.3 mV)', '+100 ms (>2.3 mV)'])

In [ ]:
plot_histogram([plus100_smallEPSPs_df, plus100_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['+100 ms (<2.3 mV)', '+100 ms (>2.3 mV)'])
plot_histogram([plus100_smallEPSPs_df, plus100_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['+100 ms (<2.3 mV)', '+100 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([plus100_smallEPSPs_df, plus100_largeEPSPs_df], labels=['+100 ms (<2.3 mV)', '+100 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([apical_only_plus100_smallEPSPs_df, apical_only_plus100_largeEPSPs_df], labels=['+100 ms (<2.3 mV)', '+100 ms (>2.3 mV)'])

###### Sjöström (only lower L4)

In [ ]:
filtered_synapses_only_smallEPSPs_df = synapses_only_smallEPSPs_df[synapses_only_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_smallEPSPs_df = filtered_synapses_only_smallEPSPs_df[filtered_synapses_only_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_synapses_only_largeEPSPs_df = synapses_only_largeEPSPs_df[synapses_only_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_largeEPSPs_df = filtered_synapses_only_largeEPSPs_df[filtered_synapses_only_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_plus100_smallEPSPs_df = plus100_smallEPSPs_df[plus100_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_plus100_smallEPSPs_df = filtered_plus100_smallEPSPs_df[filtered_plus100_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_plus100_largeEPSPs_df = plus100_largeEPSPs_df[plus100_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_plus100_largeEPSPs_df = filtered_plus100_largeEPSPs_df[filtered_plus100_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_plus100_largeEPSPs_df

In [ ]:
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_plus100_smallEPSPs_df, filtered_plus100_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+100 ms (<2.3 mV)', '+100 ms (>2.3 mV)'])
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_plus100_smallEPSPs_df, filtered_plus100_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '+100 ms (<2.3 mV)', '+100 ms (>2.3 mV)'])

In [ ]:
plot_histogram([filtered_plus100_smallEPSPs_df, filtered_plus100_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['+100 ms (<2.3 mV)', '+100 ms (>2.3 mV)'])
plot_histogram([filtered_plus100_smallEPSPs_df, filtered_plus100_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['+100 ms (<2.3 mV)', '+100 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([filtered_plus100_smallEPSPs_df, filtered_plus100_largeEPSPs_df], labels=['+100 ms (<2.3 mV)', '+100 ms (>2.3 mV)'])

#### -100 ms delay

##### 1 < EPSP amplitude < 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'minus100')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'smallEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [500]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [400]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 800

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus100_smallEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus100_smallEPSPs_df

In [ ]:
all_delays = [delay for delays in minus100_smallEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = minus100_smallEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(minus100_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_minus100_smallEPSPs_df = minus100_smallEPSPs_df[minus100_smallEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
# filtered_df = minus100_smallEPSPs_df[minus100_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
filtered_minus100_smallEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(minus100_smallEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_minus100_smallEPSPs_df = minus100_smallEPSPs_df[minus100_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_minus100_smallEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_minus100_smallEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### EPSP amplitude > 2.3

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'SjöströmEtAl2001')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'Timing')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'SynapsesOnly')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

synapse_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(synapse_results_dir):
    I.os.mkdir(synapse_results_dir)

In [ ]:
protocol_results_dir = I.os.path.join(group_results_dir, 'minus100')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'largeEPSPs')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [500]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [400]
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
protocol_params['STDP.run.tStop'] = 800

In [ ]:
run_protocol_extracellular_stim(simulation_results_dir, synapse_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus100_largeEPSPs_df = load_results_extracellular_stim(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus100_largeEPSPs_df

In [ ]:
all_delays = [delay for delays in minus100_largeEPSPs_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = minus100_largeEPSPs_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_extracellular_syn_v(minus100_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

In [ ]:
filtered_minus100_largeEPSPs_df = minus100_largeEPSPs_df[minus100_largeEPSPs_df['soma_spike_heights'].apply(lambda x: len(x) > 1)]
filtered_minus100_largeEPSPs_df['soma_dist']

In [ ]:
plot_extracellular_syn_v(minus100_largeEPSPs_df, c_column='num_soma_spikes', c_label='Num soma spikes')

In [ ]:
apical_only_minus100_largeEPSPs_df = minus100_largeEPSPs_df[minus100_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]
apical_only_minus100_largeEPSPs_df

In [ ]:
plot_extracellular_syn_v(apical_only_minus100_largeEPSPs_df, c_column='num_syns', c_label='Num synapses')

##### Analysis

In [ ]:
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, minus100_smallEPSPs_df, minus100_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-100 ms (<2.3 mV)', '-100 ms (>2.3 mV)'])
plot_histogram([synapses_only_smallEPSPs_df, synapses_only_largeEPSPs_df, minus100_smallEPSPs_df, minus100_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-100 ms (<2.3 mV)', '-100 ms (>2.3 mV)'])

In [ ]:
plot_histogram([minus100_smallEPSPs_df, minus100_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['-100 ms (<2.3 mV)', '-100 ms (>2.3 mV)'])
plot_histogram([minus100_smallEPSPs_df, minus100_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['-100 ms (<2.3 mV)', '-100 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([minus100_smallEPSPs_df, minus100_largeEPSPs_df], labels=['-100 ms (<2.3 mV)', '-100 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([apical_only_minus100_smallEPSPs_df, apical_only_minus100_largeEPSPs_df], labels=['-100 ms (<2.3 mV)', '-100 ms (>2.3 mV)'])

###### Sjöström (only lower L4)

In [ ]:
filtered_synapses_only_smallEPSPs_df = synapses_only_smallEPSPs_df[synapses_only_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_smallEPSPs_df = filtered_synapses_only_smallEPSPs_df[filtered_synapses_only_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_synapses_only_largeEPSPs_df = synapses_only_largeEPSPs_df[synapses_only_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_synapses_only_largeEPSPs_df = filtered_synapses_only_largeEPSPs_df[filtered_synapses_only_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_minus100_smallEPSPs_df = minus100_smallEPSPs_df[minus100_smallEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_minus100_smallEPSPs_df = filtered_minus100_smallEPSPs_df[filtered_minus100_smallEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_minus100_largeEPSPs_df = minus100_largeEPSPs_df[minus100_largeEPSPs_df['mean_soma_dist'].apply(lambda x: x > 50 and x < 250)]
# filtered_minus100_largeEPSPs_df = filtered_minus100_largeEPSPs_df[filtered_minus100_largeEPSPs_df['syn_dendrite_type'].apply(lambda x: not 'basal' in x)]

filtered_minus100_largeEPSPs_df

In [ ]:
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_minus100_smallEPSPs_df, filtered_minus100_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-100 ms (<2.3 mV)', '-100 ms (>2.3 mV)'])
plot_histogram([filtered_synapses_only_smallEPSPs_df, filtered_synapses_only_largeEPSPs_df, filtered_minus100_smallEPSPs_df, filtered_minus100_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['<2.3 mV', '>2.3 mV', '-100 ms (<2.3 mV)', '-100 ms (>2.3 mV)'])

In [ ]:
plot_histogram([filtered_minus100_smallEPSPs_df, filtered_minus100_largeEPSPs_df], 'mean_syn_peak_heights', xlabel='Mean Vm peak at synapse (mV)', ylabel='Num runs', labels=['-100 ms (<2.3 mV)', '-100 ms (>2.3 mV)'])
plot_histogram([filtered_minus100_smallEPSPs_df, filtered_minus100_largeEPSPs_df], 'mean_syn_auc_total', xlabel='Mean Vm AUC at synapse', ylabel='Num runs', labels=['-100 ms (<2.3 mV)', '-100 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([filtered_minus100_smallEPSPs_df, filtered_minus100_largeEPSPs_df], labels=['-100 ms (<2.3 mV)', '-100 ms (>2.3 mV)'])

#### Analysis

In [ ]:
# plot_extracellular_syn_v_compare([plus100_smallEPSPs_df, minus100_smallEPSPs_df], labels=['+100 ms (<2.3 mV)', '-100 ms (<2.3 mV)'])

In [ ]:
# plot_extracellular_syn_v_compare([plus100_largeEPSPs_df, minus100_largeEPSPs_df], labels=['+100 ms (>2.3 mV)', '-100 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([plus100_smallEPSPs_df, minus100_smallEPSPs_df, plus100_largeEPSPs_df, minus100_largeEPSPs_df], labels=['+100 ms (<2.3 mV)', '-100 ms (<2.3 mV)', '+100 ms (>2.3 mV)', '-100 ms (>2.3 mV)'])

In [ ]:
plot_extracellular_syn_v_compare([apical_only_plus100_smallEPSPs_df, apical_only_minus100_smallEPSPs_df, apical_only_plus100_largeEPSPs_df, apical_only_minus100_largeEPSPs_df], labels=['+100 ms (<2.3 mV)', '-100 ms (<2.3 mV)', '+100 ms (>2.3 mV)', '-100 ms (>2.3 mV)'])

## Letzkus et al., 2006

### 3 AP burst

#### Somatic current injection only

##### Find somatic current stimulation parameters

In [ ]:
from biophysics_fitting.ephys import find_crossing

amps = I.np.arange(1, 5.1, step=0.1)

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

breaker = False
for amp in amps:
    amp = round(amp, 1)

    cell_params = loaded_cell_params.loc[biophysics_id].copy()

    cell_params['SomaticCurrentInjection.stim.current_amp'] = amp
    cell_params['SomaticCurrentInjection.stim.current_inj_times'] = I.np.arange(start=410, stop=425, step=5)
    cell_params['SomaticCurrentInjection.stim.current_duration'] = 2
    cell_params['SomaticCurrentInjection.run.tStop'] = 800

    simulated_cell, param = loaded_simulator.get_simulated_cell(cell_params, 'SomaticCurrentInjection')

    spikes = find_crossing(simulated_cell.soma.recVList[0], thresh=0)

    apical_sect = _get_apical_sec_and_i_at_distance(simulated_cell, loaded_fixed_params['BAC.stim.dist'])
    vm_dend = I.np.array(apical_sect[0].recVList)

    if len(spikes[0]) > len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        print('Multiple spikes')
        # I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        break
    elif len(spikes[0]) == len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        spike_amp = amp
        # time_to_spike = simulated_cell.tVec[spikes[0][0]] - cell_params['SomaticCurrentInjection.stim.current_inj_times'][0]
        print(spike_amp)
        print(simulated_cell.tVec[spikes[0][0]])
        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        I.plt.plot(simulated_cell.tVec, vm_dend[apical_sect[2]], label=f'{amp} (dend)', linestyle='--')

        # Run 1 extra increment of 0.1 when reaching "rheobase"
        if breaker:
            break
        breaker = True
    else:
        continue

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.xlim(left=390)
I.plt.ylabel('Membrane potential (mV)')
I.plt.legend()
I.plt.show()

##### Somatic current injection control

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'LetzkusEtAl2006')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '3APBurst')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'SomaticCurrentInjection')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['SomaticCurrentInjection.stim.current_amp'] = spike_amp
protocol_params['SomaticCurrentInjection.stim.current_inj_times'] = I.np.arange(start=410, stop=425, step=5)
protocol_params['SomaticCurrentInjection.stim.current_duration'] = 2
protocol_params['SomaticCurrentInjection.run.tStop'] = 800

In [ ]:
run_protocol_soma_only(simulation_results_dir, loaded_simulator, loaded_cell_params.loc[biophysics_id], protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

soma_only_3APBurst_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
soma_only_3APBurst_df

In [ ]:
plot_traces(
    soma_only_3APBurst_df,
    inputs=[
        {
            'stim_times': cell_params['SomaticCurrentInjection.stim.current_inj_times'],
            'label': 'Somatic current (nA)',
            'amp': cell_params['SomaticCurrentInjection.stim.current_amp'],
            'duration': cell_params['SomaticCurrentInjection.stim.current_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, None),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
# from visualize.cell_morphology_visualizer import CellMorphologyVisualizer

# images_path = I.os.path.join(results_dir, 'soma_spike_animation_3d')

# # To remake the visualization, uncomment this code
# if I.os.path.exists(images_path):
#     I.shutil.rmtree(images_path)

# cmv = CellMorphologyVisualizer(simulated_cell)
# cmv.population_to_color_dict['inactive'] = "#f0f0f0"  # add color for inactive
# # cmv.population_to_color_dict['Generic'] = "red"

# cmv.animation(
#     images_path=images_path, 
#     color="voltage",
#     client=client, 
#     t_start=400-10, t_stop=440, t_step=0.2
# )

In [ ]:
plot_syn_v_scatter(soma_only_3APBurst_df)

#### +10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'LetzkusEtAl2006')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '3APBurst')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'plus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = I.np.arange(start=410, stop=425, step=5)
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
# protocol_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
# protocol_params['STDP.stim.pairing_delay'] = 10
protocol_params['STDP.run.tStop'] = 800

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus10_3APBurst_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus10_3APBurst_df

In [ ]:
plot_traces(
    plus10_3APBurst_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, 10),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
all_delays = [delay for delays in plus10_3APBurst_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus10_3APBurst_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_syn_v_scatter(plus10_3APBurst_df)

In [ ]:
plot_syn_v_comparisons([plus10_3APBurst_df, soma_only_3APBurst_df, synapses_only_1Hz_df], labels=['+10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

#### -10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'LetzkusEtAl2006')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '3APBurst')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'minus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = [400]  # I.np.arange(start=300, stop=600, step=200)  # 10 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = I.np.arange(start=390, stop=405, step=5)
protocol_params['STDP.stim.postsyn_stim_amp'] = spike_amp
protocol_params['STDP.stim.postsyn_stim_duration'] = 2
# protocol_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
# protocol_params['STDP.stim.pairing_delay'] = -10
protocol_params['STDP.run.tStop'] = 800

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'STDP', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

minus10_3APBurst_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
minus10_3APBurst_df

In [ ]:
plot_traces(
    minus10_3APBurst_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, 15),
    x_lim_soma_inset=(400, 425), y_lim_soma_inset=(-70, -55)
)

In [ ]:
all_delays = [delay for delays in minus10_3APBurst_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

# df = minus10_3APBurst_df.copy()

# df = df.dropna(subset=['actual_delay'])
# df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

# I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

# I.plt.xlabel('Delay between somatic and synaptic spike')
# I.plt.ylabel('Num synapses')

# %matplotlib inline
# I.plt.show()

In [ ]:
plot_syn_v_scatter(minus10_3APBurst_df)

In [ ]:
plot_syn_v_comparisons([minus10_3APBurst_df, soma_only_3APBurst_df, synapses_only_1Hz_df], labels=['-10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

#### Analysis

In [ ]:
plot_syn_v_hist([synapses_only_1Hz_df, soma_only_3APBurst_df, plus10_3APBurst_df, minus10_3APBurst_df], labels=['Glutamate uncaging', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_comparisons([plus10_3APBurst_df, minus10_3APBurst_df], labels=['+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_violin([synapses_only_1Hz_df, soma_only_3APBurst_df, plus10_3APBurst_df, minus10_3APBurst_df], labels=['Glutamate uncaging', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

##### Letzkus (only L23-L5PT)

In [ ]:
filtered_soma_only_3APBurst_df = soma_only_3APBurst_df[soma_only_3APBurst_df['presyn_cell_type'].apply(lambda x: 'L2' in x or 'L34' in x)]
filtered_synapses_only_1Hz_df = synapses_only_1Hz_df[synapses_only_1Hz_df['presyn_cell_type'].apply(lambda x: 'L2' in x or 'L34' in x)]
filtered_plus10_3APBurst_df = plus10_3APBurst_df[plus10_3APBurst_df['presyn_cell_type'].apply(lambda x: 'L2' in x or 'L34' in x)]
filtered_minus10_3APBurst_df = minus10_3APBurst_df[minus10_3APBurst_df['presyn_cell_type'].apply(lambda x: 'L2' in x or 'L34' in x)]

filtered_plus10_3APBurst_df

In [ ]:
plot_syn_v_hist([filtered_synapses_only_1Hz_df, filtered_soma_only_3APBurst_df, filtered_plus10_3APBurst_df, filtered_minus10_3APBurst_df], labels=['Synaptic stimulation', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_scatter(filtered_plus10_3APBurst_df)

In [ ]:
plot_syn_v_scatter(filtered_minus10_3APBurst_df)

In [ ]:
plot_syn_v_comparisons([filtered_plus10_3APBurst_df, filtered_soma_only_3APBurst_df, filtered_synapses_only_1Hz_df], labels=['+10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

In [ ]:
plot_syn_v_comparisons([filtered_minus10_3APBurst_df, filtered_soma_only_3APBurst_df, filtered_synapses_only_1Hz_df], labels=['-10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

In [ ]:
plot_syn_v_comparisons([filtered_plus10_3APBurst_df, filtered_minus10_3APBurst_df], labels=['+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_violin([filtered_synapses_only_1Hz_df, filtered_soma_only_3APBurst_df, filtered_plus10_3APBurst_df, filtered_minus10_3APBurst_df], labels=['Synaptic stimulation', 'Somatic current injection', '+10 ms delay', '-10 ms delay'])

In [ ]:
plot_syn_v_hist([filtered_plus10_3APBurst_df, filtered_plus10_1Hz_df], labels=['+10 ms delay (Letzkus)', '+10 ms delay (Sjöström)'])

In [ ]:
# data = [soma_only_1Hz_df['syn_max_peak_height'], synapses_only_1Hz_df['syn_max_peak_height'], plus10_1Hz_df['syn_max_peak_height'], minus10_1Hz_df['syn_max_peak_height'], plus10_10Hz_df['syn_max_peak_height'], minus10_10Hz_df['syn_max_peak_height'], plus10_20Hz_df['syn_max_peak_height'], minus10_20Hz_df['syn_max_peak_height'], plus10_40Hz_df['syn_max_peak_height'], minus10_40Hz_df['syn_max_peak_height']]

# fig, ax = I.plt.subplots()
# ax.violinplot(data, showmeans=True, showmedians=False, showextrema=False)

# # Set custom categorical labels for the x-axis
# ax.set_xticks(I.np.arange(1, len(data)+1))
# ax.set_xticklabels(['Somatic spike (1 Hz)', 'Glutamate uncaging (1 Hz)', '+10 ms delay (1 Hz)', '-10 ms delay (1 Hz)', '+10 ms delay (5x10 Hz)', '-10 ms delay x 5 (5x10 Hz)', '+10 ms delay (5x20 Hz)', '-10 ms delay x 5 (5x20 Hz)', '+10 ms delay (5x40 Hz)', '-10 ms delay x 5 (5x40 Hz)'], rotation=45)

# ax.set_ylabel('Max Vm at synapse (mV)')

# %matplotlib inline
# I.plt.show()

### BAC

#### Find somatic current stimulation parameters

In [ ]:
from biophysics_fitting.ephys import find_crossing

amps = I.np.arange(1, 5.1, step=0.1)

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

breaker = False
for amp in amps:
    amp = round(amp, 1)

    cell_params = loaded_cell_params.loc[biophysics_id].copy()

    cell_params['SomaticCurrentInjection.stim.current_amp'] = amp
    cell_params['SomaticCurrentInjection.stim.current_inj_times'] = [400]
    cell_params['SomaticCurrentInjection.stim.current_duration'] = 2
    cell_params['SomaticCurrentInjection.run.tStop'] = 700

    simulated_cell, param = loaded_simulator.get_simulated_cell(cell_params, 'SomaticCurrentInjection')

    spikes = find_crossing(simulated_cell.soma.recVList[0], thresh=0)

    if len(spikes[0]) > len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        print('Multiple spikes')
        # I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        break
    elif len(spikes[0]) == len(cell_params['SomaticCurrentInjection.stim.current_inj_times']):
        spike_amp = amp
        # time_to_spike = simulated_cell.tVec[spikes[0][0]] - cell_params['SomaticCurrentInjection.stim.current_inj_times'][0]
        print(spike_amp)
        print(simulated_cell.tVec[spikes[0][0]])
        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)

        # Run 1 extra increment of 0.1 when reaching "rheobase"
        if breaker:
            break
        breaker = True
    else:
        continue

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.xlim(left=390)
I.plt.ylabel('Membrane potential (mV)')
I.plt.legend()
I.plt.show()

#### Find dendritic current stimulation parameters

In [ ]:
from biophysics_fitting.ephys import find_crossing

amps = I.np.arange(0.05, 2.1, step=0.01)

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

breaker = False
for amp in amps:
    amp = round(amp, 2)

    cell_params = loaded_cell_params.loc[biophysics_id].copy()

    cell_params['LetzkusEtAl2006_BAC.stim.synapses'] = []
    cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_amp'] = spike_amp
    cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times'] = [400]
    cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_duration'] = 2
    cell_params['LetzkusEtAl2006_BAC.stim.dend_stim_times'] = [400]
    cell_params['LetzkusEtAl2006_BAC.stim.dend_stim_amp'] = amp
    cell_params['LetzkusEtAl2006_BAC.stim.dend_stim_duration'] = 100
    cell_params['LetzkusEtAl2006_BAC.stim.dend_stim_dist'] = loaded_fixed_params['BAC.stim.dist']
    cell_params['LetzkusEtAl2006_BAC.measure.recSite'] = loaded_fixed_params['BAC.stim.dist']
    cell_params['LetzkusEtAl2006_BAC.run.tStop'] = 700

    simulated_cell, param = loaded_simulator.get_simulated_cell(cell_params, 'LetzkusEtAl2006_BAC')

    spikes = find_crossing(simulated_cell.soma.recVList[0], thresh=0)

    apical_sect = _get_apical_sec_and_i_at_distance(simulated_cell, cell_params['LetzkusEtAl2006_BAC.stim.dend_stim_dist'])
    vm_dend = I.np.array(apical_sect[0].recVList)

    if len(spikes[0]) > len(cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times']):
        bac_spike_amp = amp

        print(bac_spike_amp)
        print('Multiple spikes')

        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        I.plt.plot(simulated_cell.tVec, vm_dend[apical_sect[2]], label=f'{amp} (dend)', linestyle='--')

        # Run 1 extra increment of 0.01 when reaching BAC
        if breaker:
            break
        breaker = True

    elif len(spikes[0]) == len(cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times']):
        subthresh_bac_spike_amp = amp
        # time_to_spike = simulated_cell.tVec[spikes[0][0]] - cell_params['SomaticCurrentInjection.stim.current_inj_times'][0]

        print(subthresh_bac_spike_amp)
        print(simulated_cell.tVec[spikes[0][0]])

        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        I.plt.plot(simulated_cell.tVec, vm_dend[apical_sect[2]], label=f'{amp} (dend)', linestyle='--')
    else:
        continue

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.xlim(left=390)
I.plt.ylabel('Membrane potential (mV)')
I.plt.legend()
I.plt.show()

#### Subthreshold

##### Postsynaptic current injection control

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'LetzkusEtAl2006')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'BAC')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'Subthreshold')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'PostsynapticCurrentInjection')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['LetzkusEtAl2006_BAC.stim.synapses'] = []
protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_amp'] = spike_amp
protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times'] = [400]
protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_duration'] = 2
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_times'] = [400]
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_amp'] = subthresh_bac_spike_amp
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_duration'] = 100
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_dist'] = loaded_fixed_params['BAC.stim.dist']
protocol_params['LetzkusEtAl2006_BAC.measure.recSite'] = loaded_fixed_params['BAC.stim.dist']
protocol_params['LetzkusEtAl2006_BAC.run.tStop'] = 700

In [ ]:
run_protocol_soma_only(simulation_results_dir, loaded_simulator, loaded_cell_params.loc[biophysics_id], protocol_params, protocol_name='LetzkusEtAl2006_BAC')

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

soma_only_subthresh_bac_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
soma_only_subthresh_bac_df

In [ ]:
plot_traces(
    soma_only_subthresh_bac_df,
    inputs=[
        {
            'stim_times': cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_amp'],
            'duration': cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, None),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
# from visualize.cell_morphology_visualizer import CellMorphologyVisualizer

# images_path = I.os.path.join(results_dir, 'soma_spike_animation_3d')

# # To remake the visualization, uncomment this code
# if I.os.path.exists(images_path):
#     I.shutil.rmtree(images_path)

# cmv = CellMorphologyVisualizer(simulated_cell)
# cmv.population_to_color_dict['inactive'] = "#f0f0f0"  # add color for inactive
# # cmv.population_to_color_dict['Generic'] = "red"

# cmv.animation(
#     images_path=images_path, 
#     color="voltage",
#     client=client, 
#     t_start=400-10, t_stop=440, t_step=0.2
# )

In [ ]:
plot_syn_v_scatter(soma_only_subthresh_bac_df)

##### +10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'LetzkusEtAl2006')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'BAC')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'Subthreshold')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'plus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['LetzkusEtAl2006_BAC.stim.syn_stim_times'] = [400]
protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_amp'] = spike_amp
protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times'] = [410]
protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_duration'] = 2
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_times'] = [410]
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_amp'] = subthresh_bac_spike_amp
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_duration'] = 100
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_dist'] = loaded_fixed_params['BAC.stim.dist']
protocol_params['LetzkusEtAl2006_BAC.measure.recSite'] = loaded_fixed_params['BAC.stim.dist']
protocol_params['LetzkusEtAl2006_BAC.run.tStop'] = 800

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'LetzkusEtAl2006_BAC', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus10_subthresh_bac_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus10_subthresh_bac_df

In [ ]:
plot_traces(
    plus10_subthresh_bac_df,
    inputs=[
        {
            'stim_times': protocol_params['LetzkusEtAl2006_BAC.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_amp'],
            'duration': protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, 10),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
all_delays = [delay for delays in plus10_subthresh_bac_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus10_subthresh_bac_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_syn_v_scatter(plus10_subthresh_bac_df)

In [ ]:
plot_syn_v_comparisons([plus10_subthresh_bac_df, soma_only_subthresh_bac_df, synapses_only_1Hz_df], labels=['+10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

In [ ]:
plot_syn_v_hist([synapses_only_1Hz_df, soma_only_subthresh_bac_df, plus10_subthresh_bac_df], labels=['Glutamate uncaging', 'Subthreshold BAC injection', '+10 ms delay'])

#### Suprathreshold

##### Postsynaptic current injection control

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'LetzkusEtAl2006')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'BAC')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'Suprathreshold')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'PostsynapticCurrentInjection')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['LetzkusEtAl2006_BAC.stim.synapses'] = []
protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_amp'] = spike_amp
protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times'] = [400]
protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_duration'] = 2
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_times'] = [400]
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_amp'] = bac_spike_amp
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_duration'] = 100
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_dist'] = loaded_fixed_params['BAC.stim.dist']
protocol_params['LetzkusEtAl2006_BAC.measure.recSite'] = loaded_fixed_params['BAC.stim.dist']
protocol_params['LetzkusEtAl2006_BAC.run.tStop'] = 700

In [ ]:
run_protocol_soma_only(simulation_results_dir, loaded_simulator, loaded_cell_params.loc[biophysics_id], protocol_params, protocol_name='LetzkusEtAl2006_BAC')

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

soma_only_bac_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
soma_only_bac_df

In [ ]:
plot_traces(
    soma_only_bac_df,
    inputs=[
        {
            'stim_times': cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_amp'],
            'duration': cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, None),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
from visualize.cell_morphology_visualizer import CellMorphologyVisualizer

client = I.get_client(ip='localhost', client_port=8786)

images_path = I.os.path.join(simulation_results_dir, 'soma_spike_animation_3d')

# To remake the visualization, uncomment this code
if I.os.path.exists(images_path):
    I.shutil.rmtree(images_path)
else:
    I.os.mkdir(images_path)

cmv = CellMorphologyVisualizer(simulated_cell)
cmv.population_to_color_dict['inactive'] = "#f0f0f0"  # add color for inactive
# cmv.population_to_color_dict['Generic'] = "red"

cmv.animation(
    images_path=images_path, 
    color="voltage",
    client=client, 
    t_start=400-5, t_stop=450, t_step=0.2
)

In [ ]:
plot_syn_v_scatter(soma_only_bac_df)

##### +10 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'LetzkusEtAl2006')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

group_results_dir = I.os.path.join(paper_results_dir, 'BAC')
if not I.os.path.exists(group_results_dir):
    I.os.mkdir(group_results_dir)

protocol_results_dir = I.os.path.join(group_results_dir, 'Suprathreshold')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'plus10')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['LetzkusEtAl2006_BAC.stim.syn_stim_times'] = [400]
protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_amp'] = spike_amp
protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times'] = [410]
protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_duration'] = 2
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_times'] = [410]
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_amp'] = bac_spike_amp
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_duration'] = 100
protocol_params['LetzkusEtAl2006_BAC.stim.dend_stim_dist'] = loaded_fixed_params['BAC.stim.dist']
protocol_params['LetzkusEtAl2006_BAC.measure.recSite'] = loaded_fixed_params['BAC.stim.dist']
protocol_params['LetzkusEtAl2006_BAC.run.tStop'] = 800

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'LetzkusEtAl2006_BAC', loaded_simulator, loaded_cell_params.loc[biophysics_id], loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

plus10_bac_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
plus10_bac_df

In [ ]:
plot_traces(
    plus10_bac_df,
    inputs=[
        {
            'stim_times': protocol_params['LetzkusEtAl2006_BAC.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_amp'],
            'duration': protocol_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, 10),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
all_delays = [delay for delays in plus10_bac_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = plus10_bac_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_syn_v_scatter(plus10_bac_df)

In [ ]:
plot_syn_v_comparisons([plus10_bac_df, soma_only_bac_df, synapses_only_1Hz_df], labels=['+10 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

In [ ]:
plot_syn_v_hist([synapses_only_1Hz_df, soma_only_bac_df, plus10_bac_df], labels=['Glutamate uncaging', 'Suprathreshold BAC injection', '+10 ms delay'])

#### Analysis

In [ ]:
plot_syn_v_hist([synapses_only_1Hz_df, soma_only_subthresh_bac_df, plus10_subthresh_bac_df, soma_only_bac_df, plus10_bac_df], labels=['Glutamate uncaging', 'Subthreshold BAC injection', '+10 ms delay (subthreshold)', 'Suprathreshold BAC injection', '+10 ms delay (suprathreshold)'])

In [ ]:
plot_syn_v_comparisons([plus10_bac_df, plus10_subthresh_bac_df], labels=['+10 ms delay (suprathreshold)', '+10 ms delay (subthreshold)'])

In [ ]:
plot_syn_v_violin([synapses_only_1Hz_df, soma_only_subthresh_bac_df, plus10_subthresh_bac_df, soma_only_bac_df, plus10_bac_df], labels=['Glutamate uncaging', 'Subthreshold BAC injection', '+10 ms delay (subthreshold)', 'Suprathreshold BAC injection', '+10 ms delay (suprathreshold)'])

##### Letzkus (only L23-L5PT)

In [ ]:
filtered_synapses_only_1Hz_df = synapses_only_1Hz_df[synapses_only_1Hz_df['presyn_cell_type'].apply(lambda x: 'L2' in x or 'L34' in x)]

filtered_soma_only_subthresh_bac_df = soma_only_subthresh_bac_df[soma_only_subthresh_bac_df['presyn_cell_type'].apply(lambda x: 'L2' in x or 'L34' in x)]
filtered_soma_only_bac_df = soma_only_bac_df[soma_only_bac_df['presyn_cell_type'].apply(lambda x: 'L2' in x or 'L34' in x)]

filtered_plus10_subthresh_bac_df = plus10_subthresh_bac_df[plus10_subthresh_bac_df['presyn_cell_type'].apply(lambda x: 'L2' in x or 'L34' in x)]
filtered_plus10_bac_df = plus10_bac_df[plus10_bac_df['presyn_cell_type'].apply(lambda x: 'L2' in x or 'L34' in x)]

filtered_plus10_subthresh_bac_df

In [ ]:
plot_syn_v_hist([filtered_synapses_only_1Hz_df, filtered_soma_only_subthresh_bac_df, filtered_plus10_subthresh_bac_df, filtered_soma_only_bac_df, filtered_plus10_bac_df], labels=['Glutamate uncaging', 'Subthreshold BAC injection', '+10 ms delay (subthreshold)', 'Suprathreshold BAC injection', '+10 ms delay (suprathreshold)'])

In [ ]:
plot_syn_v_comparisons([filtered_plus10_subthresh_bac_df, filtered_soma_only_subthresh_bac_df, filtered_synapses_only_1Hz_df], labels=['+10 ms delay', 'Subthreshold BAC injection', 'Glutamate uncaging'])

In [ ]:
plot_syn_v_comparisons([filtered_plus10_bac_df, filtered_soma_only_bac_df, filtered_synapses_only_1Hz_df], labels=['+10 ms delay', 'Suprathreshold BAC injection', 'Glutamate uncaging'])

In [ ]:
plot_syn_v_comparisons([filtered_plus10_bac_df, filtered_plus10_subthresh_bac_df], labels=['+10 ms delay (suprathreshold)', '+10 ms delay (subthreshold)'])

In [ ]:
plot_syn_v_violin([filtered_synapses_only_1Hz_df, filtered_soma_only_subthresh_bac_df, filtered_plus10_subthresh_bac_df, filtered_soma_only_bac_df, filtered_plus10_bac_df], labels=['Glutamate uncaging', 'Subthreshold BAC injection', '+10 ms delay (subthreshold)', 'Suprathreshold BAC injection', '+10 ms delay (suprathreshold)'])

## Bittner et al., 2017 / Caya-Bissonnette et al., 2023

### Synaptic stimulation only

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'BittnerEtAl2017')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '20Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'GlutamateUncaging')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['GlutamateUncaging.stim.syn_stim_times'] = I.np.arange(start=400, stop=900, step=50)  # 20 Hz
protocol_params['GlutamateUncaging.run.tStop'] = 1200

cell_params = loaded_cell_params.loc[biophysics_id]
# cell_params[cell_params.index.str.contains('K_')] = 0

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'GlutamateUncaging', loaded_simulator, cell_params, loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

synapses_only_10x20Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
synapses_only_10x20Hz_df

In [ ]:
plot_traces(
    synapses_only_10x20Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['GlutamateUncaging.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
    ],
    x_lim=(385, 950), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411)  # , y_lim_soma_inset=(-76, -74)
)

In [ ]:
plot_syn_v_scatter(synapses_only_10x20Hz_df)

### Somatic current injection only

#### Find somatic current stimulation parameters

In [ ]:
from biophysics_fitting.ephys import find_crossing

amps = I.np.arange(0.3, 0.7, step=0.1)

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

breaker = False
for amp in amps:
    amp = round(amp, 1)

    cell_params = loaded_cell_params.loc[biophysics_id].copy()

    cell_params['SomaticCurrentInjection.stim.current_amp'] = amp
    cell_params['SomaticCurrentInjection.stim.current_inj_times'] = [400]
    cell_params['SomaticCurrentInjection.stim.current_duration'] = 300
    cell_params['SomaticCurrentInjection.run.tStop'] = 900

    # Cs internal
    # cell_params[cell_params.index.str.contains('K_')] = 0

    simulated_cell, param = loaded_simulator.get_simulated_cell(cell_params, 'SomaticCurrentInjection')

    # spikes = find_crossing(simulated_cell.soma.recVList[0], thresh=0)

    spike_amp = amp
    # time_to_spike = simulated_cell.tVec[spikes[0][0]] - cell_params['SomaticCurrentInjection.stim.current_inj_times'][0]
    print(spike_amp)

    I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)

    apical_sect = _get_apical_sec_and_i_at_distance(simulated_cell, loaded_fixed_params['BAC.stim.dist'])
    vm_dend = I.np.array(apical_sect[0].recVList)

    I.plt.plot(simulated_cell.tVec, vm_dend[apical_sect[2]], label=f'{amp} (dend)', linestyle='--')

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.xlim(left=390)
I.plt.ylabel('Membrane potential (mV)')
I.plt.legend()
I.plt.show()

#### Find dendritic current stimulation parameters

In [ ]:
def get_stim_synapses(presyn_cell_types, syn_ids):
    assert len(presyn_cell_types) == len(syn_ids) and len(presyn_cell_types) > 0

    cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

    # Map synapses
    syn_dist = read_synapse_realization(syn_file_path)
    synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
    synapse_mapper.map_synapse_realization()

    stim_synapses = []
    for i in range(len(presyn_cell_types)):
        for presyn_cell_type in cell.synapses.keys():
            for syn_id, syn in enumerate(cell.synapses[presyn_cell_type]):
                if presyn_cell_type == presyn_cell_types[i] and syn_id == syn_ids[i]:
                    synapse_strength_celltype = [x for x in loaded_syn_weights.keys() if x in presyn_cell_type]
                    assert(len(synapse_strength_celltype) == 1)
                    synapse_strength_celltype = synapse_strength_celltype[0]
                    weight = loaded_syn_weights[synapse_strength_celltype]
                    print(weight)

                    syn.weight = {'glutamate_syn': [weight, weight]}
                    syn.receptors = EXC_RECEPTOR_DICT

                    print(syn.receptors)

                    stim_synapses.append(syn)

    return stim_synapses

In [ ]:
presyn_cell_type = 'L34_A1'  # proximal
# presyn_cell_type = 'L2_A3'  # distal  L34_A3
syn_id = 10  # proximal
# syn_id = 0  # distal  2
stim_synapses = get_stim_synapses(presyn_cell_types=[presyn_cell_type], syn_ids=[syn_id])  # proximal

In [ ]:
print(stim_synapses[0].secID, stim_synapses[0].ptID, stim_synapses[0].preCellType)

In [ ]:
from biophysics_fitting.ephys import find_crossing

amps = [0.5]  # I.np.arange(0.5, 2.1, step=0.5)

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

breaker = False
for amp in amps:
    amp = round(amp, 2)
    print(amp)

    cell_params = loaded_cell_params.loc[biophysics_id].copy()

    # cell_params['LetzkusEtAl2006_BAC.stim.synapses'] = []
    cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_amp'] = 2  # spike_amp
    cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times'] = [600]
    cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_duration'] = 300
    cell_params['LetzkusEtAl2006_BAC.stim.dend_stim_times'] = [400]
    cell_params['LetzkusEtAl2006_BAC.stim.dend_stim_amp'] = amp
    cell_params['LetzkusEtAl2006_BAC.stim.dend_stim_duration'] = 700
    cell_params['LetzkusEtAl2006_BAC.stim.dend_stim_dist'] = loaded_fixed_params['BAC.stim.dist']
    cell_params['LetzkusEtAl2006_BAC.measure.recSite'] = loaded_fixed_params['BAC.stim.dist']
    cell_params['LetzkusEtAl2006_BAC.run.tStop'] = 1000
    
    # cell_params['LetzkusEtAl2006_BAC.stim.synapses'] = [stim_synapses[0]]
    # cell_params['LetzkusEtAl2006_BAC.stim.syn_stim_times'] = I.np.arange(600, 900, 10)
    # cell_params['LetzkusEtAl2006_BAC.run.vardt'] = False

    simulated_cell, param = loaded_simulator.get_simulated_cell(cell_params, 'LetzkusEtAl2006_BAC')

    spikes = find_crossing(simulated_cell.soma.recVList[0], thresh=0)

    apical_sect = _get_apical_sec_and_i_at_distance(simulated_cell, cell_params['LetzkusEtAl2006_BAC.stim.dend_stim_dist'])
    vm_dend = I.np.array(apical_sect[0].recVList)

    if len(spikes[0]) > len(cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times']):
        bac_spike_amp = amp

        print(bac_spike_amp)
        print('Multiple spikes')

        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        I.plt.plot(simulated_cell.tVec, vm_dend[apical_sect[2]], label=f'{amp} (dend)', linestyle='--')

        # Run 1 extra increment of 0.01 when reaching BAC
        # break

        if breaker:
            break
        breaker = True

    elif len(spikes[0]) == len(cell_params['LetzkusEtAl2006_BAC.stim.postsyn_stim_times']):
        subthresh_bac_spike_amp = amp
        # time_to_spike = simulated_cell.tVec[spikes[0][0]] - cell_params['SomaticCurrentInjection.stim.current_inj_times'][0]

        print(subthresh_bac_spike_amp)
        print(simulated_cell.tVec[spikes[0][0]])

        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        I.plt.plot(simulated_cell.tVec, vm_dend[apical_sect[2]], label=f'{amp} (dend)', linestyle='--')
    else:
        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], label=amp)
        I.plt.plot(simulated_cell.tVec, vm_dend[apical_sect[2]], label=f'{amp} (dend)', linestyle='--')

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.xlim(left=290)
I.plt.ylabel('Membrane potential (mV)')
I.plt.legend()
I.plt.show()

In [ ]:
loaded_fixed_params['BAC.stim.dist']

In [ ]:
cell_params[cell_params.index.str.contains('hot_zone')]

In [ ]:
from visualize.cell_morphology_visualizer import CellMorphologyVisualizer

client = I.get_client(ip='localhost', client_port=8786)

images_path = I.os.path.join(simulation_results_dir, 'new_plateau_potential_animation_3d')

# To remake the visualization, uncomment this code
if I.os.path.exists(images_path):
    I.shutil.rmtree(images_path)
else:
    I.os.mkdir(images_path)

cmv = CellMorphologyVisualizer(simulated_cell)
cmv.population_to_color_dict['inactive'] = "#f0f0f0"  # add color for inactive
# cmv.population_to_color_dict['Generic'] = "red"

cmv.animation(
    images_path=images_path, 
    color="voltage",
    client=client, 
    t_start=600-5, t_stop=900+10, t_step=0.2
)

#### Somatic current injection control

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'BittnerEtAl2017')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, 'control')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'SomaticCurrentInjection')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['SomaticCurrentInjection.stim.current_amp'] = 0.6
protocol_params['SomaticCurrentInjection.stim.current_inj_times'] = [400]
protocol_params['SomaticCurrentInjection.stim.current_duration'] = 300
protocol_params['SomaticCurrentInjection.run.tStop'] = 1000

cell_params = loaded_cell_params.loc[biophysics_id]
cell_params[cell_params.index.str.contains('K_')] = 0

In [ ]:
run_protocol_soma_only(simulation_results_dir, loaded_simulator, cell_params, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

soma_only_plateau_potential_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
soma_only_plateau_potential_df

In [ ]:
plot_traces(
    soma_only_plateau_potential_df,
    inputs=[
        {
            'stim_times': protocol_params['SomaticCurrentInjection.stim.current_inj_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['SomaticCurrentInjection.stim.current_amp'],
            'duration': protocol_params['SomaticCurrentInjection.stim.current_duration']
        }
    ],
    x_lim=(385, 800), y_lim_syn=(-80, None),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
# from visualize.cell_morphology_visualizer import CellMorphologyVisualizer

# images_path = I.os.path.join(results_dir, 'soma_spike_animation_3d')

# # To remake the visualization, uncomment this code
# if I.os.path.exists(images_path):
#     I.shutil.rmtree(images_path)

# cmv = CellMorphologyVisualizer(simulated_cell)
# cmv.population_to_color_dict['inactive'] = "#f0f0f0"  # add color for inactive
# # cmv.population_to_color_dict['Generic'] = "red"

# cmv.animation(
#     images_path=images_path, 
#     color="voltage",
#     client=client, 
#     t_start=400-10, t_stop=440, t_step=0.2
# )

In [ ]:
plot_syn_v_scatter(soma_only_plateau_potential_df)

### 0 ms delay

In [ ]:
paper_results_dir = I.os.path.join(results_dir, 'BittnerEtAl2017')
if not I.os.path.exists(paper_results_dir):
    I.os.mkdir(paper_results_dir)

protocol_results_dir = I.os.path.join(paper_results_dir, '20Hz')
if not I.os.path.exists(protocol_results_dir):
    I.os.mkdir(protocol_results_dir)

simulation_results_dir = I.os.path.join(protocol_results_dir, 'plus0')
if not I.os.path.exists(simulation_results_dir):
    I.os.mkdir(simulation_results_dir)

In [ ]:
protocol_params = {}
protocol_params['STDP.stim.syn_stim_times'] = I.np.arange(start=400, stop=900, step=50)  # 20 Hz
protocol_params['STDP.stim.postsyn_stim_times'] = [600]
protocol_params['STDP.stim.postsyn_stim_amp'] = 0.6
protocol_params['STDP.stim.postsyn_stim_duration'] = 300
protocol_params['STDP.run.tStop'] = 1200

cell_params = loaded_cell_params.loc[biophysics_id]
# cell_params[cell_params.index.str.contains('K_')] = 0

In [ ]:
run_protocol_all_synapses(simulation_results_dir, 'STDP', loaded_simulator, cell_params, loaded_syn_weights, protocol_params)

In [ ]:
cell, _ = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

btsp_plus0_20Hz_df = load_protocol_results(simulation_results_dir, cell, protocol_params, loaded_syn_weights)
btsp_plus0_20Hz_df

In [ ]:
plot_traces(
    btsp_plus0_20Hz_df,
    inputs=[
        {
            'stim_times': protocol_params['STDP.stim.syn_stim_times'],
            'label': 'Synaptic input',
            'amp': 1
        },
        {
            'stim_times': protocol_params['STDP.stim.postsyn_stim_times'],
            'label': 'Somatic current (nA)',
            'amp': protocol_params['STDP.stim.postsyn_stim_amp'],
            'duration': protocol_params['STDP.stim.postsyn_stim_duration']
        }
    ],
    x_lim=(385, 460), y_lim_syn=(-80, -5),
    x_lim_soma_inset=(400, 411), y_lim_soma_inset=(-76, -74)
)

In [ ]:
all_delays = [delay for delays in btsp_plus0_20Hz_df['actual_delay'].dropna() for delay in delays]

# Calculate the average of the time differences
mean_delay = I.np.mean(all_delays)
print("Average delay:", mean_delay)

df = btsp_plus0_20Hz_df.copy()

df = df.dropna(subset=['actual_delay'])
df['actual_delay'] = df['actual_delay'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)

I.plt.hist(df['actual_delay'], bins=20, edgecolor='black')

I.plt.xlabel('Delay between somatic and synaptic spike')
I.plt.ylabel('Num synapses')

%matplotlib inline
I.plt.show()

In [ ]:
plot_syn_v_scatter(btsp_plus0_20Hz_df)

In [ ]:
plot_syn_v_comparisons([btsp_plus0_20Hz_df, soma_only_plateau_potential_df, synapses_only_10x20Hz_df], labels=['0 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

### Analysis

In [ ]:
plot_syn_v_hist([synapses_only_10x20Hz_df, soma_only_plateau_potential_df, btsp_plus0_20Hz_df], labels=['Glutamate uncaging', 'Somatic current injection', '0 ms delay'])

In [ ]:
plot_syn_v_violin([synapses_only_10x20Hz_df, soma_only_plateau_potential_df, btsp_plus0_20Hz_df], labels=['Glutamate uncaging', 'Somatic current injection', '0 ms delay'])

#### Caya-Bissonnette (only L23-L5PT)

In [ ]:
filtered_soma_only_plateau_potential_df = soma_only_plateau_potential_df[soma_only_plateau_potential_df['presyn_cell_type'].apply(lambda x: 'L2' in x or 'L34' in x)]
filtered_synapses_only_10x20Hz_df = synapses_only_10x20Hz_df[synapses_only_10x20Hz_df['presyn_cell_type'].apply(lambda x: 'L2' in x or 'L34' in x)]
filtered_btsp_plus0_20Hz_df = btsp_plus0_20Hz_df[btsp_plus0_20Hz_df['presyn_cell_type'].apply(lambda x: 'L2' in x or 'L34' in x)]

filtered_btsp_plus0_20Hz_df

In [ ]:
plot_syn_v_hist([filtered_synapses_only_10x20Hz_df, filtered_soma_only_plateau_potential_df, filtered_btsp_plus0_20Hz_df], labels=['Synaptic stimulation', 'Somatic current injection', '0 ms delay'])

In [ ]:
plot_syn_v_comparisons([filtered_btsp_plus0_20Hz_df, filtered_soma_only_plateau_potential_df, filtered_synapses_only_10x20Hz_df], labels=['0 ms delay', 'Somatic current injection', 'Glutamate uncaging'])

In [ ]:
plot_syn_v_violin([filtered_synapses_only_10x20Hz_df, filtered_soma_only_plateau_potential_df, filtered_btsp_plus0_20Hz_df], labels=['Synaptic stimulation', 'Somatic current injection', '0 ms delay'])

## Stimulate all presynaptic cells individually

### Find presynaptic cells

In [ ]:
# dict_, _ = I.scp.reader.read_functional_realization_map(con_file_path)
# connected_celltypes = dict_.keys()

# for celltype in connected_celltypes:
#     n_cells = get_number_of_connected_cells(dict_[celltype])
#     print(celltype, n_cells)

In [ ]:
from numpy import array
from copy import deepcopy

PSTHs_1ms = {'VPM': [array([-200,   19,   20,   21,   22,   23,   24,   25,   26,   27,   28,
         29,   30,   31,   32,   33,   34,   35,   40,   45,   50,   55,
         60,   80,  100,  120,  140,  160,  180,  200,  220,  240,  260,
        280,  300,  320,  340,  360,  380,  400,  420,  440,  460,  480,
        500,  520,  540,  560,  580,  600,  620,  640,  660,  680,  700,
        720,  740,  760,  780,  800,  820,  840,  860,  880,  900]), array([0.00136986, 0.        , 0.01428571, 0.03809524, 0.05714286,
       0.07619048, 0.10952381, 0.11428571, 0.07142857, 0.05238095,
       0.03809524, 0.05238095, 0.03333333, 0.04761905, 0.02857143,
       0.01428571, 0.03809524, 0.01428571, 0.01904762, 0.0152381 ,
       0.00857143, 0.01428571, 0.01119048, 0.00785714, 0.00761905,
       0.0052381 , 0.0097619 , 0.00880952, 0.00833333, 0.0097619 ,
       0.01      , 0.00738095, 0.00714286, 0.00880952, 0.00928571,
       0.00833333, 0.00809524, 0.00785714, 0.00880952, 0.00952381,
       0.0102381 , 0.01047619, 0.01095238, 0.01119048, 0.0097619 ,
       0.01214286, 0.0097619 , 0.0102381 , 0.01309524, 0.01119048,
       0.01285714, 0.01095238, 0.00928571, 0.01357143, 0.00904762,
       0.005     , 0.00047619, 0.        , 0.        , 0.        ,
       0.        , 0.0002381 , 0.        , 0.00047619])], 'L6CC': [array([-200,   19,   20,   21,   22,   23,   24,   25,   26,   27,   28,
         29,   30,   31,   32,   33,   34,   35,   40,   45,   50,   55,
         60,   80,  100,  120,  140,  160,  180,  200,  220,  240,  260,
        280,  300,  320,  340,  360,  380,  400,  420,  440,  460,  480,
        500,  520,  540,  560,  580,  600,  620,  640,  660,  680,  700,
        720,  740,  760,  780,  800,  820,  840,  860,  880,  900]), array([5.36223091e-04, 0.00000000e+00, 3.57142857e-03, 0.00000000e+00,
       0.00000000e+00, 1.43803217e-01, 0.00000000e+00, 1.35484292e-01,
       3.61791668e-02, 9.28984032e-02, 5.42072128e-02, 1.23269301e-01,
       8.51669329e-02, 4.22619048e-02, 3.11120309e-02, 3.33333333e-02,
       2.35119048e-02, 6.90085870e-03, 3.04635762e-03, 9.52380952e-04,
       9.52380952e-04, 1.66666667e-03, 5.23809524e-03, 1.27878610e-03,
       2.24019321e-03, 2.65937508e-03, 3.91068681e-03, 1.52954413e-03,
       1.88375958e-03, 1.17240127e-03, 9.45550351e-04, 1.23428046e-03,
       1.47833724e-03, 1.81299145e-03, 1.87072349e-03, 2.05842781e-03,
       2.50448308e-03, 1.29047309e-03, 2.49361206e-03, 2.29473128e-03,
       1.03836596e-03, 1.47197192e-03, 1.93095653e-03, 8.08694379e-04,
       1.00204002e-03, 1.43092681e-03, 1.03907519e-03, 2.09404452e-03,
       3.98858314e-04, 1.40746237e-03, 1.41968249e-03, 1.41271468e-03,
       1.53401094e-03, 1.45281446e-03, 1.74803924e-03, 1.37965206e-03,
       2.91471507e-03, 7.21520095e-04, 5.41972866e-04, 4.04446547e-04,
       3.63401437e-04, 2.95667447e-04, 1.05851699e-04, 3.98591746e-04])], 'L4ss': [array([-200,   19,   20,   21,   22,   23,   24,   25,   26,   27,   28,
         29,   30,   31,   32,   33,   34,   35,   40,   45,   50,   55,
         60,   80,  100,  120,  140,  160,  180,  200,  220,  240,  260,
        280,  300,  320,  340,  360,  380,  400,  420,  440,  460,  480,
        500,  520,  540,  560,  580,  600,  620,  640,  660,  680,  700,
        720,  740,  760,  780,  800,  820,  840,  860,  880,  900]), array([0.00115747, 0.        , 0.        , 0.00333333, 0.        ,
       0.        , 0.        , 0.        , 0.01591204, 0.02913043,
       0.02304348, 0.02289855, 0.03434783, 0.01144928, 0.00333333,
       0.        , 0.        , 0.00333333, 0.004     , 0.00066667,
       0.00266667, 0.00266667, 0.00371366, 0.00253516, 0.00300575,
       0.00341548, 0.00175287, 0.00346428, 0.00203516, 0.00358621,
       0.00266667, 0.00150575, 0.00235666, 0.00169608, 0.00151758,
       0.00151758, 0.00211562, 0.00191954, 0.00243137, 0.00352941,
       0.00266667, 0.00325287, 0.00208621, 0.0025    , 0.00216667,
       0.00250575, 0.00310379, 0.00216667, 0.00142529, 0.00283717,
       0.003     , 0.00211562, 0.00318808, 0.002     , 0.002     ,
       0.002     , 0.00176471, 0.00133333, 0.00083333, 0.00026471,
       0.00076471, 0.00057246, 0.0004058 , 0.00023913])], 'L5TT': [array([-200,   19,   20,   21,   22,   23,   24,   25,   26,   27,   28,
         29,   30,   31,   32,   33,   34,   35,   40,   45,   50,   55,
         60,   80,  100,  120,  140,  160,  180,  200,  220,  240,  260,
        280,  300,  320,  340,  360,  380,  400,  420,  440,  460,  480,
        500,  520,  540,  560,  580,  600,  620,  640,  660,  680,  700,
        720,  740,  760,  780,  800,  820,  840,  860,  880,  900]), array([0.00216226, 0.00108108, 0.00327485, 0.00354305, 0.0003252 ,
       0.00164446, 0.00167449, 0.00472394, 0.01466667, 0.03728979,
       0.06148892, 0.05762219, 0.04373781, 0.05666401, 0.03082494,
       0.01853883, 0.00464242, 0.00581315, 0.00272998, 0.00654799,
       0.00861074, 0.00352475, 0.00399876, 0.00657295, 0.01123286,
       0.00904555, 0.00562618, 0.0053193 , 0.00603566, 0.00728842,
       0.00525555, 0.00680474, 0.00539578, 0.00630508, 0.00682545,
       0.00645775, 0.00637406, 0.00508756, 0.0066391 , 0.00725979,
       0.00506127, 0.00617644, 0.00632364, 0.00627671, 0.00558887,
       0.00534255, 0.006778  , 0.00538644, 0.00531972, 0.00667693,
       0.00535597, 0.00602466, 0.00576317, 0.00641854, 0.00540016,
       0.00646505, 0.00459043, 0.00266334, 0.00202163, 0.00160612,
       0.00109666, 0.00231178, 0.00155425, 0.00130906])], 'INT': [array([-200,   19,   20,   21,   22,   23,   24,   25,   26,   27,   28,
         29,   30,   31,   32,   33,   34,   35,   40,   45,   50,   55,
         60,   80,  100,  120,  140,  160,  180,  200,  220,  240,  260,
        280,  300,  320,  340,  360,  380,  400,  420,  440,  460,  480,
        500,  520,  540,  560,  580,  600,  620,  640,  660,  680,  700,
        720,  740,  760,  780,  800,  820,  840,  860,  880,  900]), array([0.00251095, 0.00449696, 0.00409495, 0.00151515, 0.00419488,
       0.04128788, 0.01878341, 0.09288905, 0.20087337, 0.15959459,
       0.1272792 , 0.09521312, 0.09720255, 0.07850266, 0.04762997,
       0.02874596, 0.02528052, 0.00732227, 0.00388839, 0.00143882,
       0.00271377, 0.00332994, 0.00354427, 0.00924889, 0.0100747 ,
       0.00985577, 0.00978178, 0.00871505, 0.00771226, 0.00659895,
       0.00811807, 0.0066922 , 0.00661976, 0.00666228, 0.0067164 ,
       0.00694274, 0.00716889, 0.00693228, 0.00619805, 0.0048379 ,
       0.00547238, 0.0062619 , 0.00647745, 0.00641854, 0.00674207,
       0.00642165, 0.00673333, 0.00666788, 0.00586395, 0.00652652,
       0.00684776, 0.00643377, 0.00596619, 0.00576195, 0.00620149,
       0.00718102, 0.00332066, 0.00175557, 0.00137219, 0.00065039,
       0.00054234, 0.00139958, 0.00094145, 0.0012355 ])], 'L23': [array([-200,   19,   20,   21,   22,   23,   24,   25,   26,   27,   28,
         29,   30,   31,   32,   33,   34,   35,   40,   45,   50,   55,
         60,   80,  100,  120,  140,  160,  180,  200,  220,  240,  260,
        280,  300,  320,  340,  360,  380,  400,  420,  440,  460,  480,
        500,  520,  540,  560,  580,  600,  620,  640,  660,  680,  700,
        720,  740,  760,  780,  800,  820,  840,  860,  880,  900]), array([0.00053272, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.00416667, 0.00833333, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.00416667, 0.        , 0.00083333, 0.        , 0.        ,
       0.        , 0.        , 0.00083333, 0.00125   , 0.00104167,
       0.00104167, 0.000625  , 0.00166667, 0.00125   , 0.00125   ,
       0.00208333, 0.00020833, 0.00125   , 0.00104167, 0.00104167,
       0.00125   , 0.000625  , 0.00020833, 0.00125   , 0.000625  ,
       0.00041667, 0.00083333, 0.00020833, 0.00041667, 0.000625  ,
       0.000625  , 0.00083333, 0.00083333, 0.        , 0.00104167,
       0.00083333, 0.00041667, 0.00020833, 0.00083333, 0.00166667,
       0.00083333, 0.00125   , 0.000625  , 0.00104167, 0.00020833,
       0.        , 0.        , 0.        , 0.        ])], 'L5ST': [array([-200,   19,   20,   21,   22,   23,   24,   25,   26,   27,   28,
         29,   30,   31,   32,   33,   34,   35,   40,   45,   50,   55,
         60,   80,  100,  120,  140,  160,  180,  200,  220,  240,  260,
        280,  300,  320,  340,  360,  380,  400,  420,  440,  460,  480,
        500,  520,  540,  560,  580,  600,  620,  640,  660,  680,  700,
        720,  740,  760,  780,  800,  820,  840,  860,  880,  900]), array([0.00130593, 0.        , 0.        , 0.0037037 , 0.        ,
       0.00389157, 0.00851852, 0.        , 0.        , 0.01111111,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.0037037 , 0.        , 0.00061728, 0.00061728,
       0.00016103, 0.00185185, 0.00042002, 0.00085225, 0.00189882,
       0.00218062, 0.0022759 , 0.00325966, 0.00220397, 0.00205582,
       0.00222564, 0.0015679 , 0.00174114, 0.00256173, 0.0019323 ,
       0.00095679, 0.00082736, 0.001843  , 0.00131508, 0.0022814 ,
       0.0021672 , 0.001917  , 0.00192297, 0.00188003, 0.00075013,
       0.00208991, 0.00277851, 0.00232146, 0.00203106, 0.00173779,
       0.0009696 , 0.00079093, 0.00163312, 0.0010781 , 0.0021766 ,
       0.00233682, 0.00176865, 0.00194786, 0.00207816, 0.00092593,
       0.00096618, 0.0007037 , 0.00067901, 0.0008642 ])]}

for celltype in PSTHs_1ms:
    a,b = PSTHs_1ms[celltype]
    PSTHs_1ms[celltype] = (a,b*1000)
a,b = deepcopy(PSTHs_1ms['VPM'])
PSTHs_1ms['inactive'] = (a,I.np.zeros_like(b))

In [ ]:
def scale_PSTH(PSTH, ongoing = 1.0, onset = 1.0, sustained = 1.0, celltype = 'INT', verbose=False):
    out = {}
    PSTH_out = deepcopy(PSTH)
    bins, values = PSTH[celltype]
    values = list(values)
    for lv in range(len(values)):
        if lv == 0:
            values[0] = values[0] * ongoing
            if verbose: print('applying ongoing scaling to bin', bins[lv], bins[lv+1])
        elif bins[lv+1] <= 60:
            values[lv] = values[lv]*onset
            if verbose: print('applying onset scaling to bin', bins[lv], bins[lv+1])            
        elif bins[lv+1] <= 700:
            values[lv] = values[lv]*sustained  
            if verbose: print('applying sustained scaling to bin', bins[lv], bins[lv+1])                        
        else:
            values[lv] = values[lv]*ongoing
            if verbose: print('applying onset scaling to bin', bins[lv], bins[lv+1])     
    PSTH_out[celltype] = (bins, values)
    return PSTH_out

def timshift_PSTH(PSTH, shift = 0., celltype = 'INT'):
    out = {}
    PSTH_out = deepcopy(PSTH)
    bins, values = PSTH[celltype]
    PSTH_out[celltype] = (list(I.np.array(bins) + shift), values)
    return PSTH_out

In [ ]:
template_info = {'author': 'abast',
 'date': '23Sep2021',
 'name': 'asd'}

template_EXC = {'cellNr': None,
 'celltype': {'pointcell': {'distribution': 'PSTH_poissontrain_v2',
   'intervals': None,
   'offset': 0.0,
   'rates': None}},
 'synapses': {'connectionFile': None,
  'distributionFile': None,
  'receptors': {'glutamate_syn': {'delay': 0.0,
    'parameter': {'decayampa': 1.0,
     'decaynmda': 1.0,
     'facilampa': 0.0,
     'facilnmda': 0.0,
     'tau1': 26.0,
     'tau2': 2.0,
     'tau3': 2.0,
     'tau4': 0.1},
    'threshold': 0.0,
    'weight': [None, None]}},
  'releaseProb': 0.6}}

template_INH= {'cellNr': None,
 'celltype': {'pointcell': {'distribution': 'PSTH_poissontrain_v2',
   'intervals': None,
   'offset': 0.0,
   'rates': None}},
 'synapses': {'connectionFile': None,
  'distributionFile': None,
  'receptors': {'gaba_syn': {'delay': 0.0,
    'parameter': {'decaygaba': 1.0,
     'decaytime': 20.0,
     'e': -80.0,
     'facilgaba': 0.0,
     'risetime': 1.0},
    'threshold': 0.0,
    'weight': 1.0}},
  'releaseProb': 0.25}}

template_NMODL_mechanisms = {'VecStim': '/', 'synapses': '/'}

def get_number_of_connected_cells(list_):
    cellid = [x[1] for x in list_]
    return max(cellid) + 1 # + 1 because counting starts at 0, so total number is + 1

def match_model_celltype_to_PSTH_celltype(celltype, verbose=False):
    if '_' in celltype:
        celltype = celltype.split('_')[0]
    if celltype in INHIBITORY or celltype == 'INH':
        key = 'INT'
    elif celltype in ('L4ss', 'L4py', 'L4sp'):
        key = 'L4ss'
    elif celltype == 'L5st':
        key = 'L5ST'
    elif celltype == 'L5tt':
        key = 'L5TT'
    elif celltype == 'L6cc':
        key = 'L6CC'
    elif celltype == 'VPM':
        key = 'VPM'
    elif celltype in ('L2','L34'):
        key = 'L23'
    elif celltype in ('L6ct', 'L6ccinv'):
        key = 'inactive'
    else:
        raise ValueError(celltype)   
    if verbose:
        print('matching', celltype, 'to', key, 'PSTH')
    return key

def create_network_param_file(syn, con, PSTH, syn_strength, evoked_columns='all', offset=245, verbose=False):
    dict_, _ = I.scp.reader.read_functional_realization_map(con)
    connected_celltypes = dict_.keys()
    out = {}
    out['info'] = deepcopy(template_info)
    out['NMODL_mechanisms'] = deepcopy(template_NMODL_mechanisms)
    out['network'] = {}    
    for celltype in connected_celltypes:
        # set up template for celltype
        if (celltype in EXCITATORY) or (celltype.split('_')[0] in EXCITATORY):
            if verbose:
                print('assigning celltype', celltype, 'to excitatory template.')
            out['network'][celltype] = deepcopy(template_EXC)
            # awkward way of selecting the syn strength matching the current celltype
            synapse_strength_celltype = [x for x in syn_strength.keys() if x in celltype]
            assert(len(synapse_strength_celltype) == 1)
            synapse_strength_celltype = synapse_strength_celltype[0]
            if verbose:
                print('setting synapse strength of celltype', celltype, 'to synapse strength of',  synapse_strength_celltype)
            weight = syn_strength[synapse_strength_celltype]
            out['network'][celltype]['synapses']['receptors']['glutamate_syn']['weight'] = [weight, weight]
        elif (celltype in INHIBITORY) or (celltype.split('_')[0] in INHIBITORY):
            if verbose:
                print('assigning celltype', celltype, 'to inhibitory template.')
            out['network'][celltype] = deepcopy(template_INH)
        # fill template
        key = match_model_celltype_to_PSTH_celltype(celltype)
        bins, rates = PSTH[key]          
        bins = [b + offset for b in bins]
        bins[0] = 0
        if not celltype in ('INH', 'INH_S1'):
            if not evoked_columns == 'all':
                if not celltype.split('_')[1] in evoked_columns:
                    rates = I.np.ones_like(rates)*rates[0]
        out['network'][celltype]['cellNr'] = get_number_of_connected_cells(dict_[celltype])
        out['network'][celltype]['celltype']['pointcell']['bins'] = list(bins)
        out['network'][celltype]['celltype']['pointcell']['rates'] = list(rates)
        out['network'][celltype]['synapses']['connectionFile'] = con
        out['network'][celltype]['synapses']['distributionFile'] = syn
    # I.scp.network_param_modify_functions.change_glutamate_syn_weights(out, I.pd.Series(syn_strength))      
    return out

In [ ]:
PSTH = PSTHs_1ms
scale_ongoing = 2.25
scale_onset = 0.4
scale_sustained = 2
INH_timeshift = -2

offset = 200 # 445
tStop = offset + 200
output_name = 'test5' # 1.9,0.8,0.8, evoked_columns: 'C2'
evoked_columns = ['C2']# 'all'

currentPSTH = scale_PSTH(PSTH, scale_ongoing, scale_onset, scale_sustained)
currentPSTH = timshift_PSTH(currentPSTH, INH_timeshift)

netp = create_network_param_file(syn_file_path, con_file_path, currentPSTH, loaded_syn_weights, evoked_columns, offset=offset)
# netp

In [ ]:
cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

nwMap = I.scp.NetworkMapper(
    cell, 
    NTParameterSet(netp).network,
    {
        'T': 34.0,
        'Vinit': -75.0,
        'dt': 0.025,
        'recordingSites': [],
        'tStart': 0.0,
        'tStop': 400.0
    }
)
nwMap.create_saved_network2()
# nwMap.connected_cells

In [ ]:
presyn_cells = {}

for connected_cell_type in list(nwMap.connected_cells.keys()):
    presyn_cells[connected_cell_type] = []
    for presyn_cell in nwMap.cells[connected_cell_type]:
        presyn_cells[connected_cell_type].append(presyn_cell.synapseList if presyn_cell.synapseList != None else [])

# presyn_cells

In [ ]:
# Re-init cell and NetworkMapper to prevent errors (and NetworkMapper is not used anymore)
try:
    cell.evokedNW.re_init_network()
    print('found evokedNW attached to cell')
    print('explicitly resetting it.')
except AttributeError:
    pass

for cellType in list(nwMap.cells.keys()):
    for syn in cell.synapses[cellType]:
        syn.disconnect_hoc_synapse()

cell.re_init_cell()
nwMap.re_init_network(replayMode=True)

### Run

In [ ]:
from tqdm import tqdm

from single_cell_parser.reader import read_synapse_realization
from single_cell_parser.synapse_mapper import SynapseMapper
from single_cell_parser.analyze.synanalysis import compute_syn_distance

presyn_cell_data_file_path = I.os.path.join(current_dir, f'{morphology_id}_{biophysics_id}_presynaptic_cell_activation_data.csv')

if not I.os.path.exists(presyn_cell_data_file_path):
    presyn_cell_df = I.pd.DataFrame(columns=['presyn_cell_label', 'presyn_cell_id', 'num_syns', 'soma_dists', 'syn_weight', 'EPSP_amp'])  #, 'EPSP_t', 'EPSP_v'])

    cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

    # # Map synapses
    # syn_dist = read_synapse_realization(syn_file_path)
    # synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
    # synapse_mapper.map_synapse_realization()

    for presyn_cell_type in tqdm(list(presyn_cells.keys())):
        n_cells = len(presyn_cells[presyn_cell_type])

        for preSynCellID in range(n_cells):
            synapse_list = presyn_cells[presyn_cell_type][preSynCellID]
            assert synapse_list != None

            synapses = []
            for syn_id, syn in enumerate(synapse_list):
                if (presyn_cell_type in EXCITATORY) or (presyn_cell_type.split('_')[0] in EXCITATORY):
                    synapse_strength_celltype = [x for x in loaded_syn_weights.keys() if x in presyn_cell_type]
                    assert(len(synapse_strength_celltype) == 1)
                    synapse_strength_celltype = synapse_strength_celltype[0]
                    weight = loaded_syn_weights[synapse_strength_celltype]

                    syn.weight = {'glutamate_syn': [weight, weight]}
                    syn.receptors= EXC_RECEPTOR_DICT

                    synapses.append(syn)
                else:
                    continue

            if len(synapses) > 0:
                cell, cell_params = loaded_simulator.setup.get(cell_params)

                cell_params['GlutamateUncaging.stim.syn_stim_times'] = [400] #I.np.arange(start=300, stop=600, step=200)  # 10 Hz
                cell_params['GlutamateUncaging.stim.synapses'] = synapses

                simulated_cell, params = loaded_simulator.get_simulated_cell(cell_params, 'GlutamateUncaging')

                syn_data = {
                    'presyn_cell_label': presyn_cell_type,
                    'presyn_cell_id': preSynCellID,
                    'num_syns': len(synapses),
                    'soma_dists': [compute_syn_distance(simulated_cell, syn) for syn in synapses],
                    'syn_weight': weight,
                    'EPSP_amp': calc_epsp_amp(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], t_start=cell_params['GlutamateUncaging.stim.syn_stim_times'][0] - 10),
                    # 'EPSP_t': cell.tVec,
                    # 'EPSP_v': cell.sections[0].recVList[0]
                }
                presyn_cell_df = presyn_cell_df.append(syn_data, ignore_index=True)

                I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0])

    # Assign cell type based on presyn_cell_type
    presyn_cell_df['presyn_cell_type'] = presyn_cell_df['presyn_cell_label'].apply(lambda x: next((cat for cat in loaded_syn_weights.keys() if cat in x), 'Unknown'))

    # Calculate mean distance between synapses and soma per presynaptic cell
    presyn_cell_df['mean_soma_dist'] = presyn_cell_df['soma_dists'].apply(I.np.mean)

    presyn_cell_df.to_csv(presyn_cell_data_file_path, index=False)

    %matplotlib inline
    I.plt.xlabel('Time (ms)')
    I.plt.ylabel('Membrane potential (mV)')
    I.plt.show()

else:
    print(f'Presynaptic-cell activation data was already computed. Loading: {presyn_cell_data_file_path}')
    presyn_cell_df = I.pd.read_csv(presyn_cell_data_file_path)

In [ ]:
presyn_cell_df

### Analyze

#### All

In [ ]:
I.plt.hist(presyn_cell_df['EPSP_amp'], bins=10, color='green', edgecolor='black')
I.plt.xlabel('EPSP amplitude (mV)')
I.plt.ylabel('Num presynaptic cells')
I.plt.show()

In [ ]:
I.plt.hist(presyn_cell_df['mean_soma_dist'], bins=10, edgecolor='black')
I.plt.xlabel('Mean soma distance (µm)')
I.plt.ylabel('Num presynaptic cells')
I.plt.show()

In [ ]:
syn_weight_counts = presyn_cell_df['syn_weight'].value_counts()
syn_weight_counts = syn_weight_counts.sort_index(ascending=True)

syn_weight_counts.plot(kind='bar', edgecolor='black')

x_labels = syn_weight_counts.index
rounded_labels = [f'{label:.3f}' for label in x_labels]

I.plt.xticks(ticks=range(len(rounded_labels)), labels=rounded_labels, rotation=0)
I.plt.xlabel('Synaptic weight')
I.plt.ylabel('Num presynaptic cells')
I.plt.show()

In [ ]:
num_syns_counts = presyn_cell_df['num_syns'].value_counts()
num_syns_counts = num_syns_counts.sort_index(ascending=True)

num_syns_counts.plot(kind='bar', edgecolor='black')
I.plt.xticks(rotation=0)  # Rotating to 0 degrees for better readability
I.plt.xlabel('Num synapses')
I.plt.ylabel('Num presynaptic cells')
I.plt.show()

In [ ]:
# I.plt.scatter(presyn_cell_df['mean_soma_dist'], presyn_cell_df['EPSP_amp'], edgecolor='black')
# I.plt.xlabel('Mean soma Distance (µm)')
# I.plt.ylabel('EPSP Amplitude (mV)')
# I.plt.show()

In [ ]:
scatter = I.plt.scatter(presyn_cell_df['mean_soma_dist'], presyn_cell_df['EPSP_amp'], c=presyn_cell_df['syn_weight'], cmap='viridis', edgecolor='black')
cbar = I.plt.colorbar(scatter)
cbar.set_label('Synaptic weight')
I.plt.xlabel('Mean soma distance (µm)')
I.plt.ylabel('EPSP amplitude (mV)')
I.plt.show()

In [ ]:
scatter = I.plt.scatter(presyn_cell_df['mean_soma_dist'], presyn_cell_df['EPSP_amp'], c=presyn_cell_df['num_syns'], cmap='viridis', edgecolor='black')
cbar = I.plt.colorbar(scatter)
cbar.set_label('Num synapses')
I.plt.xlabel('Mean soma distance (µm)')
I.plt.ylabel('EPSP amplitude (mV)')
I.plt.show()

In [ ]:
scatter = I.plt.scatter(presyn_cell_df['mean_soma_dist'], presyn_cell_df['EPSP_amp'], c=presyn_cell_df['num_syns']*presyn_cell_df['syn_weight'], cmap='viridis', edgecolor='black')
cbar = I.plt.colorbar(scatter)
cbar.set_label('Total synaptic impact')
I.plt.xlabel('Mean soma distance (µm)')
I.plt.ylabel('EPSP amplitude (mV)')
I.plt.show()

In [ ]:
cell_types = presyn_cell_df['presyn_cell_type'].unique()

colors = I.plt.cm.get_cmap('Set1', len(cell_types))

# Create a dictionary to map each cell type to a color
color_mapping = {cell_type: i for i, cell_type in enumerate(cell_types)}

# Map presyn_cell_type to color indices
color_indices = presyn_cell_df['presyn_cell_type'].map(color_mapping)

scatter = I.plt.scatter(presyn_cell_df['mean_soma_dist'], presyn_cell_df['EPSP_amp'], c=color_indices, cmap=colors, edgecolor='black')

# Create a custom legend for the categories
handles = [I.plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors(i), markersize=10) 
           for i in range(len(cell_types))]
I.plt.legend(handles, cell_types, title='Presynaptic cell type')

I.plt.xlabel('Mean soma distance (µm)')
I.plt.ylabel('EPSP amplitude (mV)')
I.plt.show()

#### L5tts

In [ ]:
presyn_cell_data_file_path = I.os.path.join(current_dir, f'{morphology_id}_{biophysics_id}_presynaptic_cell_activation_data.csv')

if I.os.path.exists(presyn_cell_data_file_path):
    presyn_cell_df = I.pd.read_csv(presyn_cell_data_file_path)
else:
    raise FileNotFoundError(f'Could not find file {presyn_cell_data_file_path}')

presyn_cell_df

In [ ]:
presyn_cell_df_L5tts = presyn_cell_df[presyn_cell_df['presyn_cell_type'] == 'L5tt']
presyn_cell_df_L5tts

In [ ]:
I.plt.hist(presyn_cell_df_L5tts['EPSP_amp'], bins=10, color='green', edgecolor='black')
I.plt.xlabel('EPSP amplitude (mV)')
I.plt.ylabel('Num presynaptic cells')
I.plt.show()

In [ ]:
I.plt.hist(presyn_cell_df_L5tts['mean_soma_dist'], bins=10, edgecolor='black')
I.plt.xlabel('Mean soma distance (µm)')
I.plt.ylabel('Num presynaptic cells')
I.plt.show()

In [ ]:
num_syns_counts = presyn_cell_df_L5tts['num_syns'].value_counts()
num_syns_counts = num_syns_counts.sort_index(ascending=True)

num_syns_counts.plot(kind='bar', edgecolor='black')
I.plt.xticks(rotation=0)  # Rotating to 0 degrees for better readability
I.plt.xlabel('Num synapses')
I.plt.ylabel('Num presynaptic cells')
I.plt.show()

In [ ]:
cell_types = presyn_cell_df_L5tts['presyn_cell_type'].unique()

colors = I.plt.cm.get_cmap('Set1', len(cell_types))

# Create a dictionary to map each cell type to a color
color_mapping = {cell_type: i for i, cell_type in enumerate(cell_types)}

# Map presyn_cell_type to color indices
color_indices = presyn_cell_df_L5tts['presyn_cell_type'].map(color_mapping)


scatter = I.plt.scatter(presyn_cell_df_L5tts['mean_soma_dist'], presyn_cell_df_L5tts['EPSP_amp'], c=color_indices, cmap=colors, edgecolor='black')

# Create a custom legend for the categories
handles = [I.plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors(i), markersize=10) 
           for i in range(len(cell_types))]
I.plt.legend(handles, cell_types, title='Presynaptic cell type')

I.plt.xlabel('Mean soma distance (µm)')
I.plt.ylabel('EPSP amplitude (mV)')
I.plt.show()

In [ ]:
scatter = I.plt.scatter(presyn_cell_df_L5tts['mean_soma_dist'], presyn_cell_df_L5tts['EPSP_amp'], c=presyn_cell_df_L5tts['num_syns'], cmap='viridis', edgecolor='black')
cbar = I.plt.colorbar(scatter)
cbar.set_label('Num synapses')
I.plt.xlabel('Mean soma distance (µm)')
I.plt.ylabel('EPSP amplitude (mV)')
I.plt.show()

## Specific presynaptic L5TT cell

In [ ]:
presyn_cell_data_file_path = I.os.path.join(current_dir, f'{morphology_id}_{biophysics_id}_presynaptic_cell_activation_data.csv')

if I.os.path.exists(presyn_cell_data_file_path):
    presyn_cell_df = I.pd.read_csv(presyn_cell_data_file_path)
else:
    raise FileNotFoundError(f'Could not find file {presyn_cell_data_file_path}')

presyn_cell_df

In [ ]:
presyn_cell_df_L5tts = presyn_cell_df[presyn_cell_df['presyn_cell_type'] == 'L5tt']
presyn_cell_df_L5tts

In [ ]:
L5TT_cell_data = presyn_cell_df_L5tts[presyn_cell_df_L5tts['num_syns'] == 5]
# L5TT_cell_data = presyn_cell_df_L5tts.iloc[0]
L5TT_cell_data

### EPSP

In [ ]:
cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

# # Map synapses
# syn_dist = read_synapse_realization(syn_file_path)
# synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
# synapse_mapper.map_synapse_realization()

presyn_cell_type = L5TT_cell_data['presyn_cell_label'].values[0]
preSynCellID = L5TT_cell_data['presyn_cell_id'].values[0]

synapse_list = presyn_cells[presyn_cell_type][preSynCellID]
print(len(synapse_list))

synapses = []
for syn_id, syn in enumerate(synapse_list):
    if (presyn_cell_type in EXCITATORY) or (presyn_cell_type.split('_')[0] in EXCITATORY):
        synapse_strength_celltype = [x for x in loaded_syn_weights.keys() if x in presyn_cell_type]
        assert(len(synapse_strength_celltype) == 1)
        synapse_strength_celltype = synapse_strength_celltype[0]
        weight = loaded_syn_weights[synapse_strength_celltype]

        syn.weight = {'glutamate_syn': [weight, weight]}
        syn.receptors= EXC_RECEPTOR_DICT

        synapses.append(syn)
    else:
        continue

if len(synapses) > 0:
    cell, cell_params = loaded_simulator.setup.get(cell_params)

    cell_params['GlutamateUncaging.stim.synapses'] = synapses
    cell_params['GlutamateUncaging.stim.syn_stim_times'] = [400] #I.np.arange(start=300, stop=600, step=200)  # 10 Hz

    simulated_cell, params = loaded_simulator.get_simulated_cell(cell_params, 'GlutamateUncaging')

    syn_data = {
        'presyn_cell_label': presyn_cell_type,
        'presyn_cell_id': preSynCellID,
        'num_syns': len(synapses),
        'soma_dists': [compute_syn_distance(simulated_cell, syn) for syn in synapses],
        'syn_weight': weight,
        'EPSP_amp': calc_epsp_amp(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], t_start=cell_params['GlutamateUncaging.stim.syn_stim_times'][0] - 10),
        # 'EPSP_t': cell.tVec,
        # 'EPSP_v': cell.sections[0].recVList[0]
    }

    # Calculate mean distance between synapses and soma
    syn_data['mean_soma_dist'] = I.np.mean(syn_data['soma_dists'])

    I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0])

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.ylabel('Membrane potential (mV)')
I.plt.show()

syn_data

### Paired pulse stimulation

In [ ]:
cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

# # Map synapses
# syn_dist = read_synapse_realization(syn_file_path)
# synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
# synapse_mapper.map_synapse_realization()

presyn_cell_type = L5TT_cell_data['presyn_cell_label'].values[0]
preSynCellID = L5TT_cell_data['presyn_cell_id'].values[0]

synapse_list = presyn_cells[presyn_cell_type][preSynCellID]
print(len(synapse_list))

synapses = []
for syn_id, syn in enumerate(synapse_list):
    if (presyn_cell_type in EXCITATORY) or (presyn_cell_type.split('_')[0] in EXCITATORY):
        synapse_strength_celltype = [x for x in loaded_syn_weights.keys() if x in presyn_cell_type]
        assert(len(synapse_strength_celltype) == 1)
        synapse_strength_celltype = synapse_strength_celltype[0]
        weight = loaded_syn_weights[synapse_strength_celltype]

        syn.weight = {'glutamate_syn': [weight, weight]}
        syn.receptors= EXC_RECEPTOR_DICT

        synapses.append(syn)
    else:
        continue

if len(synapses) > 0:
    cell, cell_params = loaded_simulator.setup.get(cell_params)

    cell_params['STDP.stim.presyn_spike_times'] = [400] #I.np.arange(start=300, stop=600, step=200)  # 10 Hz
    cell_params['STDP.stim.postsyn_stim_amp'] = spike_amp  # AP_amp
    cell_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
    cell_params['STDP.stim.pairing_delay'] = 10
    cell_params['STDP.stim.synapses'] = synapses
    cell_params['STDP.run.tStop'] = 600

    simulated_cell, params = loaded_simulator.get_simulated_cell(cell_params, 'STDP')

    syn_data = {
        'presyn_cell_label': presyn_cell_type,
        'presyn_cell_id': preSynCellID,
        'num_syns': len(synapses),
        'soma_dists': [compute_syn_distance(simulated_cell, syn) for syn in synapses],
        'syn_weight': weight,
        'EPSP_amp': calc_epsp_amp(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], t_start=cell_params['STDP.stim.presyn_spike_times'][0] - 10),
        # 'EPSP_t': cell.tVec,
        # 'EPSP_v': cell.sections[0].recVList[0]
    }

    # Calculate mean distance between synapses and soma
    syn_data['mean_soma_dist'] = I.np.mean(syn_data['soma_dists'])

    I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0])
    # for syn in synapses:
    #     syn_seg_id = int(syn.x * simulated_cell.sections[syn.secID].nseg)
    #     I.plt.plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recVList[syn_seg_id])

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.ylabel('Membrane potential (mV)')
I.plt.show()

syn_data

In [ ]:
cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

# # Map synapses
# syn_dist = read_synapse_realization(syn_file_path)
# synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
# synapse_mapper.map_synapse_realization()

presyn_cell_type = L5TT_cell_data['presyn_cell_label'].values[0]
preSynCellID = L5TT_cell_data['presyn_cell_id'].values[0]

synapse_list = presyn_cells[presyn_cell_type][preSynCellID]
print(len(synapse_list))

synapses = []
for syn_id, syn in enumerate(synapse_list):
    if (presyn_cell_type in EXCITATORY) or (presyn_cell_type.split('_')[0] in EXCITATORY):
        synapse_strength_celltype = [x for x in loaded_syn_weights.keys() if x in presyn_cell_type]
        assert(len(synapse_strength_celltype) == 1)
        synapse_strength_celltype = synapse_strength_celltype[0]
        weight = loaded_syn_weights[synapse_strength_celltype]

        syn.weight = {'glutamate_syn': [weight, weight]}
        syn.receptors= EXC_RECEPTOR_DICT

        synapses.append(syn)
    else:
        continue

if len(synapses) > 0:
    cell, cell_params = loaded_simulator.setup.get(cell_params)

    cell_params['STDP.stim.presyn_spike_times'] = [400] #I.np.arange(start=300, stop=600, step=200)  # 10 Hz
    cell_params['STDP.stim.postsyn_stim_amp'] = spike_amp  # AP_amp
    cell_params['STDP.stim.postsyn_time_to_spike'] = time_to_spike
    cell_params['STDP.stim.pairing_delay'] = 100
    cell_params['STDP.stim.synapses'] = synapses
    cell_params['STDP.run.tStop'] = 600

    simulated_cell, params = loaded_simulator.get_simulated_cell(cell_params, 'STDP')

    syn_data = {
        'presyn_cell_label': presyn_cell_type,
        'presyn_cell_id': preSynCellID,
        'num_syns': len(synapses),
        'soma_dists': [compute_syn_distance(simulated_cell, syn) for syn in synapses],
        'syn_weight': weight,
        'EPSP_amp': calc_epsp_amp(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], t_start=cell_params['STDP.stim.presyn_spike_times'][0] - 10),
        # 'EPSP_t': cell.tVec,
        # 'EPSP_v': cell.sections[0].recVList[0]
    }

    # Calculate mean distance between synapses and soma
    syn_data['mean_soma_dist'] = I.np.mean(syn_data['soma_dists'])

    for syn in synapses:
        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0])
        syn_seg_id = int(syn.x * simulated_cell.sections[syn.secID].nseg)
        I.plt.plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recVList[syn_seg_id])

        %matplotlib inline
        I.plt.title(compute_syn_distance(simulated_cell, syn))
        I.plt.xlabel('Time (ms)')
        I.plt.ylabel('Membrane potential (mV)')
        I.plt.show()

syn_data

### Markram et al., 1997

In [ ]:
cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

# # Map synapses
# syn_dist = read_synapse_realization(syn_file_path)
# synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
# synapse_mapper.map_synapse_realization()

presyn_cell_type = L5TT_cell_data['presyn_cell_label'].values[0]
preSynCellID = L5TT_cell_data['presyn_cell_id'].values[0]


synapse_list = presyn_cells[presyn_cell_type][preSynCellID]
print(synapse_list)

synapses = []
for syn_id, syn in enumerate(synapse_list):
    if (presyn_cell_type in EXCITATORY) or (presyn_cell_type.split('_')[0] in EXCITATORY):
        synapse_strength_celltype = [x for x in loaded_syn_weights.keys() if x in presyn_cell_type]
        assert(len(synapse_strength_celltype) == 1)
        synapse_strength_celltype = synapse_strength_celltype[0]
        weight = loaded_syn_weights[synapse_strength_celltype]

        syn.weight = {'glutamate_syn': [weight, weight]}
        syn.receptors= EXC_RECEPTOR_DICT

        synapses.append(syn)
    else:
        continue

if len(synapses) > 0:
    cell, cell_params = loaded_simulator.setup.get(cell_params)

    cell_params['MarkramEtAl1997.stim.stim_start'] = 400 #I.np.arange(start=300, stop=600, step=200)  # 10 Hz
    cell_params['MarkramEtAl1997.stim.postsyn_stim_amp'] = spike_amp  # AP_amp
    cell_params['MarkramEtAl1997.stim.postsyn_time_to_spike'] = time_to_spike
    cell_params['MarkramEtAl1997.stim.burst_AP_freq'] = 10
    cell_params['MarkramEtAl1997.stim.num_AP_burst'] = 5
    cell_params['MarkramEtAl1997.stim.pairing_delay'] = 10
    cell_params['MarkramEtAl1997.stim.synapses'] = synapses

    simulated_cell, params = loaded_simulator.get_simulated_cell(cell_params, 'MarkramEtAl1997')

    syn_data = {
        'presyn_cell_label': presyn_cell_type,
        'presyn_cell_id': preSynCellID,
        'num_syns': len(synapses),
        'soma_dists': [compute_syn_distance(simulated_cell, syn) for syn in synapses],
        'syn_weight': weight,
        # 'EPSP_amp': calc_epsp_amp(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], t_start=cell_params['GlutamateUncaging.stim.syn_stim_times'][0] - 10),
        # 'EPSP_t': cell.tVec,
        # 'EPSP_v': cell.sections[0].recVList[0]
    }

    # Calculate mean distance between synapses and soma
    syn_data['mean_soma_dist'] = I.np.mean(syn_data['soma_dists'])

    I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0])

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.ylabel('Membrane potential (mV)')
# I.plt.xlim(390, 420)
I.plt.show()

syn_data

In [ ]:
cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

# # Map synapses
# syn_dist = read_synapse_realization(syn_file_path)
# synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
# synapse_mapper.map_synapse_realization()

presyn_cell_type = L5TT_cell_data['presyn_cell_label'].values[0]
preSynCellID = L5TT_cell_data['presyn_cell_id'].values[0]

synapse_list = presyn_cells[presyn_cell_type][preSynCellID]
print(len(synapse_list))

synapses = []
for syn_id, syn in enumerate(synapse_list):
    if (presyn_cell_type in EXCITATORY) or (presyn_cell_type.split('_')[0] in EXCITATORY):
        synapse_strength_celltype = [x for x in loaded_syn_weights.keys() if x in presyn_cell_type]
        assert(len(synapse_strength_celltype) == 1)
        synapse_strength_celltype = synapse_strength_celltype[0]
        weight = loaded_syn_weights[synapse_strength_celltype]

        syn.weight = {'glutamate_syn': [weight, weight]}
        syn.receptors= EXC_RECEPTOR_DICT

        synapses.append(syn)
    else:
        continue

if len(synapses) > 0:
    cell, cell_params = loaded_simulator.setup.get(cell_params)

    cell_params['MarkramEtAl1997.stim.stim_start'] = 400 #I.np.arange(start=300, stop=600, step=200)  # 10 Hz
    cell_params['MarkramEtAl1997.stim.postsyn_stim_amp'] = spike_amp  # AP_amp
    cell_params['MarkramEtAl1997.stim.postsyn_time_to_spike'] = time_to_spike
    cell_params['MarkramEtAl1997.stim.burst_AP_freq'] = 10
    cell_params['MarkramEtAl1997.stim.num_AP_burst'] = 5
    cell_params['MarkramEtAl1997.stim.pairing_delay'] = -10
    cell_params['MarkramEtAl1997.stim.synapses'] = synapses

    simulated_cell, params = loaded_simulator.get_simulated_cell(cell_params, 'MarkramEtAl1997')

    syn_data = {
        'presyn_cell_label': presyn_cell_type,
        'presyn_cell_id': preSynCellID,
        'num_syns': len(synapses),
        'soma_dists': [compute_syn_distance(simulated_cell, syn) for syn in synapses],
        'syn_weight': weight,
        # 'EPSP_amp': calc_epsp_amp(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], t_start=cell_params['GlutamateUncaging.stim.syn_stim_times'][0] - 10),
        # 'EPSP_t': cell.tVec,
        # 'EPSP_v': cell.sections[0].recVList[0]
    }

    # Calculate mean distance between synapses and soma
    syn_data['mean_soma_dist'] = I.np.mean(syn_data['soma_dists'])

    I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0])

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.ylabel('Membrane potential (mV)')
# I.plt.xlim(390, 420)
I.plt.show()

syn_data

## Bittner et al., 2017

In [ ]:
import random
random.seed(0)

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

# Map synapses
syn_dist = read_synapse_realization(syn_file_path)
synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
synapse_mapper.map_synapse_realization()

syn_df = I.pd.DataFrame(columns=['presyn_cell_label', 'syn_id', 'soma_dist', 'syn_weight', 'EPSP_amp'])  #, 'EPSP_t', 'EPSP_v'])

all_synapses = []
for presyn_cell_type in tqdm(cell.synapses.keys()):
    for syn_id, syn in enumerate(cell.synapses[presyn_cell_type]):
        if (presyn_cell_type in EXCITATORY) or (presyn_cell_type.split('_')[0] in EXCITATORY):
            synapse_strength_celltype = [x for x in loaded_syn_weights.keys() if x in presyn_cell_type]
            assert(len(synapse_strength_celltype) == 1)
            synapse_strength_celltype = synapse_strength_celltype[0]
            weight = loaded_syn_weights[synapse_strength_celltype]

            syn.weight = {'glutamate_syn': [weight, weight]}
            syn.receptors= EXC_RECEPTOR_DICT

            all_synapses.append(syn)

synapse_list = filter_syns_by_dist(all_synapses, cell, 200, 400)
print(f'{len(synapse_list)} / {len(all_synapses)}')

random.shuffle(synapse_list)

synapses = []
for syn_id, syn in enumerate(synapse_list):
    synapses.append(syn)

    break

if len(synapses) > 0:
    cell, cell_params = loaded_simulator.setup.get(cell_params)

    cell_params['BittnerEtAl2017.stim.stim_start'] = 400
    cell_params['BittnerEtAl2017.stim.presyn_stim_freq'] = 20
    cell_params['BittnerEtAl2017.stim.presyn_stim_num'] = 10
    cell_params['BittnerEtAl2017.stim.postsyn_stim_amp'] = 0.6
    cell_params['BittnerEtAl2017.stim.postsyn_duration'] = 300
    cell_params['BittnerEtAl2017.stim.pairing_delay'] = 0
    cell_params['BittnerEtAl2017.stim.synapses'] = synapses

    cell_params['BittnerEtAl2017.run.tStop'] = 1200
    cell_params['BittnerEtAl2017.run.vardt'] = True

    # cell_params[cell_params.index.str.contains('K_')] = 0

    simulated_cell, params = loaded_simulator.get_simulated_cell(cell_params, 'BittnerEtAl2017')

    syn_data = {
        'presyn_cell_label': syn.preCellType,
        # 'presyn_cell_id': preSynCellID,
        'num_syns': len(synapses),
        'soma_dists': [compute_syn_distance(simulated_cell, syn) for syn in synapses],
        'syn_weight': weight,
        # 'EPSP_amp': calc_epsp_amp(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], t_start=cell_params['GlutamateUncaging.stim.syn_stim_times'][0] - 10),
        # 'EPSP_t': cell.tVec,
        # 'EPSP_v': cell.sections[0].recVList[0]
    }

    # Calculate mean distance between synapses and soma
    syn_data['mean_soma_dist'] = I.np.mean(syn_data['soma_dists'])

    I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0])
    syn_seg_id = int(syn.x * simulated_cell.sections[syn.secID].nseg)
    I.plt.plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recVList[syn_seg_id])

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.ylabel('Membrane potential (mV)')
# I.plt.xlim(390, 420)
I.plt.show()

syn_data

In [ ]:
from visualize.cell_morphology_visualizer import CellMorphologyVisualizer

images_path = I.os.path.join(db_dir, 'bittner2017_animation_3d')

# To remake the visualization, uncomment this code
# if I.os.path.exists(images_path):
#     I.shutil.rmtree(images_path)

cmv = CellMorphologyVisualizer(simulated_cell)
cmv.population_to_color_dict['inactive'] = "#f0f0f0"  # add color for inactive
# cmv.population_to_color_dict['Generic'] = "red"

cmv.animation(
    images_path=images_path, 
    color="voltage",
    show_synapses=False,
    show_legend=False,
    client=client, 
    t_start=400-25, t_stop=1000, t_step=0.5
)

In [ ]:
from visualize.current_visualizer import CurrentAnalysis

%matplotlib inline
ca = CurrentAnalysis(simulated_cell, rangeVars=record_vars)
ca.plot_areas(plot_voltage=True, t_stim=390, select_window_relative_to_stim=(0, 600))

In [ ]:
import random
random.seed(0)

cell, cell_params = loaded_simulator.setup.get(loaded_cell_params.loc[biophysics_id])

# Map synapses
syn_dist = read_synapse_realization(syn_file_path)
synapse_mapper = SynapseMapper(cell, synDist=syn_dist, isDensity=False)
synapse_mapper.map_synapse_realization()

syn_df = I.pd.DataFrame(columns=['presyn_cell_label', 'syn_id', 'soma_dist', 'syn_weight', 'EPSP_amp'])  #, 'EPSP_t', 'EPSP_v'])

all_synapses = []
for presyn_cell_type in tqdm(cell.synapses.keys()):
    for syn_id, syn in enumerate(cell.synapses[presyn_cell_type]):
        if (presyn_cell_type in EXCITATORY) or (presyn_cell_type.split('_')[0] in EXCITATORY):
            synapse_strength_celltype = [x for x in loaded_syn_weights.keys() if x in presyn_cell_type]
            assert(len(synapse_strength_celltype) == 1)
            synapse_strength_celltype = synapse_strength_celltype[0]
            weight = loaded_syn_weights[synapse_strength_celltype]

            syn.weight = {'glutamate_syn': [weight, weight]}
            syn.receptors= EXC_RECEPTOR_DICT

            all_synapses.append(syn)

synapse_list = filter_syns_by_dist(all_synapses, cell, 200, 400)
print(f'{len(synapse_list)} / {len(all_synapses)}')

random.shuffle(synapse_list)

synapses = []
for syn_id, syn in enumerate(synapse_list):
    synapses.append(syn)

    break

if len(synapses) > 0:
    # Full protocol
    cell, cell_params = loaded_simulator.setup.get(cell_params)

    cell_params['BittnerEtAl2017.stim.stim_start'] = 400
    cell_params['BittnerEtAl2017.stim.presyn_stim_freq'] = 20
    cell_params['BittnerEtAl2017.stim.presyn_stim_num'] = 10
    cell_params['BittnerEtAl2017.stim.postsyn_stim_amp'] = 0.6
    cell_params['BittnerEtAl2017.stim.postsyn_duration'] = 300
    cell_params['BittnerEtAl2017.stim.pairing_delay'] = 0
    cell_params['BittnerEtAl2017.stim.synapses'] = synapses
    cell_params['BittnerEtAl2017.run.tStop'] = 1200

    simulated_cell, params = loaded_simulator.get_simulated_cell(cell_params, 'BittnerEtAl2017')
    full_protocol_data = (simulated_cell.tVec, simulated_cell.sections[0].recVList[0], simulated_cell.sections[syn.secID].recVList[0])

    syn_data = {
        'presyn_cell_label': syn.preCellType,
        # 'presyn_cell_id': preSynCellID,
        'num_syns': len(synapses),
        'soma_dists': [compute_syn_distance(simulated_cell, syn) for syn in synapses],
        'syn_weight': weight,
        # 'EPSP_amp': calc_epsp_amp(simulated_cell.tVec, simulated_cell.sections[0].recVList[0], t_start=cell_params['GlutamateUncaging.stim.syn_stim_times'][0] - 10),
        # 'EPSP_t': cell.tVec,
        # 'EPSP_v': cell.sections[0].recVList[0]
    }

    # Calculate mean distance between synapses and soma
    syn_data['mean_soma_dist'] = I.np.mean(syn_data['soma_dists'])

    I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0])
    syn_seg_id = int(syn.x * simulated_cell.sections[syn.secID].nseg)
    I.plt.plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recVList[syn_seg_id])

    %matplotlib inline
    I.plt.xlabel('Time (ms)')
    I.plt.ylabel('Membrane potential (mV)')
    # I.plt.xlim(390, 420)
    I.plt.show()


    # Only EPSPs
    cell, cell_params = loaded_simulator.setup.get(cell_params)

    cell_params['BittnerEtAl2017.stim.stim_start'] = 400
    cell_params['BittnerEtAl2017.stim.presyn_stim_freq'] = 20
    cell_params['BittnerEtAl2017.stim.presyn_stim_num'] = 10
    cell_params['BittnerEtAl2017.stim.postsyn_stim_amp'] = 0
    cell_params['BittnerEtAl2017.stim.postsyn_duration'] = 300
    cell_params['BittnerEtAl2017.stim.pairing_delay'] = 0
    cell_params['BittnerEtAl2017.stim.synapses'] = synapses
    cell_params['BittnerEtAl2017.run.tStop'] = 1200

    simulated_cell, params = loaded_simulator.get_simulated_cell(cell_params, 'BittnerEtAl2017')
    only_epsps_data = (simulated_cell.tVec, simulated_cell.sections[0].recVList[0], simulated_cell.sections[syn.secID].recVList[0])

    I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0])
    syn_seg_id = int(syn.x * simulated_cell.sections[syn.secID].nseg)
    I.plt.plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recVList[syn_seg_id])

    %matplotlib inline
    I.plt.xlabel('Time (ms)')
    I.plt.ylabel('Membrane potential (mV)')
    # I.plt.xlim(390, 420)
    I.plt.show()


    # Only somatic current injection
    cell, cell_params = loaded_simulator.setup.get(cell_params)

    cell_params['BittnerEtAl2017.stim.stim_start'] = 400
    cell_params['BittnerEtAl2017.stim.presyn_stim_freq'] = 20
    cell_params['BittnerEtAl2017.stim.presyn_stim_num'] = 10
    cell_params['BittnerEtAl2017.stim.postsyn_stim_amp'] = 0.6
    cell_params['BittnerEtAl2017.stim.postsyn_duration'] = 300
    cell_params['BittnerEtAl2017.stim.pairing_delay'] = 0
    cell_params['BittnerEtAl2017.stim.synapses'] = []
    cell_params['BittnerEtAl2017.run.tStop'] = 1200

    simulated_cell, params = loaded_simulator.get_simulated_cell(cell_params, 'BittnerEtAl2017')
    only_soma_inj_data = (simulated_cell.tVec, simulated_cell.sections[0].recVList[0], simulated_cell.sections[syn.secID].recVList[0])

    I.plt.plot(simulated_cell.tVec, simulated_cell.sections[0].recVList[0])
    syn_seg_id = int(syn.x * simulated_cell.sections[syn.secID].nseg)
    I.plt.plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recVList[syn_seg_id])

    %matplotlib inline
    I.plt.xlabel('Time (ms)')
    I.plt.ylabel('Membrane potential (mV)')
    # I.plt.xlim(390, 420)
    I.plt.show()

syn_data

In [ ]:
# from biophysics_fitting.utils import _get_apical_sec_and_i_at_distance

# section = _get_apical_sec_and_i_at_distance(simulated_cell, loaded_fixed_params['BAC.hay_measure.recSite'])
# I.plt.plot(simulated_cell.tVec, section[0].recVList[0])

In [ ]:
# I.plt.plot(full_protocol_data[0], (full_protocol_data[1] - only_epsps_data[1]))
# I.plt.plot(full_protocol_data[0], (full_protocol_data[1] - only_soma_inj_data[1]))
# I.plt.plot(full_protocol_data[0], ((full_protocol_data[1] - only_epsps_data[1]) + (full_protocol_data[1] - only_soma_inj_data[1])))
I.plt.plot(full_protocol_data[0], full_protocol_data[1])
I.plt.plot(full_protocol_data[0], only_soma_inj_data[1])
# I.plt.plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recVList[0])

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.ylabel('Membrane potential (mV)')
# I.plt.xlim(390, 1000)
I.plt.show()

In [ ]:
# I.plt.plot(full_protocol_data[0], (full_protocol_data[2] - only_epsps_data[2]))
I.plt.plot(full_protocol_data[0], (full_protocol_data[2] - only_soma_inj_data[2]))
# I.plt.plot(full_protocol_data[0], ((full_protocol_data[1] - only_epsps_data[1]) + (full_protocol_data[1] - only_soma_inj_data[1])))
# I.plt.plot(full_protocol_data[0], full_protocol_data[2])
# I.plt.plot(full_protocol_data[0], only_soma_inj_data[2])
# I.plt.plot(full_protocol_data[0], only_epsps_data[2])
# I.plt.plot(simulated_cell.tVec, simulated_cell.sections[syn.secID].recVList[0])

%matplotlib inline
I.plt.xlabel('Time (ms)')
I.plt.ylabel('Membrane potential (mV)')
# I.plt.xlim(390, 420)
I.plt.show()

In [ ]:
print(I.np.argmax((full_protocol_data[2] - only_soma_inj_data[2])))
print(I.np.argmin((full_protocol_data[2] - only_soma_inj_data[2])))